<a href="https://colab.research.google.com/github/GHWabble/mymd/blob/main/md_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Molecular dynamics pipeline

Preparation and molecular dynamics of a protein–ligand complex using:

- AmberTools for ligand parameterization and system preparation
- OpenMM for molecular dynamics
- MDTraj for trajectory analysis
- ParmEd for topology inspection

The notebook is intended to be reproducible from the original
protein and ligand input files.

In [1]:
#@title 1. Install Conda in Colab
import os, subprocess
from pathlib import Path

CONDA_DIR = "/usr/local/conda"

if not Path(f"{CONDA_DIR}/bin/conda").exists():
    print("Установка Miniconda...")
    subprocess.run([
        "wget", "-q",
        "https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh"
    ], check=True)
    subprocess.run([
        "bash", "Miniconda3-latest-Linux-x86_64.sh",
        "-b", "-p", CONDA_DIR
    ], check=True)
    subprocess.run(["rm", "Miniconda3-latest-Linux-x86_64.sh"], check=True)
    print("Miniconda успешно установлена!")
else:
    print("Miniconda уже установлена.")

# PATH обновляем ВСЕГДА — и внутри, и снаружи if
os.environ["PATH"] = f"{CONDA_DIR}/bin:" + os.environ.get("PATH", "")

# Проверим, что всё видно
import shutil
print("conda in PATH:", shutil.which("conda"))
subprocess.run(["conda", "--version"], check=True)

Miniconda уже установлена.
conda in PATH: /usr/local/conda/bin/conda


CompletedProcess(args=['conda', '--version'], returncode=0)

In [2]:
#@title 2. Check Colab environment

from pathlib import Path
import shutil
import sys

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Python prefix:", sys.prefix)
print("Conda:", shutil.which("conda"))

conda_bin = Path("/usr/local/conda/bin/conda")

if not conda_bin.is_file():
    raise RuntimeError(
        "Conda environment was not initialized correctly: "
        "Miniconda binary was not found at /usr/local/conda."
    )

if shutil.which("conda") is None:
    raise RuntimeError("Conda executable was not found in PATH.")

print("\nMiniconda environment is active.")

Python version: 3.13.15
Python executable: /usr/bin/python3
Python prefix: /usr
Conda: /usr/local/conda/bin/conda

Miniconda environment is active.


In [5]:
#@title 3. Install MD dependencies
import os, sys, shutil, subprocess, importlib
from pathlib import Path

CONDA_DIR = Path("/usr/local/conda")

if not (CONDA_DIR / "bin" / "conda").exists():
    raise RuntimeError(
        "Miniconda не найдена в /usr/local/conda.\n"
        "Сначала запустите ячейку 1."
    )

conda_bin    = str(CONDA_DIR / "bin")
conda_lib    = str(CONDA_DIR / "lib")
conda_python = str(CONDA_DIR / "bin" / "python")

# PATH для subprocess, чтобы CLI (tleap, antechamber) были видны
os.environ["PATH"] = conda_bin + os.pathsep + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = conda_lib + os.pathsep + os.environ.get("LD_LIBRARY_PATH", "")

# ---------- Чистое окружение для conda-python subprocess'ов ----------
# Colab ставит MPLBACKEND=module://matplotlib_inline.backend_inline,
# что валидно только внутри Jupyter. В отдельном процессе это ломает
# импорт matplotlib. Подменяем на headless Agg.
CONDA_ENV = os.environ.copy()
CONDA_ENV.pop("MPLBACKEND", None)
CONDA_ENV["MPLBACKEND"] = "Agg"

# ---------- Проверка, что уже стоит ----------
def has_module_in_conda(name):
    r = subprocess.run(
        [conda_python, "-c", f"import {name}"],
        capture_output=True,
        env=CONDA_ENV,
    )
    return r.returncode == 0

need = {
    "ambertools": not all(shutil.which(t) for t in ["antechamber", "parmchk2", "tleap"]),
    "openmm":     not has_module_in_conda("openmm"),
    "pdbfixer":   not has_module_in_conda("pdbfixer"),
    "parmed":     not has_module_in_conda("parmed"),
    "mdtraj":     not has_module_in_conda("mdtraj"),
}

to_install = [pkg for pkg, missing in need.items() if missing]

if not to_install:
    print("✅ Все зависимости уже установлены, установка пропущена.")
else:
    print("Будут установлены:", ", ".join(to_install))

# ---------- Solver ----------
def which_local(name):
    p = CONDA_DIR / "bin" / name
    return str(p) if p.exists() else None

solver = (
    which_local("micromamba")
    or which_local("mamba")
    or which_local("conda")
)

if solver is None:
    raise RuntimeError("В /usr/local/conda/bin нет ни conda, ни mamba, ни micromamba.")

print(f"⚡ Пакетный менеджер: {solver}")

def run_step(cmd, description):
    print(f"\n[📦] {description}\n" + "-" * 50)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"Команда провалилась (код {p.returncode}): {' '.join(cmd)}")
    print("-" * 50 + f"\n✅ {description}")

if to_install:
    run_step(
        [solver, "install", "-y", "-c", "conda-forge",
         "--override-channels", *to_install],
        "Установка MD-стека"
    )

# ---------- Верификация через conda-python (чистое окружение) ----------
print("\n[🔍] Проверка компонентов (через conda-python)...")

for tool in ["antechamber", "parmchk2", "tleap"]:
    path = shutil.which(tool)
    if not path:
        raise RuntimeError(f"Не найдена утилита: {tool}")
    print(f"  ✅ {tool}: {path}")

verify_script = '''
import importlib
mods = ["openmm", "pdbfixer", "parmed", "mdtraj", "scipy", "pandas", "matplotlib"]
for m in mods:
    try:
        mod = importlib.import_module(m)
        print(f"  ✅ {m} {getattr(mod, '__version__', '')}")
    except Exception as e:
        print(f"  ❌ {m}: {e}")
        raise
'''
res = subprocess.run(
    [conda_python, "-c", verify_script],
    capture_output=True, text=True,
    env=CONDA_ENV,
)
print(res.stdout, end="")
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Верификация в conda-python провалилась.")

print("\n[🧪] openmm.testInstallation...")
res = subprocess.run(
    [conda_python, "-m", "openmm.testInstallation"],
    capture_output=True, text=True,
    env=CONDA_ENV,
)
if res.returncode == 0:
    print("  ✅ OpenMM работает.")
else:
    print("  ⚠️ Вывод теста:")
    print(res.stdout or res.stderr)

print("\n🎉 Готово.")

✅ Все зависимости уже установлены, установка пропущена.
⚡ Пакетный менеджер: /usr/local/conda/bin/conda

[🔍] Проверка компонентов (через conda-python)...
  ✅ antechamber: /usr/local/conda/bin/antechamber
  ✅ parmchk2: /usr/local/conda/bin/parmchk2
  ✅ tleap: /usr/local/conda/bin/tleap
  ✅ openmm 8.6.1
  ✅ pdbfixer 
  ✅ parmed 4.3.1
  ✅ mdtraj 1.11.1
  ✅ scipy 1.18.1
  ✅ pandas 2.3.3
  ✅ matplotlib 3.10.9

[🧪] openmm.testInstallation...
  ✅ OpenMM работает.

🎉 Готово.


In [6]:
#@title 4. Verify Python packages
import importlib, subprocess, json, os

# --- Kernel (Python 3.13, pip) ---
kernel_modules = [
    "gemmi", "rdkit", "numpy", "scipy", "pandas", "matplotlib",
    "openmm", "pdbfixer", "parmed", "mdtraj",
]
failed = []

for name in kernel_modules:
    try:
        m = importlib.import_module(name)
        print(f"[kernel] {name:<12} OK   {getattr(m, '__version__', '?')}")
    except Exception as exc:
        failed.append(name)
        print(f"[kernel] {name:<12} FAIL {type(exc).__name__}: {exc}")

# --- Conda (Python 3.14, только через subprocess) ---
conda_python = "/usr/local/conda/bin/python"

CONDA_ENV = os.environ.copy()
CONDA_ENV.pop("MPLBACKEND", None)
CONDA_ENV["MPLBACKEND"] = "Agg"

script = '''
import importlib, json
mods = ["openmm", "parmed", "numpy", "scipy"]
out = {}
for m in mods:
    try:
        out[m] = getattr(importlib.import_module(m), "__version__", "?")
    except Exception as e:
        out[m] = f"FAIL: {type(e).__name__}: {e}"
print(json.dumps(out))
'''
res = subprocess.run([conda_python, "-c", script],
                     capture_output=True, text=True,
                     env=CONDA_ENV)
conda_report = json.loads(res.stdout.strip().splitlines()[-1])

for name, ver in conda_report.items():
    if isinstance(ver, str) and ver.startswith("FAIL"):
        failed.append(f"conda:{name}")
        print(f"[conda ] {name:<12} FAIL {ver}")
    else:
        print(f"[conda ] {name:<12} OK   {ver}")

if failed:
    raise RuntimeError("Проблемы с пакетами: " + ", ".join(failed))

print("\nВсе пакеты на месте (kernel = pip, conda = subprocess).")

[kernel] gemmi        OK   0.7.5
[kernel] rdkit        OK   2026.03.6
[kernel] numpy        OK   2.1.3
[kernel] scipy        OK   1.16.3
[kernel] pandas       OK   2.2.3
[kernel] matplotlib   OK   3.10.0
[kernel] openmm       OK   8.6.1
[kernel] pdbfixer     OK   ?
[kernel] parmed       OK   4.3.1
[kernel] mdtraj       OK   1.11.1.post2
[conda ] openmm       OK   8.6.1
[conda ] parmed       OK   4.3.1
[conda ] numpy        OK   2.4.6
[conda ] scipy        OK   1.18.1

Все пакеты на месте (kernel = pip, conda = subprocess).


In [7]:
#@title 5. Verify AmberTools

import os
from shutil import which

# Убедимся, что путь к бинарникам Miniconda прописан в PATH для поиска через which
conda_bin_dir = "/usr/local/conda/bin"
if conda_bin_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{conda_bin_dir}:{os.environ.get('PATH', '')}"

amber_tools = [
    "antechamber",
    "parmchk2",
    "tleap",
    "pdb4amber",
    "cpptraj",
    "MMPBSA.py",
]

missing = []

for program in amber_tools:
    path = which(program)

    if path is None:
        missing.append(program)
        print(f"{program:<12} NOT FOUND")
    else:
        print(f"{program:<12} {path}")

if missing:
    raise RuntimeError(
        "Missing AmberTools executables: "
        + ", ".join(missing)
    )

print("\nAmberTools check passed.")

antechamber  /usr/local/conda/bin/antechamber
parmchk2     /usr/local/conda/bin/parmchk2
tleap        /usr/local/conda/bin/tleap
pdb4amber    /usr/local/conda/bin/pdb4amber
cpptraj      /usr/local/conda/bin/cpptraj
MMPBSA.py    /usr/local/conda/bin/MMPBSA.py

AmberTools check passed.


In [8]:
#@title 6. Check OpenMM and GPU
import os, subprocess, textwrap

conda_python = "/usr/local/conda/bin/python"

CONDA_ENV = os.environ.copy()
CONDA_ENV.pop("MPLBACKEND", None)
CONDA_ENV["MPLBACKEND"] = "Agg"

script = textwrap.dedent("""
    import openmm as mm
    print("OpenMM version:", mm.__version__)
    platforms = [mm.Platform.getPlatform(i).getName()
                 for i in range(mm.Platform.getNumPlatforms())]
    print("Platforms:", ", ".join(platforms))
    print("CUDA:", "CUDA" in platforms)
""")
res = subprocess.run([conda_python, "-c", script],
                     capture_output=True, text=True,
                     env=CONDA_ENV)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("OpenMM check failed in conda Python.")

OpenMM version: 8.6.1
Platforms: Reference, CPU, CUDA, OpenCL
CUDA: True



In [9]:
#@title 7. Run OpenMM self-test
import os, subprocess, sys
from pathlib import Path

conda_python = "/usr/local/conda/bin/python"
python_executable = conda_python if Path(conda_python).exists() else sys.executable

CONDA_ENV = os.environ.copy()
CONDA_ENV.pop("MPLBACKEND", None)
CONDA_ENV["MPLBACKEND"] = "Agg"

subprocess.run(
    [python_executable, "-m", "openmm.testInstallation"],
    check=True,
    env=CONDA_ENV,
)

CompletedProcess(args=['/usr/local/conda/bin/python', '-m', 'openmm.testInstallation'], returncode=0)

In [ ]:
#@title 8. Install Python packages in kernel (pip)
import subprocess, sys, importlib

print(f"Kernel Python: {sys.executable} ({sys.version.split()[0]})")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "rdkit",
        "gemmi",
        "openmm",
        "pdbfixer",
        "parmed",
        "mdtraj",
    ],
    check=True,
)

for mod in ("gemmi", "rdkit", "openmm", "pdbfixer", "parmed", "mdtraj"):
    importlib.invalidate_caches()

import gemmi
from rdkit import Chem
import openmm
import parmed

print(f"Gemmi:   {gemmi.__version__}")
print(f"RDKit:   {Chem.rdBase.rdkitVersion}")
print(f"OpenMM:  {openmm.__version__}")
print(f"ParmEd:  {parmed.__version__}")
print("\n✅ Все Python-пакеты доступны в kernel.")

In [10]:
#@title 9. Check optional AmberTools capabilities

import os
from shutil import which

# Гарантируем, что путь к бинарникам Miniconda прописан в PATH
conda_bin_dir = "/usr/local/conda/bin"
if conda_bin_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{conda_bin_dir}:{os.environ.get('PATH', '')}"

optional_tools = {
    "packmol-memgen": "membrane builder",
    "MCPB.py": "metal-center parameterization",
    "nab": "nucleic-acid builder",
}

for executable, purpose in optional_tools.items():
    path = which(executable)

    if path is None:
        print(f"{executable:<16} not found   ({purpose})")
    else:
        print(f"{executable:<16} {path}")

packmol-memgen   /usr/local/conda/bin/packmol-memgen
MCPB.py          /usr/local/conda/bin/MCPB.py
nab              not found   (nucleic-acid builder)


In [11]:
#@title 10. Mount Google Drive

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
#@title 11. System settings

from pathlib import Path

# Files
WORKDIR_STR = "/content/drive/MyDrive/oncotarget_pipeline/md/ythdf2" #@param {type:"string"}
TARGET_INPUT_NAME = "YTHDF2.pdb" #@param {type:"string"}
LIGAND_INPUT_NAME = "irinotecan.sdf" #@param {type:"string"}
SYSTEM_NAME = "ythdf2_irinotecan" #@param {type:"string"}

# Target
TARGET_KIND = "protein" #@param ["auto", "protein", "dna", "rna", "protein_dna", "protein_rna", "dna_rna", "mixed"]
SYSTEM_ENVIRONMENT = "soluble" #@param ["soluble", "membrane"]

# Ligand
LIGAND_KIND = "small_molecule" #@param ["none", "small_molecule", "nucleic_acid", "peptide", "preparameterized"]
LIGAND_RESNAME = "IRI" #@param {type:"string"}
LIGAND_NET_CHARGE = 0 #@param {type:"integer"}
LIGAND_PARAMETERIZATION = "auto" #@param ["auto", "gaff2", "preparameterized"]

# Biomolecular force fields
PROTEIN_FORCE_FIELD = "ff19SB" #@param ["ff19SB", "ff14SB"]
DNA_FORCE_FIELD = "OL21" #@param ["OL21", "OL15", "bsc1"]
RNA_FORCE_FIELD = "OL3" #@param ["OL3"]
SMALL_MOLECULE_FORCE_FIELD = "gaff2" #@param ["gaff2", "gaff"]

# Solvent
WATER_MODEL = "OPC" #@param ["TIP3P", "OPC", "SPCE"]
WATER_PADDING_ANGSTROM = 10.0 #@param {type:"number"}
SALT_CONCENTRATION_MOLAR = 0.15 #@param {type:"number"}
TARGET_PH = 7.4 #@param {type:"number"}

# Membrane (игнорируется для soluble)
MEMBRANE_LIPIDS = "POPC" #@param {type:"string"}
MEMBRANE_LIPID_RATIO = "1" #@param {type:"string"}
MEMBRANE_PREORIENTED = False #@param {type:"boolean"}

# MD
TEMPERATURE_K = 310.15 #@param {type:"number"}
PRESSURE_BAR = 1.0 #@param {type:"number"}
TIMESTEP_FS = 2.0 #@param {type:"number"}
FRICTION_PER_PS = 1.0 #@param {type:"number"}

EQUILIBRATION_NS = 2.0 #@param {type:"number"}
PRODUCTION_NS = 10.0 #@param {type:"number"}

TRAJECTORY_INTERVAL_PS = 10.0 #@param {type:"number"}
LOG_INTERVAL_PS = 10.0 #@param {type:"number"}
CHECKPOINT_INTERVAL_PS = 100.0 #@param {type:"number"}


WORKDIR = Path(WORKDIR_STR)
WORKDIR.mkdir(parents=True, exist_ok=True)

TARGET_PATH = WORKDIR / TARGET_INPUT_NAME

LIGAND_PATH = (
    WORKDIR / LIGAND_INPUT_NAME
    if LIGAND_KIND != "none"
    else None
)

print("Work directory:", WORKDIR)
print("Target:", TARGET_PATH)
print("Ligand:", LIGAND_PATH or "none")

Work directory: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2
Target: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/YTHDF2.pdb
Ligand: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/irinotecan.sdf


In [13]:
#@title 12. Inspect ligand

from collections import Counter
from pathlib import Path

import gemmi
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors


def load_small_molecule(path: Path) -> Chem.Mol:
    """
    Read one small molecule while preserving explicit hydrogens
    and the coordinates stored in the input file.
    """
    suffix = path.suffix.lower()

    if suffix in {".sdf", ".sd"}:
        supplier = Chem.SDMolSupplier(
            str(path),
            removeHs=False,
            sanitize=True,
            strictParsing=True,
        )

        if len(supplier) != 1:
            raise ValueError(
                f"{path.name} must contain exactly one molecule; "
                f"found {len(supplier)} records."
            )

        molecule = supplier[0]

    elif suffix == ".mol":
        molecule = Chem.MolFromMolFile(
            str(path),
            removeHs=False,
            sanitize=True,
            strictParsing=True,
        )

    elif suffix == ".mol2":
        molecule = Chem.MolFromMol2File(
            str(path),
            removeHs=False,
            sanitize=True,
        )

    else:
        raise ValueError(
            "For an unparameterized small molecule, use SDF, MOL, "
            "or MOL2. PDB is intentionally not accepted because "
            "bond orders may be ambiguous."
        )

    if molecule is None:
        raise ValueError(
            f"RDKit could not read or sanitize {path.name}."
        )

    return molecule


def get_element_counts(molecule: Chem.Mol) -> Counter:
    return Counter(
        atom.GetSymbol()
        for atom in molecule.GetAtoms()
    )


def get_metal_elements(molecule: Chem.Mol) -> list[str]:
    elements = {
        atom.GetSymbol()
        for atom in molecule.GetAtoms()
    }

    return sorted(
        element
        for element in elements
        if gemmi.Element(element).is_metal
    )


def validate_3d_coordinates(molecule: Chem.Mol) -> None:
    if molecule.GetNumConformers() != 1:
        raise ValueError(
            "The ligand must contain exactly one coordinate set."
        )

    conformer = molecule.GetConformer()

    if not conformer.Is3D():
        raise ValueError(
            "The ligand is not marked as a 3D structure. "
            "Provide the docked 3D pose rather than a 2D structure."
        )


def count_explicit_hydrogens(molecule: Chem.Mol) -> int:
    return sum(
        atom.GetAtomicNum() == 1
        for atom in molecule.GetAtoms()
    )


ligand_mol = None
ligand_metals = []
parameterization_route = None


if LIGAND_KIND == "none":
    print("No ligand was supplied.")

elif LIGAND_KIND == "small_molecule":
    if LIGAND_PATH is None:
        raise ValueError(
            "LIGAND_KIND is 'small_molecule', "
            "but no ligand file was specified."
        )

    ligand_mol = load_small_molecule(
        LIGAND_PATH
    )

    validate_3d_coordinates(
        ligand_mol
    )

    fragments = Chem.GetMolFrags(
        ligand_mol
    )

    if len(fragments) != 1:
        raise ValueError(
            "The ligand contains disconnected fragments "
            f"({len(fragments)} detected). "
            "Salts, counterions, and mixtures must be handled "
            "explicitly before parameterization."
        )

    element_counts = get_element_counts(
        ligand_mol
    )

    ligand_metals = get_metal_elements(
        ligand_mol
    )

    formal_charge = Chem.GetFormalCharge(
        ligand_mol
    )

    heavy_atom_count = (
        ligand_mol.GetNumHeavyAtoms()
    )

    explicit_h_count = (
        count_explicit_hydrogens(
            ligand_mol
        )
    )

    formula = (
        rdMolDescriptors.CalcMolFormula(
            ligand_mol
        )
    )

    molecular_weight = (
        Descriptors.MolWt(
            ligand_mol
        )
    )

    if formal_charge != LIGAND_NET_CHARGE:
        raise ValueError(
            "Ligand charge mismatch:\n"
            f"  charge stored in structure: {formal_charge}\n"
            f"  LIGAND_NET_CHARGE:          {LIGAND_NET_CHARGE}\n\n"
            "Do not continue until the intended protonation "
            "state and net charge have been checked."
        )

    if ligand_metals:
        parameterization_route = (
            "metal_complex"
        )

    elif (
        LIGAND_PARAMETERIZATION
        == "preparameterized"
    ):
        parameterization_route = (
            "preparameterized"
        )

    else:
        parameterization_route = (
            "gaff"
        )


    print("Ligand inspection")
    print("-----------------")
    print("File:", LIGAND_PATH.name)
    print("Formula:", formula)
    print(
        "Molecular weight:",
        f"{molecular_weight:.2f} Da",
    )
    print(
        "Heavy atoms:",
        heavy_atom_count,
    )
    print(
        "Explicit hydrogens:",
        explicit_h_count,
    )
    print(
        "Formal charge:",
        formal_charge,
    )
    print(
        "Elements:",
        dict(element_counts),
    )

    print(
        "Metals:",
        ", ".join(ligand_metals)
        if ligand_metals
        else "none",
    )

    print()
    print(
        "Parameterization route:",
        parameterization_route,
    )


    if parameterization_route == "metal_complex":
        print(
            "\nStandard GAFF/AM1-BCC parameterization "
            "will not be used automatically because "
            "the ligand contains a metal center."
        )

else:
    print(
        f"Ligand type '{LIGAND_KIND}' will use "
        "the biomolecular/preparameterized branch later."
    )

Ligand inspection
-----------------
File: irinotecan.sdf
Formula: C33H38N4O6
Molecular weight: 586.69 Da
Heavy atoms: 43
Explicit hydrogens: 38
Formal charge: 0
Elements: {'O': 6, 'N': 4, 'C': 33, 'H': 38}
Metals: none

Parameterization route: gaff


In [14]:
#@title 13. Preparation settings

PREP_DIR = WORKDIR / "preparation"
TARGET_PREP_DIR = PREP_DIR / "target"
LIGAND_PREP_DIR = PREP_DIR / "ligand"
LOG_DIR = PREP_DIR / "logs"

for directory in (
    TARGET_PREP_DIR,
    LIGAND_PREP_DIR,
    LOG_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


# Non-polymer components in the target structure.
# YTHDF2 не содержит обязательных структурных ионов металлов (в отличие от цинковых пальцев/нуклеаз).
KEEP_NONPOLYMER_RESNAMES = "" #@param {type:"string"}

# Список распространенных кристаллизационных буферов и осадителей для принудительного удаления
DROP_NONPOLYMER_RESNAMES = "EDO,GOL,SO4,PO4,DMS,FMT,ACY,PEG,PEG4,1PE,DTT" #@param {type:"string"}

# Удаляем кристаллографическую воду, так как бокс будет заново залит водой OPC
KEEP_CRYSTAL_WATERS = False #@param {type:"boolean"}

# Автоматически удаляем сокристаллизованный лиганд/ингибитор из исходного PDB
AUTO_REMOVE_MATCHING_LIGAND = True #@param {type:"boolean"}

EMBEDDED_LIGAND_RMSD_LIMIT_A = 0.15 #@param {type:"number"}
EMBEDDED_LIGAND_MAX_SHIFT_A = 0.30 #@param {type:"number"}

# Ставим False, чтобы PDBFixer автоматически выбирал главную конформацию A для альтлоков и не останавливал расчет
FAIL_ON_ALTLOCS = False #@param {type:"boolean"}

# Стандартный порог идентификации дисульфидных связей (Cys-Cys)
DISULFIDE_MAX_DISTANCE_A = 2.30 #@param {type:"number"}


def parse_residue_names(value: str) -> set[str]:
    return {
        name.strip().upper()
        for name in value.split(",")
        if name.strip()
    }


keep_nonpolymer = parse_residue_names(
    KEEP_NONPOLYMER_RESNAMES
)

drop_nonpolymer = parse_residue_names(
    DROP_NONPOLYMER_RESNAMES
)


overlap = keep_nonpolymer & drop_nonpolymer

if overlap:
    raise ValueError(
        "The same residue names cannot be both kept and removed: "
        + ", ".join(sorted(overlap))
    )


print("Preparation directory:", PREP_DIR)
print("Keep non-polymer residues:", keep_nonpolymer or "none")
print("Drop non-polymer residues:", drop_nonpolymer or "none")
print("Fail on altlocs:", FAIL_ON_ALTLOCS)

Preparation directory: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation
Keep non-polymer residues: none
Drop non-polymer residues: {'DMS', 'PEG4', 'SO4', 'DTT', 'PEG', 'FMT', 'ACY', '1PE', 'EDO', 'PO4', 'GOL'}
Fail on altlocs: False


In [17]:
from pathlib import Path

WORKDIR = Path("/content/drive/MyDrive/oncotarget_pipeline/md/ythdf2")

print("Все JSON-манифесты в WORKDIR:")
print("-" * 60)
for p in sorted(WORKDIR.rglob("*.json")):
    rel = p.relative_to(WORKDIR)
    print(f"  {rel}  ({p.stat().st_size} байт)")

Все JSON-манифесты в WORKDIR:
------------------------------------------------------------
  preparation/ligand/ligand_preparation.json  (748 байт)
  preparation/target/disulfides.json  (2 байт)
  preparation/target/final_target_audit.json  (214 байт)
  preparation/target/geometry_audit.json  (114 байт)
  preparation/target/protonation.json  (1365 байт)
  preparation/target/reference_applied.json  (103 байт)
  preparation/target/structural_repair_plan.json  (249 байт)
  preparation/target/target_ready.json  (466 байт)


In [21]:
#@title 14. Preparation utilities

from collections import Counter
from pathlib import Path
import shlex
import subprocess

import gemmi
import numpy as np
from scipy.optimize import linear_sum_assignment


def run_command(
    command: list[str],
    *,
    cwd: Path,
    log_path: Path,
) -> str:
    print("$", shlex.join(command))

    result = subprocess.run(
        command,
        cwd=cwd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    output = result.stdout or ""
    log_path.write_text(output)

    if result.returncode != 0:
        tail = "\n".join(
            output.splitlines()[-30:]
        )

        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n"
            f"{shlex.join(command)}\n\n"
            f"{tail}\n\n"
            f"Full log: {log_path}"
        )

    return output


def residue_key(
    chain: gemmi.Chain,
    residue: gemmi.Residue,
) -> tuple[str, int, str, str]:
    return (
        chain.name,
        residue.seqid.num,
        residue.seqid.icode.strip(),
        residue.name.strip().upper(),
    )


def residue_label(
    chain: gemmi.Chain,
    residue: gemmi.Residue,
) -> str:
    insertion_code = residue.seqid.icode.strip()

    number = str(residue.seqid.num)

    if insertion_code:
        number += insertion_code

    return (
        f"{chain.name}:"
        f"{residue.name.strip()} {number}"
    )


def rdkit_heavy_atoms(molecule):
    if molecule.GetNumConformers() != 1:
        raise ValueError(
            "Expected exactly one ligand coordinate set."
        )

    conformer = molecule.GetConformer()
    atoms = []

    for atom in molecule.GetAtoms():
        if atom.GetAtomicNum() <= 1:
            continue

        position = conformer.GetAtomPosition(
            atom.GetIdx()
        )

        atoms.append(
            (
                atom.GetSymbol(),
                np.array(
                    [position.x, position.y, position.z],
                    dtype=float,
                ),
            )
        )

    return atoms


def gemmi_heavy_atoms(
    residue: gemmi.Residue,
):
    return [
        (
            atom.element.name,
            np.array(
                [
                    atom.pos.x,
                    atom.pos.y,
                    atom.pos.z,
                ],
                dtype=float,
            ),
        )
        for atom in residue
        if atom.element.atomic_number > 1
    ]


def coordinate_match(
    reference,
    candidate,
):
    """
    Compare absolute coordinates without superposition.

    This detects the same ligand in the same pose, rather than merely
    recognising two molecules with similar geometry.
    """
    reference_elements = Counter(
        element for element, _ in reference
    )

    candidate_elements = Counter(
        element for element, _ in candidate
    )

    if reference_elements != candidate_elements:
        return None

    n_atoms = len(reference)

    if n_atoms == 0:
        return None

    distances = np.full(
        (n_atoms, n_atoms),
        np.inf,
        dtype=float,
    )

    for i, (element_a, xyz_a) in enumerate(reference):
        for j, (element_b, xyz_b) in enumerate(candidate):
            if element_a != element_b:
                continue

            distances[i, j] = np.linalg.norm(
                xyz_a - xyz_b
            )

    rows, columns = linear_sum_assignment(
        distances
    )

    matched_distances = distances[
        rows,
        columns,
    ]

    if not np.all(
        np.isfinite(matched_distances)
    ):
        return None

    return {
        "rmsd_A": float(
            np.sqrt(
                np.mean(
                    matched_distances**2
                )
            )
        ),
        "max_shift_A": float(
            matched_distances.max()
        ),
    }

In [33]:
#@title 15. Build target preparation plan

target_structure = gemmi.read_structure(
    str(TARGET_PATH)
)

if len(target_structure) == 0:
    raise ValueError(
        "Target structure contains no models."
    )

if len(target_structure) > 1:
    print(
        f"Target contains {len(target_structure)} models. "
        "Only the first model will be used."
    )

# Entity assignment is particularly important for PDB files:
# polymer, non-polymer and water must be distinguished chemically.
target_structure.setup_entities()

model = target_structure[0]


altloc_atoms = [
    (
        chain.name,
        residue.name,
        residue.seqid.num,
        atom.name,
        atom.altloc,
    )
    for chain in model
    for residue in chain
    for atom in residue
    if atom.has_altloc()
]


if altloc_atoms and FAIL_ON_ALTLOCS:
    examples = "\n".join(
        f"- {chain}:{residue}{number} "
        f"{atom} altloc={altloc}"
        for (
            chain,
            residue,
            number,
            atom,
            altloc,
        ) in altloc_atoms[:10]
    )

    raise ValueError(
        f"Alternative conformations were detected "
        f"({len(altloc_atoms)} atom records).\n\n"
        f"{examples}\n\n"
        "They should be resolved explicitly rather than "
        "silently choosing one conformation."
    )


ligand_reference_atoms = None

if (
    LIGAND_KIND == "small_molecule"
    and ligand_mol is not None
):
    ligand_reference_atoms = (
        rdkit_heavy_atoms(ligand_mol)
    )


preparation_plan = []
embedded_ligand_matches = []
unresolved_components = []


for chain in model:
    for residue in chain:
        key = residue_key(
            chain,
            residue,
        )

        label = residue_label(
            chain,
            residue,
        )

        details = ""

        if (
            residue.entity_type
            == gemmi.EntityType.Polymer
        ):
            action = "keep_polymer"

        elif residue.is_water():
            action = (
                "keep_water"
                if KEEP_CRYSTAL_WATERS
                else "drop_water"
            )

        else:
            residue_name = (
                residue.name.strip().upper()
            )

            match = None

            if (
                AUTO_REMOVE_MATCHING_LIGAND
                and ligand_reference_atoms
            ):
                match = coordinate_match(
                    ligand_reference_atoms,
                    gemmi_heavy_atoms(residue),
                )

            if (
                match is not None
                and match["rmsd_A"]
                <= EMBEDDED_LIGAND_RMSD_LIMIT_A
                and match["max_shift_A"]
                <= EMBEDDED_LIGAND_MAX_SHIFT_A
            ):
                action = "remove_embedded_ligand"

                details = (
                    f"RMSD={match['rmsd_A']:.3f} Å, "
                    f"max shift={match['max_shift_A']:.3f} Å"
                )

                embedded_ligand_matches.append(
                    {
                        "key": key,
                        "label": label,
                        **match,
                    }
                )

            elif residue_name in keep_nonpolymer:
                action = "keep_nonpolymer"
                details = "explicitly retained"

            elif residue_name in drop_nonpolymer:
                action = "drop_nonpolymer"
                details = "explicitly removed"

            else:
                action = "unresolved_nonpolymer"

                elements = sorted(
                    {
                        atom.element.name
                        for atom in residue
                    }
                )

                details = (
                    "elements="
                    + ",".join(elements)
                )

                unresolved_components.append(
                    {
                        "label": label,
                        "name": residue_name,
                        "elements": elements,
                    }
                )

        preparation_plan.append(
            {
                "key": key,
                "label": label,
                "action": action,
                "details": details,
            }
        )


if len(embedded_ligand_matches) > 1:
    matches = "\n".join(
        f"- {item['label']}"
        for item in embedded_ligand_matches
    )

    raise ValueError(
        "More than one component matches the supplied ligand:\n"
        + matches
        + "\nAutomatic removal would be ambiguous."
    )


print("Target preparation plan")
print("-----------------------")

for item in preparation_plan:
    if item["action"] == "keep_polymer":
        continue

    line = (
        f"{item['label']:<20} "
        f"{item['action']}"
    )

    if item["details"]:
        line += f"  ({item['details']})"

    print(line)


if unresolved_components:
    print("\nUnresolved components")
    print("---------------------")

    for component in unresolved_components:
        print(
            f"{component['label']:<20} "
            f"{','.join(component['elements'])}"
        )

    raise ValueError(
        "\nEvery non-polymer component must have an explicit fate. "
        "Add residue names to KEEP_NONPOLYMER_RESNAMES or "
        "DROP_NONPOLYMER_RESNAMES."
    )


print()
print(
    "Embedded ligand matches:",
    len(embedded_ligand_matches),
)

Target preparation plan
-----------------------

Embedded ligand matches: 0


In [34]:
#@title 16. Create clean target

TARGET_HEAVY_PATH = (
    TARGET_PREP_DIR
    / "target_heavy.pdb"
)


action_by_key = {
    item["key"]: item["action"]
    for item in preparation_plan
}


working_structure = gemmi.read_structure(
    str(TARGET_PATH)
)

# Only model 1 belongs to this simulation.
while len(working_structure) > 1:
    del working_structure[-1]

working_structure.setup_entities()

working_model = working_structure[0]

removed = Counter()
removed_hydrogens = 0


for chain in working_model:
    for residue_index in range(
        len(chain) - 1,
        -1,
        -1,
    ):
        residue = chain[residue_index]

        key = residue_key(
            chain,
            residue,
        )

        action = action_by_key.get(key)

        if action is None:
            raise RuntimeError(
                f"No preparation action for "
                f"{residue_label(chain, residue)}"
            )

        if action in {
            "remove_embedded_ligand",
            "drop_water",
            "drop_nonpolymer",
        }:
            removed[action] += 1
            del chain[residue_index]
            continue

        # Existing H on biopolymers are discarded.
        # They will be generated consistently later.
        if action == "keep_polymer":
            for atom_index in range(
                len(residue) - 1,
                -1,
                -1,
            ):
                if (
                    residue[atom_index]
                    .element.atomic_number
                    == 1
                ):
                    del residue[atom_index]
                    removed_hydrogens += 1


working_structure.remove_empty_chains()


working_structure.write_pdb(
    str(TARGET_HEAVY_PATH)
)


# The crystallographic unit cell is not our MD box.
pdb_lines = (
    TARGET_HEAVY_PATH
    .read_text()
    .splitlines()
)

pdb_lines = [
    line
    for line in pdb_lines
    if not line.startswith("CRYST1")
]

TARGET_HEAVY_PATH.write_text(
    "\n".join(pdb_lines) + "\n"
)


print("Clean target:", TARGET_HEAVY_PATH)

print(
    "Removed components:",
    dict(removed) if removed else "none",
)

print(
    "Removed polymer hydrogens:",
    removed_hydrogens,
)

Clean target: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/target_heavy.pdb
Removed components: none
Removed polymer hydrogens: 4225


In [35]:
#@title 17. Repair missing heavy atoms

from pdbfixer import PDBFixer
from openmm.app import PDBFile


TARGET_FIXED_PATH = (
    TARGET_PREP_DIR
    / "target_fixed_heavy.pdb"
)


fixer = PDBFixer(
    filename=str(TARGET_HEAVY_PATH)
)


# Missing residues are inspected but never created automatically.
fixer.findMissingResidues()

missing_residues = {
    key: list(names)
    for key, names
    in fixer.missingResidues.items()
}

missing_residue_count = sum(
    len(names)
    for names in missing_residues.values()
)


# Critical:
# addMissingAtoms() also creates anything left here.
fixer.missingResidues = {}


fixer.findNonstandardResidues()

nonstandard_residues = [
    (
        residue.chain.id,
        residue.id,
        residue.name,
        replacement,
    )
    for (
        residue,
        replacement,
    ) in fixer.nonstandardResidues
]


if nonstandard_residues:
    lines = "\n".join(
        f"- chain {chain}, residue {number}: "
        f"{name} → suggested {replacement}"
        for (
            chain,
            number,
            name,
            replacement,
        ) in nonstandard_residues
    )

    raise ValueError(
        "Non-standard polymer residues were detected:\n"
        + lines
        + "\n\nThey will not be mutated automatically."
    )


fixer.findMissingAtoms()


missing_internal_atoms = sum(
    len(atoms)
    for atoms
    in fixer.missingAtoms.values()
)

missing_terminal_atoms = sum(
    len(atoms)
    for atoms
    in fixer.missingTerminals.values()
)


print(
    "Missing residues:",
    missing_residue_count,
)

print(
    "Missing internal heavy atoms:",
    missing_internal_atoms,
)

print(
    "Missing terminal heavy atoms:",
    missing_terminal_atoms,
)


if missing_residue_count:
    print(
        "\nMissing residues were detected but "
        "will NOT be built automatically."
    )


fixer.addMissingAtoms()


with TARGET_FIXED_PATH.open("w") as handle:
    PDBFile.writeFile(
        fixer.topology,
        fixer.positions,
        handle,
        keepIds=True,
    )


print(
    "\nHeavy-atom-complete target:",
    TARGET_FIXED_PATH,
)

Missing residues: 0
Missing internal heavy atoms: 0
Missing terminal heavy atoms: 0

Heavy-atom-complete target: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/target_fixed_heavy.pdb


In [36]:
#@title 18. Detect disulfide bonds

from itertools import combinations
import json


DISULFIDE_PATH = (
    TARGET_PREP_DIR
    / "disulfides.json"
)


structure = gemmi.read_structure(
    str(TARGET_FIXED_PATH)
)

model = structure[0]

cysteines = []


for chain in model:
    for residue in chain:
        residue_name = (
            residue.name.strip().upper()
        )

        if residue_name not in {
            "CYS",
            "CYX",
        }:
            continue

        sulfur = residue.find_atom(
            "SG",
            "*",
            gemmi.Element("S"),
        )

        if sulfur is None:
            continue

        cysteines.append(
            {
                "chain": chain.name,
                "residue_number": residue.seqid.num,
                "insertion_code": (
                    residue.seqid.icode.strip()
                ),
                "position": np.array(
                    [
                        sulfur.pos.x,
                        sulfur.pos.y,
                        sulfur.pos.z,
                    ],
                    dtype=float,
                ),
            }
        )


candidates = []


for first, second in combinations(
    cysteines,
    2,
):
    distance = float(
        np.linalg.norm(
            first["position"]
            - second["position"]
        )
    )

    if distance <= DISULFIDE_MAX_DISTANCE_A:
        candidates.append(
            (
                distance,
                first,
                second,
            )
        )


# Shortest plausible S-S pairs are assigned first.
# One cysteine cannot belong to two disulfides.
candidates.sort(
    key=lambda item: item[0]
)

used_cysteines = set()
disulfides = []


def cysteine_id(cysteine):
    return (
        cysteine["chain"],
        cysteine["residue_number"],
        cysteine["insertion_code"],
    )


for distance, first, second in candidates:
    first_id = cysteine_id(first)
    second_id = cysteine_id(second)

    if (
        first_id in used_cysteines
        or second_id in used_cysteines
    ):
        continue

    used_cysteines.add(first_id)
    used_cysteines.add(second_id)

    disulfides.append(
        {
            "chain_1": first["chain"],
            "residue_1": first["residue_number"],
            "icode_1": first["insertion_code"],
            "chain_2": second["chain"],
            "residue_2": second["residue_number"],
            "icode_2": second["insertion_code"],
            "distance_A": distance,
        }
    )


DISULFIDE_PATH.write_text(
    json.dumps(
        disulfides,
        indent=2,
    )
)


print(
    "Disulfide bonds detected:",
    len(disulfides),
)


for bond in disulfides:
    print(
        f"{bond['chain_1']}:{bond['residue_1']}"
        f"{bond['icode_1']}  —  "
        f"{bond['chain_2']}:{bond['residue_2']}"
        f"{bond['icode_2']}   "
        f"{bond['distance_A']:.3f} Å"
    )


if not disulfides:
    print(
        "No Cys–Cys pairs within "
        f"{DISULFIDE_MAX_DISTANCE_A:.2f} Å."
    )

Disulfide bonds detected: 0
No Cys–Cys pairs within 2.30 Å.


In [37]:
#@title 19. Normalize target for Amber

import os

TARGET_AMBER_HEAVY_PATH = (
    TARGET_PREP_DIR
    / "target_amber_heavy.pdb"
)

PDB4AMBER_LOG = (
    LOG_DIR
    / "pdb4amber.log"
)

# Убедимся, что путь к бинарникам Miniconda прописан в PATH для поиска pdb4amber
conda_bin_dir = "/usr/local/conda/bin"
if conda_bin_dir not in os.environ.get("PATH", ""):
    os.environ["PATH"] = f"{conda_bin_dir}:{os.environ.get('PATH', '')}"

# Remove a possible partial file left by a failed run.
TARGET_AMBER_HEAVY_PATH.unlink(
    missing_ok=True
)

pdb4amber_output = run_command(
    [
        "pdb4amber",
        "-i",
        str(TARGET_FIXED_PATH),
        "-o",
        str(TARGET_AMBER_HEAVY_PATH),
    ],
    cwd=TARGET_PREP_DIR,
    log_path=PDB4AMBER_LOG,
)

if not TARGET_AMBER_HEAVY_PATH.is_file():
    raise RuntimeError(
        "pdb4amber finished without creating "
        f"{TARGET_AMBER_HEAVY_PATH}"
    )

print()
print(
    "Amber-normalized target:",
    TARGET_AMBER_HEAVY_PATH,
)

print(
    "Hydrogens will be added in the next step "
    f"at pH {TARGET_PH:g}."
)

$ pdb4amber -i /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/target_fixed_heavy.pdb -o /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/target_amber_heavy.pdb

Amber-normalized target: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/target_amber_heavy.pdb
Hydrogens will be added in the next step at pH 7.4.


In [38]:
#@title 20. Determine target protonation states

import json

import gemmi

from openmm.app import ForceField, Modeller, PDBFile


# Important:
# this file remains HEAVY-ATOM ONLY.
# LEaP will create Amber-compatible hydrogens later.
TARGET_AMBER_PATH = (
    TARGET_PREP_DIR
    / "target_leap_ready.pdb"
)

PROTONATION_PATH = (
    TARGET_PREP_DIR
    / "protonation.json"
)


PROTEIN_XML = {
    "ff19SB": "amber19/protein.ff19SB.xml",
    "ff14SB": "amber14/protein.ff14SB.xml",
}

DNA_XML = {
    "OL21": "amber19/DNA.OL21.xml",
    "OL15": "amber14/DNA.OL15.xml",
    "bsc1": "amber14/DNA.bsc1.xml",
}

RNA_XML = {
    "OL3": "amber14/RNA.OL3.xml",
}

WATER_XML = {
    "TIP3P": "amber19/tip3p.xml",
    "OPC": "amber19/opc.xml",
    "SPCE": "amber19/spce.xml",
}


def build_protonation_forcefield() -> ForceField:
    files = [
        PROTEIN_XML[PROTEIN_FORCE_FIELD],
        DNA_XML[DNA_FORCE_FIELD],
        RNA_XML[RNA_FORCE_FIELD],
        WATER_XML[WATER_MODEL],
    ]

    print("Force-field definitions used for protonation:")

    for filename in files:
        print(f"  {filename}")

    return ForceField(*files)


forcefield = build_protonation_forcefield()


source = PDBFile(
    str(TARGET_AMBER_HEAVY_PATH)
)

probe = Modeller(
    source.topology,
    source.positions,
)


# We use OpenMM only to decide protonation variants.
# The generated hydrogen coordinates are deliberately discarded.
selected_variants = probe.addHydrogens(
    forcefield=forcefield,
    pH=TARGET_PH,
)


topology_residues = list(
    probe.topology.residues()
)


if (
    len(selected_variants)
    != len(topology_residues)
):
    raise RuntimeError(
        "OpenMM returned an inconsistent "
        "protonation-state list."
    )


unmatched = forcefield.getUnmatchedResidues(
    probe.topology
)


if unmatched:
    details = "\n".join(
        (
            f"- chain {residue.chain.id}, "
            f"residue {residue.id} "
            f"{residue.name}"
        )
        for residue in unmatched
    )

    raise RuntimeError(
        "Some target residues are not described "
        "by the selected force fields:\n"
        + details
    )


# Now reopen the HEAVY-ATOM structure.
# We change residue names only; no OpenMM hydrogen
# coordinates are copied into the Amber input.
structure = gemmi.read_structure(
    str(TARGET_AMBER_HEAVY_PATH)
)

structure_residues = [
    residue
    for chain in structure[0]
    for residue in chain
]


if (
    len(structure_residues)
    != len(selected_variants)
):
    raise RuntimeError(
        "Residue count differs between OpenMM "
        "and the structural file."
    )


amber_variants = {
    "ASH",
    "CYM",
    "CYX",
    "GLH",
    "HID",
    "HIE",
    "HIP",
    "HIN",
    "LYN",
}


protonation_records = []


for structure_residue, topology_residue, variant in zip(
    structure_residues,
    topology_residues,
    selected_variants,
):
    if variant is None:
        continue

    protonation_records.append(
        {
            "chain": topology_residue.chain.id,
            "residue_number": topology_residue.id,
            "original_name": topology_residue.name,
            "variant": variant,
        }
    )

    if variant in amber_variants:
        structure_residue.name = variant


# Defensive check: this Amber input must contain no H.
for chain in structure[0]:
    for residue in chain:
        for atom_index in range(
            len(residue) - 1,
            -1,
            -1,
        ):
            if (
                residue[atom_index]
                .element.atomic_number
                == 1
            ):
                del residue[atom_index]


structure.write_pdb(
    str(TARGET_AMBER_PATH)
)


PROTONATION_PATH.write_text(
    json.dumps(
        protonation_records,
        indent=2,
    )
)


print()
print("Selected non-default protonation states")
print("---------------------------------------")

interesting = [
    record
    for record in protonation_records
    if record["variant"] in amber_variants
]


if interesting:
    for record in interesting:
        print(
            f"chain {record['chain']:<3} "
            f"residue {record['residue_number']:<5} "
            f"{record['original_name']} "
            f"→ {record['variant']}"
        )
else:
    print("none")


print()
print("LEaP-ready target:", TARGET_AMBER_PATH)
print("Hydrogens in this file: 0")
print("Protonation record:", PROTONATION_PATH)

Force-field definitions used for protonation:
  amber19/protein.ff19SB.xml
  amber19/DNA.OL21.xml
  amber14/RNA.OL3.xml
  amber19/opc.xml

Selected non-default protonation states
---------------------------------------
chain     residue 24    HIS → HID
chain     residue 98    HIS → HID
chain     residue 120   HIS → HID
chain     residue 271   HIS → HID
chain     residue 363   HIS → HID
chain     residue 386   HIS → HID
chain     residue 409   HIS → HID
chain     residue 424   HIS → HIE
chain     residue 437   HIS → HIE
chain     residue 466   HIS → HID
chain     residue 512   HIS → HID
chain     residue 549   HIS → HID
chain     residue 559   HIS → HIE

LEaP-ready target: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/target_leap_ready.pdb
Hydrogens in this file: 0
Protonation record: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/protonation.json


In [39]:
#@title 21. Audit target geometry

import json
from collections import defaultdict

import numpy as np
from openmm import unit
from openmm.app import PDBFile
from scipy.spatial import cKDTree


GEOMETRY_AUDIT_PATH = (
    TARGET_PREP_DIR
    / "geometry_audit.json"
)

TRUE_CLASH_DISTANCE_A = 1.50


def angle_degrees(a, b, c) -> float:
    ab = a - b
    cb = c - b

    cosine = np.dot(ab, cb) / (
        np.linalg.norm(ab)
        * np.linalg.norm(cb)
    )

    return float(
        np.degrees(
            np.arccos(
                np.clip(cosine, -1.0, 1.0)
            )
        )
    )


pdb = PDBFile(
    str(TARGET_AMBER_PATH)
)

topology = pdb.topology

positions_A = np.asarray(
    pdb.positions.value_in_unit(
        unit.angstrom
    ),
    dtype=float,
)


atoms = list(
    topology.atoms()
)

residues = list(
    topology.residues()
)


residue_index = {
    residue: index
    for index, residue
    in enumerate(residues)
}


# ------------------------------------------------------------------
# Covalent graph
# ------------------------------------------------------------------

adjacency = defaultdict(set)

for atom_a, atom_b in topology.bonds():
    adjacency[atom_a.index].add(
        atom_b.index
    )

    adjacency[atom_b.index].add(
        atom_a.index
    )


# Pairs separated by one, two, or three bonds are not ordinary
# nonbonded clash pairs.
excluded_pairs = set()


for atom in atoms:
    start = atom.index

    visited = {
        start
    }

    frontier = {
        start
    }

    for _ in range(3):

        next_frontier = set()

        for current in frontier:
            next_frontier.update(
                adjacency[current]
            )

        next_frontier -= visited

        for other in next_frontier:
            excluded_pairs.add(
                tuple(
                    sorted(
                        (
                            start,
                            other,
                        )
                    )
                )
            )

        visited.update(
            next_frontier
        )

        frontier = next_frontier


# ------------------------------------------------------------------
# Gross protein backbone geometry
#
# These intentionally broad ranges are not intended to replace
# MolProbity or crystallographic validation. They only detect
# geometries too distorted to enter MD untreated.
# ------------------------------------------------------------------

backbone_outliers = []


def record_outlier(
    residue_indices,
    label,
    value,
    expected,
):
    backbone_outliers.append(
        {
            "residue_indices": list(
                residue_indices
            ),
            "metric": label,
            "value": float(value),
            "expected": expected,
        }
    )


chains = defaultdict(list)

for index, residue in enumerate(
    residues
):
    chains[
        residue.chain.id
    ].append(
        (
            index,
            residue,
        )
    )


for chain_id, chain_residues in chains.items():

    for local_index, (
        global_index,
        residue,
    ) in enumerate(chain_residues):

        atom_by_name = {
            atom.name: atom
            for atom in residue.atoms()
        }

        # Only protein-like residues have this backbone.
        required = {
            "N",
            "CA",
            "C",
            "O",
        }

        if not required.issubset(
            atom_by_name
        ):
            continue


        N = positions_A[
            atom_by_name["N"].index
        ]

        CA = positions_A[
            atom_by_name["CA"].index
        ]

        C = positions_A[
            atom_by_name["C"].index
        ]

        O = positions_A[
            atom_by_name["O"].index
        ]


        measurements = (
            (
                "N-CA bond",
                np.linalg.norm(N - CA),
                1.30,
                1.60,
                "1.30–1.60 Å",
            ),
            (
                "CA-C bond",
                np.linalg.norm(CA - C),
                1.35,
                1.65,
                "1.35–1.65 Å",
            ),
            (
                "C-O bond",
                np.linalg.norm(C - O),
                1.10,
                1.35,
                "1.10–1.35 Å",
            ),
            (
                "N-CA-C angle",
                angle_degrees(
                    N,
                    CA,
                    C,
                ),
                90.0,
                135.0,
                "90–135°",
            ),
            (
                "CA-C-O angle",
                angle_degrees(
                    CA,
                    C,
                    O,
                ),
                90.0,
                145.0,
                "90–145°",
            ),
        )


        for (
            metric,
            value,
            lower,
            upper,
            expected,
        ) in measurements:

            if not (
                lower
                <= value
                <= upper
            ):
                record_outlier(
                    [global_index],
                    metric,
                    value,
                    expected,
                )


        # Check the peptide connection to the next residue only if
        # OpenMM actually considers C(i)-N(i+1) covalently bonded.

        if (
            local_index + 1
            >= len(chain_residues)
        ):
            continue


        next_global_index, next_residue = (
            chain_residues[
                local_index + 1
            ]
        )


        next_atoms = {
            atom.name: atom
            for atom
            in next_residue.atoms()
        }


        if not {
            "N",
            "CA",
        }.issubset(
            next_atoms
        ):
            continue


        C_atom = atom_by_name["C"]
        next_N_atom = next_atoms["N"]


        if (
            next_N_atom.index
            not in adjacency[
                C_atom.index
            ]
        ):
            continue


        next_N = positions_A[
            next_N_atom.index
        ]

        next_CA = positions_A[
            next_atoms["CA"].index
        ]


        peptide_distance = float(
            np.linalg.norm(
                C - next_N
            )
        )

        ca_c_n_angle = (
            angle_degrees(
                CA,
                C,
                next_N,
            )
        )

        c_n_ca_angle = (
            angle_degrees(
                C,
                next_N,
                next_CA,
            )
        )


        if not (
            1.15
            <= peptide_distance
            <= 1.50
        ):
            record_outlier(
                [
                    global_index,
                    next_global_index,
                ],
                "peptide C-N bond",
                peptide_distance,
                "1.15–1.50 Å",
            )


        if not (
            90.0
            <= ca_c_n_angle
            <= 145.0
        ):
            record_outlier(
                [
                    global_index,
                    next_global_index,
                ],
                "CA-C-N(next) angle",
                ca_c_n_angle,
                "90–145°",
            )


        if not (
            90.0
            <= c_n_ca_angle
            <= 145.0
        ):
            record_outlier(
                [
                    global_index,
                    next_global_index,
                ],
                "C(prev)-N-CA angle",
                c_n_ca_angle,
                "90–145°",
            )


# ------------------------------------------------------------------
# True nonbonded clashes
# ------------------------------------------------------------------

heavy_atoms = [
    atom
    for atom in atoms
    if (
        atom.element is not None
        and atom.element.atomic_number > 1
    )
]


heavy_xyz = np.asarray(
    [
        positions_A[
            atom.index
        ]
        for atom in heavy_atoms
    ],
    dtype=float,
)


tree = cKDTree(
    heavy_xyz
)


true_clashes = []


for local_i, local_j in tree.query_pairs(
    r=TRUE_CLASH_DISTANCE_A
):

    atom_i = heavy_atoms[
        local_i
    ]

    atom_j = heavy_atoms[
        local_j
    ]


    pair = tuple(
        sorted(
            (
                atom_i.index,
                atom_j.index,
            )
        )
    )


    if pair in excluded_pairs:
        continue


    distance = float(
        np.linalg.norm(
            heavy_xyz[local_i]
            - heavy_xyz[local_j]
        )
    )


    true_clashes.append(
        {
            "distance_A": distance,

            "atom_1": {
                "residue_index": (
                    residue_index[
                        atom_i.residue
                    ]
                ),
                "chain": (
                    atom_i.residue.chain.id
                ),
                "residue": (
                    atom_i.residue.name
                ),
                "number": (
                    atom_i.residue.id
                ),
                "atom": atom_i.name,
            },

            "atom_2": {
                "residue_index": (
                    residue_index[
                        atom_j.residue
                    ]
                ),
                "chain": (
                    atom_j.residue.chain.id
                ),
                "residue": (
                    atom_j.residue.name
                ),
                "number": (
                    atom_j.residue.id
                ),
                "atom": atom_j.name,
            },
        }
    )


true_clashes.sort(
    key=lambda item: item["distance_A"]
)


# ------------------------------------------------------------------
# Residues that require local relaxation
# ------------------------------------------------------------------

repair_indices = set()


for issue in backbone_outliers:
    repair_indices.update(
        issue[
            "residue_indices"
        ]
    )


for clash in true_clashes:
    repair_indices.add(
        clash[
            "atom_1"
        ][
            "residue_index"
        ]
    )

    repair_indices.add(
        clash[
            "atom_2"
        ][
            "residue_index"
        ]
    )


repair_residues = [
    {
        "index": index,
        "chain": residues[
            index
        ].chain.id,
        "number": residues[
            index
        ].id,
        "name": residues[
            index
        ].name,
    }
    for index in sorted(
        repair_indices
    )
]


audit = {
    "backbone_outliers":
        backbone_outliers,

    "true_nonbonded_clashes":
        true_clashes,

    "repair_residues":
        repair_residues,

    "repair_required":
        bool(
            backbone_outliers
            or true_clashes
        ),
}


GEOMETRY_AUDIT_PATH.write_text(
    json.dumps(
        audit,
        indent=2,
    )
)


print("Target geometry audit")
print("---------------------")

print(
    "Backbone outliers:",
    len(backbone_outliers),
)

print(
    "True nonbonded clashes:",
    len(true_clashes),
)


if backbone_outliers:

    print()
    print("Backbone geometry")
    print("-----------------")


    for issue in backbone_outliers:

        affected = ", ".join(
            (
                f"{residues[index].chain.id}:"
                f"{residues[index].name}"
                f"{residues[index].id}"
            )
            for index in issue[
                "residue_indices"
            ]
        )


        print(
            f"{affected:<25} "
            f"{issue['metric']}: "
            f"{issue['value']:.2f} "
            f"(expected {issue['expected']})"
        )


if true_clashes:

    print()
    print("True nonbonded clashes")
    print("----------------------")


    for clash in true_clashes:

        a = clash["atom_1"]
        b = clash["atom_2"]


        print(
            f"{clash['distance_A']:.3f} Å   "
            f"{a['chain']}:{a['residue']}"
            f"{a['number']}:{a['atom']} "
            "— "
            f"{b['chain']}:{b['residue']}"
            f"{b['number']}:{b['atom']}"
        )


print()
print(
    "Local repair required:",
    audit["repair_required"],
)

print(
    "Audit:",
    GEOMETRY_AUDIT_PATH,
)

Target geometry audit
---------------------
Backbone outliers: 0
True nonbonded clashes: 0

Local repair required: False
Audit: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/geometry_audit.json


In [40]:
#@title 22. Build structural repair plan

import hashlib
import json
from collections import defaultdict


REFERENCE_DIR = (
    WORKDIR
    / "references"
)

REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


REPAIR_PLAN_PATH = (
    TARGET_PREP_DIR
    / "structural_repair_plan.json"
)


MAX_GAP_INSIDE_DAMAGE_REGION = 2

REFERENCE_MIN_FLAGGED_RESIDUES = 3
REFERENCE_MIN_BACKBONE_OUTLIERS = 3


def sha256_file(path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


audit = json.loads(
    GEOMETRY_AUDIT_PATH.read_text()
)


if not audit["repair_required"]:

    repair_plan = {
        "repair_required": False,
        "target_file": str(
            TARGET_AMBER_PATH
        ),
        "target_sha256": (
            sha256_file(
                TARGET_AMBER_PATH
            )
        ),
        "segments": [],
    }


else:

    problem_residues = sorted(
        audit["repair_residues"],
        key=lambda item: item["index"],
    )


    by_chain = defaultdict(list)


    for residue in problem_residues:
        by_chain[
            residue["chain"]
        ].append(
            residue
        )


    raw_segments = []


    for chain_id, residues in by_chain.items():

        if not residues:
            continue


        current = [
            residues[0]
        ]


        for residue in residues[1:]:

            previous = current[-1]

            gap = (
                int(residue["index"])
                - int(previous["index"])
            )


            if (
                gap
                <= MAX_GAP_INSIDE_DAMAGE_REGION + 1
            ):
                current.append(
                    residue
                )

            else:
                raw_segments.append(
                    current
                )

                current = [
                    residue
                ]


        raw_segments.append(
            current
        )


    segments = []


    for segment_id, residues in enumerate(
        raw_segments,
        start=1,
    ):

        chain = residues[0][
            "chain"
        ]


        indices = {
            int(residue["index"])
            for residue in residues
        }


        relevant_outliers = []


        for issue in audit[
            "backbone_outliers"
        ]:

            issue_indices = {
                int(index)
                for index
                in issue[
                    "residue_indices"
                ]
            }


            if (
                indices
                & issue_indices
            ):
                relevant_outliers.append(
                    issue
                )


        unique_outliers = []

        seen = set()


        for issue in relevant_outliers:

            key = (
                tuple(
                    sorted(
                        int(index)
                        for index
                        in issue[
                            "residue_indices"
                        ]
                    )
                ),
                issue["metric"],
                round(
                    float(
                        issue["value"]
                    ),
                    6,
                ),
            )


            if key in seen:
                continue


            seen.add(key)

            unique_outliers.append(
                issue
            )


        if (
            len(residues)
            >= REFERENCE_MIN_FLAGGED_RESIDUES
            or len(unique_outliers)
            >= REFERENCE_MIN_BACKBONE_OUTLIERS
        ):

            repair_route = (
                "reference_required"
            )

        else:

            repair_route = (
                "local_review"
            )


        segments.append(
            {
                "segment_id":
                    segment_id,

                "chain":
                    chain,

                "start_index":
                    min(indices),

                "end_index":
                    max(indices),

                "flagged_residues":
                    residues,

                "backbone_outlier_count":
                    len(
                        unique_outliers
                    ),

                "route":
                    repair_route,
            }
        )


    repair_plan = {
        "repair_required": True,

        "target_file": str(
            TARGET_AMBER_PATH
        ),

        "target_sha256": (
            sha256_file(
                TARGET_AMBER_PATH
            )
        ),

        "segments":
            segments,
    }


REPAIR_PLAN_PATH.write_text(
    json.dumps(
        repair_plan,
        indent=2,
    )
)


print("Structural repair plan")
print("----------------------")


if not repair_plan["repair_required"]:

    print(
        "No structural repair required."
    )

else:

    for segment in repair_plan[
        "segments"
    ]:

        residues = (
            segment[
                "flagged_residues"
            ]
        )


        labels = ", ".join(
            (
                f"{item['name']}"
                f"{item['number']}"
            )
            for item in residues
        )


        print(
            f"Segment "
            f"{segment['segment_id']}: "
            f"chain "
            f"{segment['chain']}"
        )

        print(
            "  flagged:",
            labels,
        )

        print(
            "  backbone outliers:",
            segment[
                "backbone_outlier_count"
            ],
        )

        print(
            "  route:",
            segment["route"],
        )

        print()


reference_required = any(
    segment["route"]
    == "reference_required"
    for segment in repair_plan[
        "segments"
    ]
)


print(
    "Reference required:",
    reference_required,
)

print(
    "Reference directory:",
    REFERENCE_DIR,
)

print(
    "Repair plan:",
    REPAIR_PLAN_PATH,
)

Structural repair plan
----------------------
No structural repair required.
Reference required: False
Reference directory: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/references
Repair plan: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/structural_repair_plan.json


In [41]:
#@title 23. Find reference and plan closed structural graft

import json
from pathlib import Path

import gemmi
import numpy as np


REFERENCE_APPLY_PATH = (
    TARGET_PREP_DIR
    / "reference_applied.json"
)
REFERENCE_MATCH_PATH = (
    TARGET_PREP_DIR
    / "reference_match.json"
)

CANONICAL_RESIDUE_NAME = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY",
    "HIS", "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER",
    "THR", "TRP", "TYR", "VAL", "HID", "HIE", "HIP", "CYX", "ASH", "GLH"
}
PROTEIN_RESIDUES = CANONICAL_RESIDUE_NAME


def apply_transformation(
    residue: gemmi.Residue,
    rotation: np.ndarray,
    translation: np.ndarray,
):
    for atom in residue:
        pos = np.array(
            [
                atom.pos.x,
                atom.pos.y,
                atom.pos.z,
            ],
            dtype=float,
        )

        new_pos = pos @ rotation + translation

        atom.pos.x = float(new_pos[0])
        atom.pos.y = float(new_pos[1])
        atom.pos.z = float(new_pos[2])


# Если файл сопоставления отсутствует, мягко пропускаем этап Grafting
if not REFERENCE_MATCH_PATH.exists():
    print(f"⚠️ Файл {REFERENCE_MATCH_PATH.name} не найден. Пропускаем reference-guided grafting.")

    REFERENCE_APPLY_PATH.write_text(
        json.dumps(
            {
                "applied": False,
                "segments_grafted": 0,
                "reason": "reference_match.json not found (skipped)",
            },
            indent=2,
        )
    )
    segments = []
else:
    match_data = json.loads(REFERENCE_MATCH_PATH.read_text())
    segments = match_data.get("segments", [])


if not segments:
    print("No reference-guided grafts to apply. Moving forward.")
    if not REFERENCE_APPLY_PATH.exists():
        REFERENCE_APPLY_PATH.write_text(
            json.dumps(
                {
                    "applied": False,
                    "segments_grafted": 0,
                },
                indent=2,
            )
        )

else:
    target_structure = gemmi.read_structure(str(TARGET_AMBER_PATH))
    references_cache = {}
    grafted_count = 0

    for match in segments:
        segment_id = match["segment_id"]
        target_chain_name = match["target_chain"]
        reference_file = match["reference_file"]
        reference_chain_name = match["reference_chain"]

        target_graft_start = match["target_graft_start"]
        target_graft_end = match["target_graft_end"]
        ref_graft_start = match["reference_graft_start"]
        ref_graft_end = match["reference_graft_end"]

        rotation = np.array(match["rotation"], dtype=float)
        translation = np.array(match["translation_A"], dtype=float)

        if reference_file not in references_cache:
            references_cache[reference_file] = gemmi.read_structure(reference_file)

        ref_structure = references_cache[reference_file]

        target_chain = None
        for chain in target_structure[0]:
            if chain.name == target_chain_name:
                target_chain = chain
                break

        if target_chain is None:
            raise RuntimeError(f"Target chain {target_chain_name} not found in structure.")

        ref_chain = None
        for chain in ref_structure[0]:
            if chain.name == reference_chain_name:
                ref_chain = chain
                break

        if ref_chain is None:
            raise RuntimeError(f"Reference chain {reference_chain_name} not found in {reference_file}.")

        target_entries = [r for r in target_chain if r.name.strip().upper() in PROTEIN_RESIDUES]
        ref_entries = [r for r in ref_chain if r.name.strip().upper() in PROTEIN_RESIDUES]

        if target_graft_start < 0 or target_graft_end >= len(target_entries):
            raise IndexError("Target graft slice indices out of bounds.")

        if ref_graft_start < 0 or ref_graft_end >= len(ref_entries):
            raise IndexError("Reference graft slice indices out of bounds.")

        donor_residues_to_graft = ref_entries[ref_graft_start : ref_graft_end + 1]

        first_target_res = target_entries[target_graft_start]
        last_target_res = target_entries[target_graft_end]

        remove_start_idx = None
        remove_end_idx = None

        chain_residues = list(target_chain)
        for idx, res in enumerate(chain_residues):
            if res == first_target_res:
                remove_start_idx = idx
            if res == last_target_res:
                remove_end_idx = idx
                break

        if remove_start_idx is None or remove_end_idx is None:
            raise RuntimeError("Could not locate target residues within chain sequence.")

        new_residues = []
        for ref_res in donor_residues_to_graft:
            cloned_res = ref_res.clone()
            apply_transformation(cloned_res, rotation, translation)
            new_residues.append(cloned_res)

        del target_chain[remove_start_idx : remove_end_idx + 1]

        for offset, res in enumerate(new_residues):
            target_chain.insert(remove_start_idx + offset, res)

        grafted_count += 1
        print(f"Successfully grafted segment {segment_id} ({len(new_residues)} residues inserted).")

    target_structure.remove_waters()
    target_structure.write_pdb(str(TARGET_AMBER_PATH))

    REFERENCE_APPLY_PATH.write_text(
        json.dumps(
            {
                "applied": True,
                "segments_grafted": grafted_count,
            },
            indent=2,
        )
    )

    print()
    print("Reference graft application complete.")
    print("Updated structure written to:", TARGET_AMBER_PATH)
    print("Record:", REFERENCE_APPLY_PATH)

⚠️ Файл reference_match.json не найден. Пропускаем reference-guided grafting.
No reference-guided grafts to apply. Moving forward.


In [42]:
# @title 24. Apply validated structural graft

import json
import shutil
from pathlib import Path

import gemmi
import numpy as np


TARGET_PRE_REPAIR_BACKUP = (
    TARGET_PREP_DIR / "target_before_reference_repair.pdb"
)

REFERENCE_REPAIR_RECORD = TARGET_PREP_DIR / "reference_repair.json"


CANONICAL_RESIDUE_NAME = {
    "ASH": "ASP",
    "GLH": "GLU",
    "CYM": "CYS",
    "CYX": "CYS",
    "HID": "HIS",
    "HIE": "HIS",
    "HIP": "HIS",
    "HIN": "HIS",
    "LYN": "LYS",
}


PROTEIN_RESIDUES = {
    "ALA",
    "ARG",
    "ASN",
    "ASP",
    "CYS",
    "GLN",
    "GLU",
    "GLY",
    "HIS",
    "ILE",
    "LEU",
    "LYS",
    "MET",
    "PHE",
    "PRO",
    "SER",
    "THR",
    "TRP",
    "TYR",
    "VAL",
}


def canonical_name(residue_name: str) -> str:
    name = residue_name.strip().upper()
    return CANONICAL_RESIDUE_NAME.get(name, name)


def protein_chain_entries(
    structure: gemmi.Structure, chain_name: str
) -> list[dict]:
    entries = []
    for chain in structure[0]:
        if chain.name != chain_name:
            continue
        for residue in chain:
            canonical = canonical_name(residue.name)
            if canonical not in PROTEIN_RESIDUES:
                continue
            entries.append(
                {
                    "chain": chain.name,
                    "residue": residue,
                    "name": canonical,
                }
            )
    return entries


def residue_atom_xyz(entry: dict, atom_name: str) -> np.ndarray:
    residue = entry["residue"]
    for atom in residue:
        if atom.name.strip() == atom_name:
            return np.array(
                [atom.pos.x, atom.pos.y, atom.pos.z], dtype=float
            )
    raise RuntimeError(
        f"Missing atom {atom_name} in {entry['chain']}:{residue.name}{residue.seqid.num}"
    )


# Чистая обработка опционального артефакта графтинга
if REFERENCE_MATCH_PATH.is_file():
    matches = json.loads(REFERENCE_MATCH_PATH.read_text())
else:
    matches = {"segments": []}


if not matches.get("segments"):
    print("No reference-guided graft required. Target structure left intact.")

else:
    target = gemmi.read_structure(str(TARGET_AMBER_PATH))
    reference_cache = {}

    for match in matches["segments"]:
        path = Path(match["reference_file"])
        if not path.is_file():
            raise FileNotFoundError(f"Reference disappeared: {path}")

        if path not in reference_cache:
            reference_cache[path] = gemmi.read_structure(str(path))

    if not TARGET_PRE_REPAIR_BACKUP.exists():
        shutil.copy2(TARGET_AMBER_PATH, TARGET_PRE_REPAIR_BACKUP)

    repair_records = []

    for match in matches["segments"]:
        reference_path = Path(match["reference_file"])
        reference = reference_cache[reference_path]

        target_entries = protein_chain_entries(
            target, match["target_chain"]
        )
        donor_entries = protein_chain_entries(
            reference, match["reference_chain"]
        )

        target_start = int(match["target_graft_start"])
        target_end = int(match["target_graft_end"])
        donor_start = int(match["reference_graft_start"])
        donor_end = int(match["reference_graft_end"])

        target_segment = target_entries[target_start : target_end + 1]
        donor_segment = donor_entries[donor_start : donor_end + 1]

        if len(target_segment) != len(donor_segment):
            raise RuntimeError("Validated graft lengths no longer match.")

        target_sequence = tuple(entry["name"] for entry in target_segment)
        donor_sequence = tuple(entry["name"] for entry in donor_segment)

        if target_sequence != donor_sequence:
            raise RuntimeError("Validated graft sequence no longer matches.")

        rotation = np.asarray(match["rotation"], dtype=float)
        translation = np.asarray(match["translation_A"], dtype=float)

        replaced_atoms = 0

        for target_entry, donor_entry in zip(
            target_segment, donor_segment
        ):
            target_residue = target_entry["residue"]
            donor_residue = donor_entry["residue"]

            target_residue.remove_hydrogens()

            donor_atoms = {
                atom.name.strip(): atom
                for atom in donor_residue
                if atom.element.atomic_number > 1
            }

            for target_atom in target_residue:
                if target_atom.element.atomic_number <= 1:
                    continue

                atom_name = target_atom.name.strip()
                donor_atom = donor_atoms.get(atom_name)

                if donor_atom is None:
                    raise RuntimeError(
                        f"Donor lacks required heavy atom {atom_name} "
                        f"for residue {target_residue.name}{target_residue.seqid.num}."
                    )

                donor_xyz = np.array(
                    [donor_atom.pos.x, donor_atom.pos.y, donor_atom.pos.z],
                    dtype=float,
                )
                transformed = donor_xyz @ rotation + translation

                target_atom.pos = gemmi.Position(
                    float(transformed[0]),
                    float(transformed[1]),
                    float(transformed[2]),
                )
                replaced_atoms += 1

        repair_records.append(
            {
                "segment_id": match["segment_id"],
                "reference_file": str(reference_path),
                "reference_chain": match["reference_chain"],
                "target_graft_start": target_start,
                "target_graft_end": target_end,
                "left_extension": match["left_extension"],
                "right_extension": match["right_extension"],
                "anchor_rmsd_A": match["anchor_rmsd_A"],
                "replaced_heavy_atoms": replaced_atoms,
                "junctions": match["junctions"],
            }
        )

    final_junctions = []

    for match in matches["segments"]:
        entries = protein_chain_entries(target, match["target_chain"])
        start = int(match["target_graft_start"])
        end = int(match["target_graft_end"])

        if start > 0:
            prev_res = entries[start - 1]["residue"]
            curr_res = entries[start]["residue"]

            if curr_res.seqid.num == prev_res.seqid.num + 1:
                distance = float(
                    np.linalg.norm(
                        residue_atom_xyz(entries[start - 1], "C")
                        - residue_atom_xyz(entries[start], "N")
                    )
                )
                final_junctions.append(
                    {
                        "segment_id": match["segment_id"],
                        "side": "left",
                        "distance_A": distance,
                    }
                )

        if end + 1 < len(entries):
            curr_res = entries[end]["residue"]
            next_res = entries[end + 1]["residue"]

            if next_res.seqid.num == curr_res.seqid.num + 1:
                distance = float(
                    np.linalg.norm(
                        residue_atom_xyz(entries[end], "C")
                        - residue_atom_xyz(entries[end + 1], "N")
                    )
                )
                final_junctions.append(
                    {
                        "segment_id": match["segment_id"],
                        "side": "right",
                        "distance_A": distance,
                    }
                )

    invalid = [
        item
        for item in final_junctions
        if not (1.15 <= item["distance_A"] <= 1.50)
    ]

    if invalid:
        raise RuntimeError(
            "Internal consistency error: a previously validated graft "
            "failed after application.\n\nThe target was NOT overwritten."
        )

    target.write_pdb(str(TARGET_AMBER_PATH))

    record = {
        "method": "adaptive_reference_graft",
        "segments": repair_records,
        "final_junctions": final_junctions,
        "backup": str(TARGET_PRE_REPAIR_BACKUP),
    }

    REFERENCE_REPAIR_RECORD.write_text(json.dumps(record, indent=2))

    print("Structural graft applied")
    print("------------------------")
    for segment in repair_records:
        print(f"Segment {segment['segment_id']}")
        print("  donor:", Path(segment["reference_file"]).name)
        print(
            "  extension:",
            f"{segment['left_extension']} left, {segment['right_extension']} right",
        )
        print("  anchor RMSD:", f"{segment['anchor_rmsd_A']:.3f} Å")
        print("  heavy atoms replaced:", segment["replaced_heavy_atoms"])
        print()

    print("Final peptide junctions")
    print("-----------------------")
    for junction in final_junctions:
        print(
            f"segment {junction['segment_id']} {junction['side']}: {junction['distance_A']:.3f} Å"
        )

    print()
    print("Validated graft written to:", TARGET_AMBER_PATH)

No reference-guided graft required. Target structure left intact.


In [43]:
#@title 25. Finalize target preparation

import hashlib
import json
from collections import defaultdict

import gemmi
import numpy as np

from openmm import unit
from openmm.app import PDBFile
from scipy.spatial import cKDTree


TARGET_READY_PATH = (
    TARGET_PREP_DIR
    / "target_ready.json"
)

FINAL_TARGET_AUDIT_PATH = (
    TARGET_PREP_DIR
    / "final_target_audit.json"
)


TRUE_CLASH_DISTANCE_A = 1.50

# Angle-only defects inside this many residues from either terminus
# may be deferred to the normal whole-system minimization stage.
TERMINAL_RELAXATION_WINDOW = 5


def sha256_file(path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def angle_degrees(a, b, c) -> float:

    ab = a - b
    cb = c - b

    denominator = (
        np.linalg.norm(ab)
        * np.linalg.norm(cb)
    )

    if denominator == 0:
        return float("nan")

    cosine = (
        np.dot(ab, cb)
        / denominator
    )

    return float(
        np.degrees(
            np.arccos(
                np.clip(
                    cosine,
                    -1.0,
                    1.0,
                )
            )
        )
    )


if not TARGET_AMBER_PATH.is_file():

    raise FileNotFoundError(
        f"Prepared target not found:\n"
        f"{TARGET_AMBER_PATH}"
    )


pdb = PDBFile(
    str(TARGET_AMBER_PATH)
)

topology = pdb.topology

positions_A = np.asarray(
    pdb.positions.value_in_unit(
        unit.angstrom
    ),
    dtype=float,
)


atoms = list(
    topology.atoms()
)

residues = list(
    topology.residues()
)


residue_index = {
    residue: index
    for index, residue
    in enumerate(residues)
}


# --------------------------------------------------------------
# Covalent graph
# --------------------------------------------------------------

adjacency = defaultdict(set)


for atom_a, atom_b in topology.bonds():

    adjacency[
        atom_a.index
    ].add(
        atom_b.index
    )

    adjacency[
        atom_b.index
    ].add(
        atom_a.index
    )


# Exclude 1-2, 1-3 and 1-4 atom pairs from ordinary
# nonbonded-clash analysis.

excluded_pairs = set()


for atom in atoms:

    start = atom.index

    visited = {
        start
    }

    frontier = {
        start
    }


    for _ in range(3):

        next_frontier = set()

        for current in frontier:
            next_frontier.update(
                adjacency[current]
            )

        next_frontier -= visited


        for other in next_frontier:

            excluded_pairs.add(
                tuple(
                    sorted(
                        (
                            start,
                            other,
                        )
                    )
                )
            )


        visited.update(
            next_frontier
        )

        frontier = next_frontier


# --------------------------------------------------------------
# Backbone geometry
# --------------------------------------------------------------

backbone_outliers = []


def add_backbone_issue(
    residue_indices,
    metric,
    value,
    expected,
):

    backbone_outliers.append(
        {
            "residue_indices":
                list(
                    residue_indices
                ),

            "metric":
                metric,

            "value":
                float(value),

            "expected":
                expected,
        }
    )


chains = defaultdict(list)


for index, residue in enumerate(
    residues
):

    chains[
        residue.chain.id
    ].append(
        (
            index,
            residue,
        )
    )


for chain_id, chain_residues in chains.items():

    for local_index, (
        global_index,
        residue,
    ) in enumerate(
        chain_residues
    ):

        atom_by_name = {
            atom.name: atom
            for atom
            in residue.atoms()
        }


        if not {
            "N",
            "CA",
            "C",
            "O",
        }.issubset(
            atom_by_name
        ):
            continue


        N = positions_A[
            atom_by_name["N"].index
        ]

        CA = positions_A[
            atom_by_name["CA"].index
        ]

        C = positions_A[
            atom_by_name["C"].index
        ]

        O = positions_A[
            atom_by_name["O"].index
        ]


        checks = (
            (
                "N-CA bond",
                np.linalg.norm(N - CA),
                1.30,
                1.60,
                "1.30–1.60 Å",
            ),
            (
                "CA-C bond",
                np.linalg.norm(CA - C),
                1.35,
                1.65,
                "1.35–1.65 Å",
            ),
            (
                "C-O bond",
                np.linalg.norm(C - O),
                1.10,
                1.35,
                "1.10–1.35 Å",
            ),
            (
                "N-CA-C angle",
                angle_degrees(
                    N,
                    CA,
                    C,
                ),
                90.0,
                135.0,
                "90–135°",
            ),
            (
                "CA-C-O angle",
                angle_degrees(
                    CA,
                    C,
                    O,
                ),
                90.0,
                145.0,
                "90–145°",
            ),
        )


        for (
            metric,
            value,
            lower,
            upper,
            expected,
        ) in checks:

            if not (
                lower
                <= value
                <= upper
            ):

                add_backbone_issue(
                    [global_index],
                    metric,
                    value,
                    expected,
                )


        if (
            local_index + 1
            >= len(chain_residues)
        ):
            continue


        (
            next_global_index,
            next_residue,
        ) = chain_residues[
            local_index + 1
        ]


        next_atoms = {
            atom.name: atom
            for atom
            in next_residue.atoms()
        }


        if not {
            "N",
            "CA",
        }.issubset(
            next_atoms
        ):
            continue


        current_c = (
            atom_by_name["C"]
        )

        next_n = (
            next_atoms["N"]
        )


        if (
            next_n.index
            not in adjacency[
                current_c.index
            ]
        ):
            continue


        next_N = positions_A[
            next_n.index
        ]

        next_CA = positions_A[
            next_atoms["CA"].index
        ]


        peptide_distance = float(
            np.linalg.norm(
                C - next_N
            )
        )


        ca_c_n = angle_degrees(
            CA,
            C,
            next_N,
        )


        c_n_ca = angle_degrees(
            C,
            next_N,
            next_CA,
        )


        if not (
            1.15
            <= peptide_distance
            <= 1.50
        ):

            add_backbone_issue(
                [
                    global_index,
                    next_global_index,
                ],
                "peptide C-N bond",
                peptide_distance,
                "1.15–1.50 Å",
            )


        if not (
            90.0
            <= ca_c_n
            <= 145.0
        ):

            add_backbone_issue(
                [
                    global_index,
                    next_global_index,
                ],
                "CA-C-N(next) angle",
                ca_c_n,
                "90–145°",
            )


        if not (
            90.0
            <= c_n_ca
            <= 145.0
        ):

            add_backbone_issue(
                [
                    global_index,
                    next_global_index,
                ],
                "C(prev)-N-CA angle",
                c_n_ca,
                "90–145°",
            )


# --------------------------------------------------------------
# True heavy-atom clashes
# --------------------------------------------------------------

heavy_atoms = [
    atom
    for atom in atoms
    if (
        atom.element is not None
        and atom.element.atomic_number > 1
    )
]


heavy_xyz = np.asarray(
    [
        positions_A[
            atom.index
        ]
        for atom in heavy_atoms
    ],
    dtype=float,
)


tree = cKDTree(
    heavy_xyz
)


true_clashes = []


for local_i, local_j in tree.query_pairs(
    r=TRUE_CLASH_DISTANCE_A
):

    atom_i = heavy_atoms[
        local_i
    ]

    atom_j = heavy_atoms[
        local_j
    ]


    pair = tuple(
        sorted(
            (
                atom_i.index,
                atom_j.index,
            )
        )
    )


    if pair in excluded_pairs:
        continue


    distance = float(
        np.linalg.norm(
            heavy_xyz[local_i]
            - heavy_xyz[local_j]
        )
    )


    true_clashes.append(
        {
            "distance_A":
                distance,

            "atom_1": {
                "chain":
                    atom_i.residue.chain.id,

                "residue":
                    atom_i.residue.name,

                "number":
                    atom_i.residue.id,

                "atom":
                    atom_i.name,
            },

            "atom_2": {
                "chain":
                    atom_j.residue.chain.id,

                "residue":
                    atom_j.residue.name,

                "number":
                    atom_j.residue.id,

                "atom":
                    atom_j.name,
            },
        }
    )


true_clashes.sort(
    key=lambda item:
        item["distance_A"]
)


# --------------------------------------------------------------
# Amber-state validation
# --------------------------------------------------------------

structure = gemmi.read_structure(
    str(TARGET_AMBER_PATH)
)


if len(structure) != 1:

    raise RuntimeError(
        "Prepared target must contain "
        "exactly one model."
    )


hydrogen_count = sum(
    1
    for chain in structure[0]
    for residue in chain
    for atom in residue
    if atom.element.atomic_number == 1
)


residue_names = [
    residue.name.strip().upper()
    for chain in structure[0]
    for residue in chain
]


generic_histidines = (
    residue_names.count(
        "HIS"
    )
)


stored_disulfides = json.loads(
    DISULFIDE_PATH.read_text()
)


cyx_count = (
    residue_names.count(
        "CYX"
    )
)


expected_cyx = (
    2
    * len(stored_disulfides)
)


# --------------------------------------------------------------
# Severity classification
# --------------------------------------------------------------

fatal_backbone_outliers = []
deferred_terminal_warnings = []


chain_positions = {}


for chain_id, chain_residues in chains.items():

    for local_index, (
        global_index,
        residue,
    ) in enumerate(
        chain_residues
    ):

        chain_positions[
            global_index
        ] = {
            "local_index":
                local_index,

            "chain_length":
                len(
                    chain_residues
                ),
        }


for issue in backbone_outliers:

    affected_indices = [
        int(index)
        for index
        in issue[
            "residue_indices"
        ]
    ]


    is_terminal = all(
        (
            chain_positions[
                index
            ][
                "local_index"
            ]
            < TERMINAL_RELAXATION_WINDOW
        )
        or
        (
            chain_positions[
                index
            ][
                "local_index"
            ]
            >= (
                chain_positions[
                    index
                ][
                    "chain_length"
                ]
                -
                TERMINAL_RELAXATION_WINDOW
            )
        )
        for index
        in affected_indices
    )


    is_angle_only = (
        "angle"
        in issue[
            "metric"
        ].lower()
    )


    if (
        is_terminal
        and is_angle_only
    ):

        deferred_terminal_warnings.append(
            issue
        )

    else:

        fatal_backbone_outliers.append(
            issue
        )


# --------------------------------------------------------------
# Write audit before any failure
# --------------------------------------------------------------

audit_record = {
    "backbone_outliers":
        backbone_outliers,

    "fatal_backbone_outliers":
        fatal_backbone_outliers,

    "deferred_terminal_warnings":
        deferred_terminal_warnings,

    "true_nonbonded_clashes":
        true_clashes,

    "hydrogens":
        hydrogen_count,

    "generic_histidines":
        generic_histidines,

    "cyx_count":
        cyx_count,

    "expected_cyx":
        expected_cyx,
}


FINAL_TARGET_AUDIT_PATH.write_text(
    json.dumps(
        audit_record,
        indent=2,
    )
)


print("Final target validation")
print("-----------------------")

print(
    "Backbone outliers:",
    len(
        backbone_outliers
    ),
)

print(
    "Fatal backbone outliers:",
    len(
        fatal_backbone_outliers
    ),
)

print(
    "Deferred terminal warnings:",
    len(
        deferred_terminal_warnings
    ),
)

print(
    "True nonbonded clashes:",
    len(
        true_clashes
    ),
)

print(
    "Hydrogens:",
    hydrogen_count,
)

print(
    "Disulfides:",
    len(
        stored_disulfides
    ),
)


if deferred_terminal_warnings:

    print()
    print("Deferred terminal geometry")
    print("--------------------------")


    for issue in (
        deferred_terminal_warnings
    ):

        labels = ", ".join(
            (
                f"{residues[index].chain.id}:"
                f"{residues[index].name}"
                f"{residues[index].id}"
            )
            for index
            in issue[
                "residue_indices"
            ]
        )


        print(
            f"{labels}: "
            f"{issue['metric']} = "
            f"{issue['value']:.2f}"
        )


if fatal_backbone_outliers:

    details = []


    for issue in (
        fatal_backbone_outliers
    ):

        labels = ", ".join(
            (
                f"{residues[index].chain.id}:"
                f"{residues[index].name}"
                f"{residues[index].id}"
            )
            for index
            in issue[
                "residue_indices"
            ]
        )


        details.append(
            f"- {labels}: "
            f"{issue['metric']} = "
            f"{issue['value']:.2f}"
        )


    raise RuntimeError(
        "Target still contains structural "
        "backbone defects:\n"
        + "\n".join(details)
    )


if true_clashes:

    raise RuntimeError(
        "Target still contains severe "
        "nonbonded heavy-atom clashes."
    )


if hydrogen_count != 0:

    raise RuntimeError(
        "LEaP-ready target must contain "
        "no hydrogens."
    )


if generic_histidines != 0:

    raise RuntimeError(
        "Generic HIS remains after "
        "protonation assignment."
    )


if cyx_count != expected_cyx:

    raise RuntimeError(
        "Disulfide-state mismatch:\n"
        f"expected CYX: {expected_cyx}\n"
        f"observed CYX: {cyx_count}"
    )


# --------------------------------------------------------------
# Downstream contract
# --------------------------------------------------------------

deferred_residue_indices = sorted(
    {
        int(index)
        for issue in deferred_terminal_warnings
        for index in issue[
            "residue_indices"
        ]
    }
)


deferred_residues = [
    {
        "index":
            index,

        "chain":
            residues[
                index
            ].chain.id,

        "name":
            residues[
                index
            ].name,

        "number":
            residues[
                index
            ].id,
    }
    for index
    in deferred_residue_indices
]


target_ready = {
    "status":
        "ready",

    "target_file":
        str(
            TARGET_AMBER_PATH
        ),

    "target_sha256":
        sha256_file(
            TARGET_AMBER_PATH
        ),

    "atoms":
        len(atoms),

    "residues":
        len(residues),

    "heavy_atoms":
        len(
            heavy_atoms
        ),

    "disulfides":
        len(
            stored_disulfides
        ),

    "geometry": {
        "fatal_backbone_outliers":
            0,

        "true_nonbonded_clashes":
            0,

        "deferred_terminal_warnings":
            len(
                deferred_terminal_warnings
            ),

        "deferred_relaxation_residues":
            deferred_residues,
    },
}


TARGET_READY_PATH.write_text(
    json.dumps(
        target_ready,
        indent=2,
    )
)


print()
print("TARGET READY")

if deferred_terminal_warnings:

    print(
        "Terminal angle strain is explicitly "
        "deferred to whole-system minimization."
    )


print()
print(
    "Manifest:",
    TARGET_READY_PATH,
)

print(
    "Audit:",
    FINAL_TARGET_AUDIT_PATH,
)

Final target validation
-----------------------
Backbone outliers: 0
Fatal backbone outliers: 0
Deferred terminal warnings: 0
True nonbonded clashes: 0
Hydrogens: 0
Disulfides: 0

TARGET READY

Manifest: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/target_ready.json
Audit: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target/final_target_audit.json


In [22]:
#@title 26. Prepare small-molecule ligand geometry

import hashlib
import json
from pathlib import Path

import gemmi
import numpy as np

from rdkit import Chem
from rdkit.Chem import (
    AllChem,
    Descriptors,
    rdMolDescriptors,
)


LIGAND_H_SDF = (
    LIGAND_PREP_DIR
    / "ligand_with_h.sdf"
)

LIGAND_PREP_MANIFEST_PATH = (
    LIGAND_PREP_DIR
    / "ligand_preparation.json"
)


MIN_NONBONDED_H_DISTANCE_A = 1.00


def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def read_small_molecule(
    path: Path,
) -> Chem.Mol:

    suffix = path.suffix.lower()


    if suffix in {".sdf", ".sd"}:

        supplier = Chem.SDMolSupplier(
            str(path),
            removeHs=False,
            sanitize=True,
        )

        molecules = [
            molecule
            for molecule in supplier
            if molecule is not None
        ]

        if len(molecules) != 1:

            raise ValueError(
                "Small-molecule input must contain "
                "exactly one molecule."
            )

        molecule = molecules[0]


    elif suffix == ".mol":

        molecule = Chem.MolFromMolFile(
            str(path),
            removeHs=False,
            sanitize=True,
        )


    elif suffix == ".mol2":

        molecule = Chem.MolFromMol2File(
            str(path),
            removeHs=False,
            sanitize=True,
        )


    elif suffix == ".pdb":

        raise ValueError(
            "PDB is not accepted as a generic "
            "small-molecule input because bond "
            "orders are ambiguous. Use SDF or MOL2."
        )


    else:

        raise ValueError(
            f"Unsupported ligand format: {suffix}"
        )


    if molecule is None:

        raise ValueError(
            "RDKit could not read the ligand."
        )


    if molecule.GetNumConformers() != 1:

        raise ValueError(
            "Ligand must contain exactly "
            "one conformer."
        )


    if not molecule.GetConformer().Is3D():

        raise ValueError(
            "Ligand must contain 3D coordinates."
        )


    if len(Chem.GetMolFrags(molecule)) != 1:

        raise ValueError(
            "Ligand contains disconnected fragments."
        )


    return molecule


def ligand_metals(
    molecule: Chem.Mol,
) -> list[str]:

    elements = {
        atom.GetSymbol()
        for atom in molecule.GetAtoms()
    }

    return sorted(
        element
        for element in elements
        if gemmi.Element(element).is_metal
    )


def short_h_contacts(
    molecule: Chem.Mol,
    cutoff_A: float,
) -> list[dict]:

    conformer = molecule.GetConformer()

    bonded = {
        frozenset(
            (
                bond.GetBeginAtomIdx(),
                bond.GetEndAtomIdx(),
            )
        )
        for bond in molecule.GetBonds()
    }

    contacts = []


    for i in range(molecule.GetNumAtoms()):

        atom_i = molecule.GetAtomWithIdx(i)

        xyz_i = np.asarray(
            conformer.GetAtomPosition(i),
            dtype=float,
        )


        for j in range(
            i + 1,
            molecule.GetNumAtoms(),
        ):

            if frozenset((i, j)) in bonded:
                continue

            atom_j = molecule.GetAtomWithIdx(j)

            if (
                atom_i.GetAtomicNum() != 1
                and atom_j.GetAtomicNum() != 1
            ):
                continue

            xyz_j = np.asarray(
                conformer.GetAtomPosition(j),
                dtype=float,
            )

            distance = float(
                np.linalg.norm(
                    xyz_i - xyz_j
                )
            )

            if distance < cutoff_A:

                contacts.append(
                    {
                        "atom_1": i,
                        "atom_2": j,
                        "distance_A": distance,
                    }
                )


    return sorted(
        contacts,
        key=lambda item:
            item["distance_A"],
    )


if LIGAND_KIND == "none":

    manifest = {
        "status": "not_present",
        "route": "none",
    }

    LIGAND_PREP_MANIFEST_PATH.write_text(
        json.dumps(
            manifest,
            indent=2,
        )
    )

    print("No ligand supplied.")


elif LIGAND_KIND != "small_molecule":

    raise RuntimeError(
        f"Ligand type '{LIGAND_KIND}' does not use "
        "the small-molecule preparation route."
    )


else:

    if not LIGAND_PATH.is_file():

        raise FileNotFoundError(
            f"Ligand not found:\n{LIGAND_PATH}"
        )


    ligand = read_small_molecule(
        LIGAND_PATH
    )


    metals = ligand_metals(
        ligand
    )


    if metals:

        raise RuntimeError(
            "Metal-containing ligand detected: "
            + ", ".join(metals)
            + "\nGeneric GAFF2 parameterization "
            "is intentionally disabled."
        )


    formal_charge = Chem.GetFormalCharge(
        ligand
    )


    if formal_charge != LIGAND_NET_CHARGE:

        raise RuntimeError(
            "Ligand formal charge mismatch:\n"
            f"structure:  {formal_charge}\n"
            f"configured: {LIGAND_NET_CHARGE}"
        )


    radical_electrons = sum(
        atom.GetNumRadicalElectrons()
        for atom in ligand.GetAtoms()
    )


    if radical_electrons:

        raise RuntimeError(
            "Radical ligand detected. Automatic "
            "closed-shell AM1-BCC preparation "
            "is disabled."
        )


    heavy_indices = [
        atom.GetIdx()
        for atom in ligand.GetAtoms()
        if atom.GetAtomicNum() > 1
    ]


    original_conformer = (
        ligand.GetConformer()
    )


    reference_heavy_xyz = np.asarray(
        [
            original_conformer.GetAtomPosition(index)
            for index in heavy_indices
        ],
        dtype=float,
    )


    ligand_with_h = Chem.AddHs(
        ligand,
        addCoords=True,
    )


    contacts_before = short_h_contacts(
        ligand_with_h,
        MIN_NONBONDED_H_DISTANCE_A,
    )


    if AllChem.MMFFHasAllMoleculeParams(
        ligand_with_h
    ):

        properties = (
            AllChem.MMFFGetMoleculeProperties(
                ligand_with_h,
                mmffVariant="MMFF94s",
            )
        )

        forcefield = (
            AllChem.MMFFGetMoleculeForceField(
                ligand_with_h,
                properties,
            )
        )

        relaxation_method = "MMFF94s"


    elif AllChem.UFFHasAllMoleculeParams(
        ligand_with_h
    ):

        forcefield = (
            AllChem.UFFGetMoleculeForceField(
                ligand_with_h
            )
        )

        relaxation_method = "UFF"


    else:

        raise RuntimeError(
            "Neither MMFF94s nor UFF can describe "
            "the ligand for hydrogen-only relaxation."
        )


    for atom in ligand_with_h.GetAtoms():

        if atom.GetAtomicNum() > 1:
            forcefield.AddFixedPoint(
                atom.GetIdx()
            )


    status = forcefield.Minimize(
        maxIts=2000
    )


    if status != 0:

        raise RuntimeError(
            "Hydrogen-only ligand relaxation "
            "did not converge."
        )


    final_conformer = (
        ligand_with_h.GetConformer()
    )


    final_heavy_xyz = np.asarray(
        [
            final_conformer.GetAtomPosition(index)
            for index in heavy_indices
        ],
        dtype=float,
    )


    max_heavy_shift = float(
        np.linalg.norm(
            final_heavy_xyz
            - reference_heavy_xyz,
            axis=1,
        ).max()
    )


    if max_heavy_shift > 1e-4:

        raise RuntimeError(
            "Ligand heavy atoms moved during "
            "hydrogen-only relaxation:\n"
            f"{max_heavy_shift:.6f} Å"
        )


    contacts_after = short_h_contacts(
        ligand_with_h,
        MIN_NONBONDED_H_DISTANCE_A,
    )


    if contacts_after:

        raise RuntimeError(
            "Implausibly short ligand H contacts "
            "remain after relaxation."
        )


    writer = Chem.SDWriter(
        str(LIGAND_H_SDF)
    )

    writer.write(
        ligand_with_h
    )

    writer.close()


    manifest = {
        "status": "prepared",

        "route":
            SMALL_MOLECULE_FORCE_FIELD,

        "source_file":
            str(LIGAND_PATH),

        "source_sha256":
            sha256_file(
                LIGAND_PATH
            ),

        "prepared_file":
            str(LIGAND_H_SDF),

        "prepared_sha256":
            sha256_file(
                LIGAND_H_SDF
            ),

        "residue_name":
            LIGAND_RESNAME,

        "formula":
            rdMolDescriptors.CalcMolFormula(
                ligand_with_h
            ),

        "molecular_weight_Da":
            float(
                Descriptors.MolWt(
                    ligand_with_h
                )
            ),

        "formal_charge":
            int(formal_charge),

        "multiplicity":
            1,

        "atoms":
            ligand_with_h.GetNumAtoms(),

        "heavy_atoms":
            ligand_with_h.GetNumHeavyAtoms(),

        "hydrogens":
            sum(
                atom.GetAtomicNum() == 1
                for atom
                in ligand_with_h.GetAtoms()
            ),

        "hydrogen_relaxation":
            relaxation_method,

        "short_h_contacts_before":
            len(contacts_before),

        "short_h_contacts_after":
            len(contacts_after),

        "max_heavy_atom_shift_A":
            max_heavy_shift,
    }


    LIGAND_PREP_MANIFEST_PATH.write_text(
        json.dumps(
            manifest,
            indent=2,
        )
    )


    print("Ligand geometry prepared")
    print("------------------------")

    print(
        "Formula:",
        manifest["formula"],
    )

    print(
        "Atoms:",
        manifest["atoms"],
    )

    print(
        "Heavy atoms:",
        manifest["heavy_atoms"],
    )

    print(
        "Hydrogens:",
        manifest["hydrogens"],
    )

    print(
        "Formal charge:",
        manifest["formal_charge"],
    )

    print(
        "H relaxation:",
        relaxation_method,
    )

    print(
        "Short H contacts:",
        f"{len(contacts_before)} "
        f"→ {len(contacts_after)}",
    )

    print(
        "Max heavy-atom shift:",
        f"{max_heavy_shift:.6f} Å",
    )

    print()
    print(
        "Manifest:",
        LIGAND_PREP_MANIFEST_PATH,
    )

Ligand geometry prepared
------------------------
Formula: C33H38N4O6
Atoms: 81
Heavy atoms: 43
Hydrogens: 38
Formal charge: 0
H relaxation: MMFF94s
Short H contacts: 0 → 0
Max heavy-atom shift: 0.000000 Å

Manifest: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand_preparation.json


In [23]:
#@title 27. Parameterize small-molecule ligand

import hashlib
import json
import re
from pathlib import Path


LIGAND_PARAMETERIZATION_PATH = (
    LIGAND_PREP_DIR
    / "ligand_parameterization.json"
)

LIGAND_MOL2 = (
    LIGAND_PREP_DIR
    / f"ligand_{SMALL_MOLECULE_FORCE_FIELD.lower()}.mol2"
)

LIGAND_FRCMOD = (
    LIGAND_PREP_DIR
    / "ligand.frcmod"
)

ANTECHAMBER_LOG = (
    LOG_DIR
    / "antechamber.log"
)

PARMCHK_LOG = (
    LOG_DIR
    / "parmchk2.log"
)


def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


if not callable(
    globals().get("run_command")
):
    raise RuntimeError(
        "run_command() is unavailable. "
        "Run the preparation utilities block first."
    )


if LIGAND_KIND == "none":

    result = {
        "status": "not_present",
        "route": "none",
    }

    LIGAND_PARAMETERIZATION_PATH.write_text(
        json.dumps(
            result,
            indent=2,
        )
    )

    print("No ligand to parameterize.")


elif LIGAND_KIND != "small_molecule":

    raise RuntimeError(
        f"Ligand type '{LIGAND_KIND}' does not "
        "belong to the generic GAFF parameterization route."
    )


else:

    if not LIGAND_PREP_MANIFEST_PATH.is_file():

        raise FileNotFoundError(
            "Ligand preparation manifest not found:\n"
            f"{LIGAND_PREP_MANIFEST_PATH}"
        )


    preparation = json.loads(
        LIGAND_PREP_MANIFEST_PATH.read_text()
    )


    if preparation.get("status") != "prepared":

        raise RuntimeError(
            "Ligand geometry has not been finalized."
        )


    prepared_sdf = Path(
        preparation["prepared_file"]
    )


    if not prepared_sdf.is_file():

        raise FileNotFoundError(
            f"Prepared ligand not found:\n"
            f"{prepared_sdf}"
        )


    if (
        sha256_file(prepared_sdf)
        != preparation["prepared_sha256"]
    ):

        raise RuntimeError(
            "Prepared ligand changed after block 26.\n"
            "Refusing to parameterize stale input."
        )


    force_field_name = (
        SMALL_MOLECULE_FORCE_FIELD
        .strip()
        .lower()
    )


    if force_field_name not in {
        "gaff",
        "gaff2",
    }:

        raise ValueError(
            "SMALL_MOLECULE_FORCE_FIELD must be "
            "'gaff' or 'gaff2'."
        )


    if not re.fullmatch(
        r"[A-Za-z0-9]{1,3}",
        LIGAND_RESNAME,
    ):

        raise ValueError(
            "LIGAND_RESNAME must contain "
            "1–3 alphanumeric characters."
        )


    formal_charge = int(
        preparation["formal_charge"]
    )

    multiplicity = int(
        preparation.get(
            "multiplicity",
            1,
        )
    )


    if multiplicity < 1:

        raise ValueError(
            "Ligand multiplicity must be >= 1."
        )


    for path in (
        LIGAND_MOL2,
        LIGAND_FRCMOD,
    ):
        path.unlink(
            missing_ok=True
        )


    antechamber_output = run_command(
        [
            "antechamber",
            "-i",
            str(prepared_sdf),
            "-fi",
            "sdf",
            "-o",
            str(LIGAND_MOL2),
            "-fo",
            "mol2",
            "-c",
            "bcc",
            "-nc",
            str(formal_charge),
            "-m",
            str(multiplicity),
            "-at",
            force_field_name,
            "-rn",
            LIGAND_RESNAME,
            "-pf",
            "y",
            "-s",
            "2",
        ],
        cwd=LIGAND_PREP_DIR,
        log_path=ANTECHAMBER_LOG,
    )


    if (
        not LIGAND_MOL2.is_file()
        or LIGAND_MOL2.stat().st_size == 0
    ):

        raise RuntimeError(
            "Antechamber completed without producing "
            "a valid MOL2 file."
        )


    parmchk_selector = (
        "2"
        if force_field_name == "gaff2"
        else "1"
    )


    parmchk_output = run_command(
        [
            "parmchk2",
            "-i",
            str(LIGAND_MOL2),
            "-f",
            "mol2",
            "-o",
            str(LIGAND_FRCMOD),
            "-s",
            parmchk_selector,
        ],
        cwd=LIGAND_PREP_DIR,
        log_path=PARMCHK_LOG,
    )


    if (
        not LIGAND_FRCMOD.is_file()
        or LIGAND_FRCMOD.stat().st_size == 0
    ):

        raise RuntimeError(
            "parmchk2 completed without producing "
            "a valid FRCMOD file."
        )


    result = {
        "status":
            "parameterized",

        "force_field":
            force_field_name,

        "charge_model":
            "AM1-BCC",

        "formal_charge":
            formal_charge,

        "multiplicity":
            multiplicity,

        "input_file":
            str(prepared_sdf),

        "input_sha256":
            sha256_file(
                prepared_sdf
            ),

        "mol2_file":
            str(LIGAND_MOL2),

        "mol2_sha256":
            sha256_file(
                LIGAND_MOL2
            ),

        "frcmod_file":
            str(LIGAND_FRCMOD),

        "frcmod_sha256":
            sha256_file(
                LIGAND_FRCMOD
            ),

        "antechamber_log":
            str(ANTECHAMBER_LOG),

        "parmchk_log":
            str(PARMCHK_LOG),
    }


    LIGAND_PARAMETERIZATION_PATH.write_text(
        json.dumps(
            result,
            indent=2,
        )
    )


    print("Ligand parameterization completed")
    print("---------------------------------")
    print("Charge model: AM1-BCC")
    print("Force field:", force_field_name)
    print("Formal charge:", formal_charge)
    print()
    print("MOL2:", LIGAND_MOL2)
    print("FRCMOD:", LIGAND_FRCMOD)
    print()
    print(
        "Manifest:",
        LIGAND_PARAMETERIZATION_PATH,
    )

$ antechamber -i /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand_with_h.sdf -fi sdf -o /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand_gaff2.mol2 -fo mol2 -c bcc -nc 0 -m 1 -at gaff2 -rn IRI -pf y -s 2
$ parmchk2 -i /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand_gaff2.mol2 -f mol2 -o /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand.frcmod -s 2
Ligand parameterization completed
---------------------------------
Charge model: AM1-BCC
Force field: gaff2
Formal charge: 0

MOL2: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand_gaff2.mol2
FRCMOD: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand.frcmod

Manifest: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand_parameterization.json


In [24]:
#@title 28. Audit ligand parameters

import hashlib
import json
from pathlib import Path

import numpy as np
import parmed as pmd

from rdkit import Chem
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


LIGAND_PARAMETER_AUDIT_PATH = (
    LIGAND_PREP_DIR
    / "ligand_parameter_audit.json"
)


MAX_PARAMETERIZATION_HEAVY_SHIFT_A = 0.05
CHARGE_TOLERANCE_E = 0.02


def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def read_single_sdf(
    path: Path,
) -> Chem.Mol:

    supplier = Chem.SDMolSupplier(
        str(path),
        removeHs=False,
        sanitize=True,
    )

    molecules = [
        molecule
        for molecule in supplier
        if molecule is not None
    ]

    if len(molecules) != 1:

        raise RuntimeError(
            "Prepared SDF must contain "
            "exactly one molecule."
        )

    return molecules[0]


def match_coordinates_by_element(
    reference,
    observed,
):

    reference_elements = [
        element
        for element, _
        in reference
    ]

    observed_elements = [
        element
        for element, _
        in observed
    ]


    if sorted(reference_elements) != sorted(
        observed_elements
    ):
        return None


    squared_distances = []


    for element in sorted(
        set(reference_elements)
    ):

        reference_xyz = np.asarray(
            [
                xyz
                for current_element, xyz
                in reference
                if current_element == element
            ],
            dtype=float,
        )

        observed_xyz = np.asarray(
            [
                xyz
                for current_element, xyz
                in observed
                if current_element == element
            ],
            dtype=float,
        )


        matrix = cdist(
            reference_xyz,
            observed_xyz,
        )


        row_indices, column_indices = (
            linear_sum_assignment(
                matrix
            )
        )


        squared_distances.extend(
            matrix[
                row_indices,
                column_indices,
            ] ** 2
        )


    distances = np.sqrt(
        np.asarray(
            squared_distances,
            dtype=float,
        )
    )


    return {
        "rmsd_A":
            float(
                np.sqrt(
                    np.mean(
                        distances ** 2
                    )
                )
            ),

        "max_shift_A":
            float(
                distances.max()
            ),
    }


if LIGAND_KIND == "none":

    audit = {
        "status": "not_present",
        "passed": True,
    }

    LIGAND_PARAMETER_AUDIT_PATH.write_text(
        json.dumps(
            audit,
            indent=2,
        )
    )

    print("No ligand parameter audit required.")


else:

    parameterization = json.loads(
        LIGAND_PARAMETERIZATION_PATH.read_text()
    )

    preparation = json.loads(
        LIGAND_PREP_MANIFEST_PATH.read_text()
    )


    if parameterization.get("status") != "parameterized":

        raise RuntimeError(
            "Ligand has not been parameterized."
        )


    mol2_path = Path(
        parameterization["mol2_file"]
    )

    frcmod_path = Path(
        parameterization["frcmod_file"]
    )

    prepared_sdf_path = Path(
        preparation["prepared_file"]
    )


    for path in (
        mol2_path,
        frcmod_path,
        prepared_sdf_path,
    ):

        if not path.is_file():

            raise FileNotFoundError(
                f"Required ligand file missing:\n{path}"
            )


    if (
        sha256_file(mol2_path)
        != parameterization["mol2_sha256"]
    ):

        raise RuntimeError(
            "Ligand MOL2 changed after parameterization."
        )


    if (
        sha256_file(frcmod_path)
        != parameterization["frcmod_sha256"]
    ):

        raise RuntimeError(
            "Ligand FRCMOD changed after parameterization."
        )


    prepared_molecule = read_single_sdf(
        prepared_sdf_path
    )

    prepared_conformer = (
        prepared_molecule.GetConformer()
    )


    reference_heavy = [
        (
            atom.GetSymbol(),
            np.asarray(
                prepared_conformer.GetAtomPosition(
                    atom.GetIdx()
                ),
                dtype=float,
            ),
        )
        for atom
        in prepared_molecule.GetAtoms()
        if atom.GetAtomicNum() > 1
    ]


    parameterized = pmd.load_file(
        str(mol2_path)
    )


    if parameterized.coordinates is None:

        raise RuntimeError(
            "Parameterized MOL2 contains "
            "no coordinates."
        )


    coordinates = np.asarray(
        parameterized.coordinates,
        dtype=float,
    )


    if len(coordinates) != len(
        parameterized.atoms
    ):

        raise RuntimeError(
            "MOL2 atom/coordinate count mismatch."
        )


    periodic_table = (
        Chem.GetPeriodicTable()
    )


    observed_heavy = []


    for atom, xyz in zip(
        parameterized.atoms,
        coordinates,
    ):

        atomic_number = int(
            atom.atomic_number
        )


        if atomic_number <= 0:

            raise RuntimeError(
                "Could not determine element for "
                f"MOL2 atom '{atom.name}'."
            )


        if atomic_number == 1:
            continue


        observed_heavy.append(
            (
                periodic_table
                .GetElementSymbol(
                    atomic_number
                ),

                np.asarray(
                    xyz,
                    dtype=float,
                ),
            )
        )


    pose = match_coordinates_by_element(
        reference_heavy,
        observed_heavy,
    )


    if pose is None:

        raise RuntimeError(
            "Heavy-atom elemental composition changed "
            "during parameterization."
        )


    atom_count = len(
        parameterized.atoms
    )

    heavy_atom_count = len(
        observed_heavy
    )

    partial_charge_sum = float(
        sum(
            atom.charge
            for atom
            in parameterized.atoms
        )
    )


    if atom_count != int(
        preparation["atoms"]
    ):

        raise RuntimeError(
            "Ligand atom count changed:\n"
            f"expected: {preparation['atoms']}\n"
            f"found:    {atom_count}"
        )


    if heavy_atom_count != int(
        preparation["heavy_atoms"]
    ):

        raise RuntimeError(
            "Ligand heavy-atom count changed."
        )


    if abs(
        partial_charge_sum
        - int(
            preparation["formal_charge"]
        )
    ) > CHARGE_TOLERANCE_E:

        raise RuntimeError(
            "AM1-BCC partial charges do not sum "
            "to the configured molecular charge:\n"
            f"{partial_charge_sum:.6f}"
        )


    if (
        pose["max_shift_A"]
        > MAX_PARAMETERIZATION_HEAVY_SHIFT_A
    ):

        raise RuntimeError(
            "Ligand heavy-atom pose moved during "
            "parameterization:\n"
            f"{pose['max_shift_A']:.4f} Å"
        )


    frcmod_text = frcmod_path.read_text(
        errors="replace"
    )


    suspicious_parameter_lines = [
        line.strip()
        for line in frcmod_text.splitlines()
        if (
            "ATTN" in line.upper()
            or "NEEDS REVISION" in line.upper()
            or "NEED REVISION" in line.upper()
        )
    ]


    if suspicious_parameter_lines:

        details = "\n".join(
            f"- {line}"
            for line
            in suspicious_parameter_lines
        )

        raise RuntimeError(
            "parmchk2 produced parameters requiring "
            "manual revision:\n"
            + details
        )


    audit = {
        "status":
            "audited",

        "passed":
            True,

        "atoms":
            atom_count,

        "heavy_atoms":
            heavy_atom_count,

        "partial_charge_sum":
            partial_charge_sum,

        "heavy_atom_rmsd_A":
            pose["rmsd_A"],

        "heavy_atom_max_shift_A":
            pose["max_shift_A"],

        "frcmod_revision_flags":
            suspicious_parameter_lines,
    }


    LIGAND_PARAMETER_AUDIT_PATH.write_text(
        json.dumps(
            audit,
            indent=2,
        )
    )


    print("Ligand parameter audit")
    print("----------------------")
    print("Atoms:", atom_count)
    print("Heavy atoms:", heavy_atom_count)
    print(
        "AM1-BCC charge sum:",
        f"{partial_charge_sum:.6f}",
    )
    print(
        "Heavy-atom RMSD:",
        f"{pose['rmsd_A']:.6f} Å",
    )
    print(
        "Maximum heavy-atom shift:",
        f"{pose['max_shift_A']:.6f} Å",
    )
    print(
        "parmchk2 revision flags:",
        len(
            suspicious_parameter_lines
        ),
    )
    print()
    print("LIGAND PARAMETERS PASSED")

Ligand parameter audit
----------------------
Atoms: 81
Heavy atoms: 43
AM1-BCC charge sum: -0.000002
Heavy-atom RMSD: 0.000532 Å
Maximum heavy-atom shift: 0.000707 Å
parmchk2 revision flags: 0

LIGAND PARAMETERS PASSED


In [25]:
#@title 29. Validate ligand in LEaP

import hashlib
import json
import re
from pathlib import Path

import parmed as pmd


LIGAND_LEAP_DIR = (
    LIGAND_PREP_DIR
    / "leap_test"
)

LIGAND_LEAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


LIGAND_TEST_PRMTOP = (
    LIGAND_LEAP_DIR
    / "ligand.prmtop"
)

LIGAND_TEST_RST7 = (
    LIGAND_LEAP_DIR
    / "ligand.rst7"
)

LIGAND_TEST_PDB = (
    LIGAND_LEAP_DIR
    / "ligand.pdb"
)

LIGAND_LEAP_INPUT = (
    LIGAND_LEAP_DIR
    / "ligand.leap.in"
)

LIGAND_LEAP_LOG = (
    LOG_DIR
    / "tleap_ligand.log"
)

LIGAND_READY_PATH = (
    LIGAND_PREP_DIR
    / "ligand_ready.json"
)


def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


if not callable(
    globals().get("run_command")
):
    raise RuntimeError(
        "run_command() is unavailable."
    )


if LIGAND_KIND == "none":

    result = {
        "status": "not_present",
        "ready": True,
    }

    LIGAND_READY_PATH.write_text(
        json.dumps(
            result,
            indent=2,
        )
    )

    print("No ligand LEaP test required.")


else:

    audit = json.loads(
        LIGAND_PARAMETER_AUDIT_PATH.read_text()
    )

    parameterization = json.loads(
        LIGAND_PARAMETERIZATION_PATH.read_text()
    )

    preparation = json.loads(
        LIGAND_PREP_MANIFEST_PATH.read_text()
    )


    if not audit.get("passed", False):

        raise RuntimeError(
            "Ligand parameter audit has not passed."
        )


    mol2_path = Path(
        parameterization["mol2_file"]
    )

    frcmod_path = Path(
        parameterization["frcmod_file"]
    )


    if (
        sha256_file(mol2_path)
        != parameterization["mol2_sha256"]
    ):

        raise RuntimeError(
            "MOL2 changed after audit."
        )


    if (
        sha256_file(frcmod_path)
        != parameterization["frcmod_sha256"]
    ):

        raise RuntimeError(
            "FRCMOD changed after audit."
        )


    force_field_name = (
        parameterization[
            "force_field"
        ]
    )


    leaprc = (
        "leaprc.gaff2"
        if force_field_name == "gaff2"
        else "leaprc.gaff"
    )


    for output_path in (
        LIGAND_TEST_PRMTOP,
        LIGAND_TEST_RST7,
        LIGAND_TEST_PDB,
    ):

        output_path.unlink(
            missing_ok=True
        )


    leap_script = "\n".join(
        [
            f"source {leaprc}",
            f"loadamberparams {frcmod_path.name}",
            f"LIG = loadmol2 {mol2_path.name}",
            "check LIG",
            (
                f"saveamberparm LIG "
                f"{LIGAND_TEST_PRMTOP.name} "
                f"{LIGAND_TEST_RST7.name}"
            ),
            (
                f"savepdb LIG "
                f"{LIGAND_TEST_PDB.name}"
            ),
            "quit",
            "",
        ]
    )


    LIGAND_LEAP_INPUT.write_text(
        leap_script
    )


    tleap_output = run_command(
        [
            "tleap",
            "-I",
            str(LIGAND_PREP_DIR),
            "-f",
            str(LIGAND_LEAP_INPUT),
        ],
        cwd=LIGAND_LEAP_DIR,
        log_path=LIGAND_LEAP_LOG,
    )


    error_counts = [
        int(value)
        for value in re.findall(
            r"Errors\s*=\s*(\d+)",
            tleap_output,
            flags=re.IGNORECASE,
        )
    ]


    if (
        error_counts
        and max(error_counts) != 0
    ):

        raise RuntimeError(
            "LEaP reported ligand errors. "
            f"See:\n{LIGAND_LEAP_LOG}"
        )


    if "FATAL" in tleap_output.upper():

        raise RuntimeError(
            "LEaP reported a fatal ligand error. "
            f"See:\n{LIGAND_LEAP_LOG}"
        )


    for path in (
        LIGAND_TEST_PRMTOP,
        LIGAND_TEST_RST7,
        LIGAND_TEST_PDB,
    ):

        if (
            not path.is_file()
            or path.stat().st_size == 0
        ):

            raise RuntimeError(
                "LEaP did not create expected file:\n"
                f"{path}"
            )


    topology = pmd.load_file(
        str(LIGAND_TEST_PRMTOP),
        str(LIGAND_TEST_RST7),
    )


    topology_charge = float(
        sum(
            atom.charge
            for atom
            in topology.atoms
        )
    )


    if len(topology.residues) != 1:

        raise RuntimeError(
            "Standalone ligand topology must "
            "contain exactly one residue."
        )


    if (
        topology.residues[0].name.upper()
        != LIGAND_RESNAME.upper()
    ):

        raise RuntimeError(
            "Ligand residue name changed in LEaP."
        )


    if len(topology.atoms) != int(
        preparation["atoms"]
    ):

        raise RuntimeError(
            "LEaP ligand atom count mismatch."
        )


    if abs(
        topology_charge
        - int(
            preparation["formal_charge"]
        )
    ) > 0.02:

        raise RuntimeError(
            "LEaP topology charge does not match "
            "the configured ligand charge."
        )


    result = {
        "status":
            "ready",

        "ready":
            True,

        "residue_name":
            LIGAND_RESNAME,

        "atoms":
            len(
                topology.atoms
            ),

        "charge":
            topology_charge,

        "mol2_file":
            str(
                mol2_path
            ),

        "mol2_sha256":
            sha256_file(
                mol2_path
            ),

        "frcmod_file":
            str(
                frcmod_path
            ),

        "frcmod_sha256":
            sha256_file(
                frcmod_path
            ),

        "test_prmtop":
            str(
                LIGAND_TEST_PRMTOP
            ),

        "test_prmtop_sha256":
            sha256_file(
                LIGAND_TEST_PRMTOP
            ),

        "test_rst7":
            str(
                LIGAND_TEST_RST7
            ),
    }


    LIGAND_READY_PATH.write_text(
        json.dumps(
            result,
            indent=2,
        )
    )


    print("Ligand LEaP self-test")
    print("----------------------")
    print(
        "Topology atoms:",
        len(topology.atoms),
    )
    print(
        "Residues:",
        len(topology.residues),
    )
    print(
        "Charge:",
        f"{topology_charge:.6f}",
    )
    print()
    print("LIGAND READY")
    print(
        "Manifest:",
        LIGAND_READY_PATH,
    )

$ tleap -I /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand -f /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/leap_test/ligand.leap.in
Ligand LEaP self-test
----------------------
Topology atoms: 81
Residues: 1
Charge: -0.000002

LIGAND READY
Manifest: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand/ligand_ready.json


In [44]:
#@title 30. Assemble validated unsolvated complex

import hashlib
import json
import re

from pathlib import Path
from shutil import which

import gemmi
import numpy as np


COMPLEX_DIR = (
    WORKDIR
    / "complex"
)

COMPLEX_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


COMPLEX_PRMTOP = (
    COMPLEX_DIR
    / "complex.prmtop"
)

COMPLEX_RST7 = (
    COMPLEX_DIR
    / "complex.rst7"
)

COMPLEX_PDB = (
    COMPLEX_DIR
    / "complex.pdb"
)

COMPLEX_LEAP_INPUT = (
    COMPLEX_DIR
    / "complex.leap.in"
)

COMPLEX_LEAP_LOG = (
    LOG_DIR
    / "tleap_complex.log"
)

COMPLEX_ASSEMBLY_PATH = (
    COMPLEX_DIR
    / "complex_assembly.json"
)


PROTEIN_LEAPRC = {
    "ff19SB":
        "leaprc.protein.ff19SB",

    "ff14SB":
        "leaprc.protein.ff14SB",
}


DNA_LEAPRC = {
    "OL21":
        "leaprc.DNA.OL21",

    "OL15":
        "leaprc.DNA.OL15",

    "bsc1":
        "leaprc.DNA.bsc1",
}


RNA_LEAPRC = {
    "OL3":
        "leaprc.RNA.OL3",
}


WATER_LEAPRC = {
    "TIP3P":
        "leaprc.water.tip3p",

    "OPC":
        "leaprc.water.opc",

    "SPCE":
        "leaprc.water.spce",
}


GAFF_LEAPRC = {
    "gaff":
        "leaprc.gaff",

    "gaff2":
        "leaprc.gaff2",
}


PROTEIN_RESIDUES = {
    "ALA", "ARG", "ASN",
    "ASP", "ASH",
    "CYS", "CYM", "CYX",
    "GLN", "GLU", "GLH",
    "GLY",
    "HID", "HIE", "HIP", "HIN",
    "ILE", "LEU",
    "LYS", "LYN",
    "MET", "PHE", "PRO",
    "SER", "THR",
    "TRP", "TYR", "VAL",
}


DNA_RESIDUES = {
    "DA", "DC", "DG", "DT", "DI",
    "DA3", "DC3", "DG3", "DT3",
    "DA5", "DC5", "DG5", "DT5",
}


RNA_RESIDUES = {
    "A", "C", "G", "U", "I",
    "RA", "RC", "RG", "RU",
    "A3", "C3", "G3", "U3",
    "A5", "C5", "G5", "U5",
}


WATER_RESIDUES = {
    "HOH",
    "WAT",
}


def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def amber_leap_command_dir() -> Path:

    executable = which(
        "tleap"
    )

    if executable is None:

        raise RuntimeError(
            "tleap is unavailable."
        )


    prefix = (
        Path(executable)
        .resolve()
        .parent
        .parent
    )


    command_dir = (
        prefix
        / "dat"
        / "leap"
        / "cmd"
    )


    if not command_dir.is_dir():

        raise RuntimeError(
            "Amber LEaP command directory "
            f"not found:\n{command_dir}"
        )


    return command_dir


def inspect_target_components(
    path: Path,
):

    structure = gemmi.read_structure(
        str(path)
    )

    structure.setup_entities()


    detected = set()

    unsupported = []


    for chain in structure[0]:

        for residue in chain:

            name = (
                residue.name
                .strip()
                .upper()
            )


            if name in PROTEIN_RESIDUES:
                detected.add(
                    "protein"
                )

            elif name in DNA_RESIDUES:
                detected.add(
                    "dna"
                )

            elif name in RNA_RESIDUES:
                detected.add(
                    "rna"
                )

            elif name in WATER_RESIDUES:
                continue

            else:

                unsupported.append(
                    f"{chain.name}:"
                    f"{residue.name}"
                    f"{residue.seqid.num}"
                )


    if unsupported:

        raise RuntimeError(
            "Prepared target still contains "
            "components without an explicit "
            "parameterization route:\n"
            + "\n".join(
                f"- {item}"
                for item in unsupported
            )
        )


    if not detected:

        raise RuntimeError(
            "No supported biopolymer was found "
            "in the target."
        )


    return structure, detected


def detect_disulfide_pairs(
    structure: gemmi.Structure,
):

    cysteines = []

    leap_residue_index = 0


    for chain in structure[0]:

        for residue in chain:

            leap_residue_index += 1


            if (
                residue.name
                .strip()
                .upper()
                != "CYX"
            ):

                continue


            sulfur = next(
                (
                    atom
                    for atom in residue
                    if (
                        atom.name.strip()
                        == "SG"
                    )
                ),
                None,
            )


            if sulfur is None:

                raise RuntimeError(
                    "CYX residue lacks SG atom:\n"
                    f"{chain.name}:"
                    f"{residue.name}"
                    f"{residue.seqid.num}"
                )


            cysteines.append(
                {
                    "leap_index":
                        leap_residue_index,

                    "chain":
                        chain.name,

                    "number":
                        residue.seqid.num,

                    "xyz":
                        np.asarray(
                            [
                                sulfur.pos.x,
                                sulfur.pos.y,
                                sulfur.pos.z,
                            ],
                            dtype=float,
                        ),
                }
            )


    candidates = []


    for i, first in enumerate(
        cysteines
    ):

        for second in cysteines[
            i + 1:
        ]:

            distance = float(
                np.linalg.norm(
                    first["xyz"]
                    - second["xyz"]
                )
            )


            if (
                distance
                <= DISULFIDE_MAX_DISTANCE_A
            ):

                candidates.append(
                    (
                        distance,
                        first,
                        second,
                    )
                )


    candidates.sort(
        key=lambda item:
            item[0]
    )


    used = set()

    pairs = []


    for distance, first, second in candidates:

        if (
            first["leap_index"] in used
            or second["leap_index"] in used
        ):
            continue


        used.add(
            first["leap_index"]
        )

        used.add(
            second["leap_index"]
        )


        pairs.append(
            {
                "residue_1":
                    first[
                        "leap_index"
                    ],

                "residue_2":
                    second[
                        "leap_index"
                    ],

                "distance_A":
                    distance,
            }
        )


    return pairs


if not callable(
    globals().get(
        "run_command"
    )
):

    raise RuntimeError(
        "run_command() is unavailable."
    )


target_ready = json.loads(
    TARGET_READY_PATH.read_text()
)


if (
    target_ready.get("status")
    != "ready"
):

    raise RuntimeError(
        "Target has not passed final validation."
    )


target_path = Path(
    target_ready[
        "target_file"
    ]
)


if (
    sha256_file(target_path)
    != target_ready[
        "target_sha256"
    ]
):

    raise RuntimeError(
        "Target changed after final validation."
    )


target_structure, target_types = (
    inspect_target_components(
        target_path
    )
)


leaprc_files = []


if "protein" in target_types:

    leaprc_files.append(
        PROTEIN_LEAPRC[
            PROTEIN_FORCE_FIELD
        ]
    )


if "dna" in target_types:

    leaprc_files.append(
        DNA_LEAPRC[
            DNA_FORCE_FIELD
        ]
    )


if "rna" in target_types:

    leaprc_files.append(
        RNA_LEAPRC[
            RNA_FORCE_FIELD
        ]
    )


leaprc_files.append(
    WATER_LEAPRC[
        WATER_MODEL
    ]
)


ligand_ready = json.loads(
    LIGAND_READY_PATH.read_text()
)


if LIGAND_KIND == "small_molecule":

    if not ligand_ready.get(
        "ready",
        False,
    ):

        raise RuntimeError(
            "Ligand has not passed LEaP validation."
        )


    mol2_path = Path(
        ligand_ready[
            "mol2_file"
        ]
    )

    frcmod_path = Path(
        ligand_ready[
            "frcmod_file"
        ]
    )


    if (
        sha256_file(mol2_path)
        != ligand_ready[
            "mol2_sha256"
        ]
    ):

        raise RuntimeError(
            "Ligand MOL2 changed after validation."
        )


    if (
        sha256_file(frcmod_path)
        != ligand_ready[
            "frcmod_sha256"
        ]
    ):

        raise RuntimeError(
            "Ligand FRCMOD changed after validation."
        )


    leaprc_files.append(
        GAFF_LEAPRC[
            SMALL_MOLECULE_FORCE_FIELD
            .lower()
        ]
    )


elif LIGAND_KIND != "none":

    raise RuntimeError(
        f"Complex assembly for ligand type "
        f"'{LIGAND_KIND}' is not implemented "
        "in this branch."
    )


leaprc_files = list(
    dict.fromkeys(
        leaprc_files
    )
)


command_dir = (
    amber_leap_command_dir()
)


missing_leaprc = [
    filename
    for filename in leaprc_files
    if not (
        command_dir
        / filename
    ).is_file()
]


if missing_leaprc:

    raise RuntimeError(
        "Selected LEaP force-field files "
        "are unavailable:\n"
        + "\n".join(
            f"- {item}"
            for item
            in missing_leaprc
        )
    )


disulfide_pairs = (
    detect_disulfide_pairs(
        target_structure
    )
)


expected_disulfides = int(
    target_ready[
        "disulfides"
    ]
)


if (
    len(disulfide_pairs)
    != expected_disulfides
):

    raise RuntimeError(
        "Disulfide count changed before "
        "complex assembly:\n"
        f"expected: {expected_disulfides}\n"
        f"found:    {len(disulfide_pairs)}"
    )


for path in (
    COMPLEX_PRMTOP,
    COMPLEX_RST7,
    COMPLEX_PDB,
):

    path.unlink(
        missing_ok=True
    )


lines = [
    f"source {filename}"
    for filename
    in leaprc_files
]


if LIGAND_KIND == "small_molecule":

    lines.extend(
        [
            "",
            (
                "loadamberparams "
                f"{frcmod_path.name}"
            ),
        ]
    )


lines.extend(
    [
        "",
        (
            "TGT = loadpdb "
            f"{target_path.name}"
        ),
    ]
)


for pair in disulfide_pairs:

    lines.append(
        "bond "
        f"TGT.{pair['residue_1']}.SG "
        f"TGT.{pair['residue_2']}.SG"
    )


lines.extend(
    [
        "",
        "check TGT",
    ]
)


if LIGAND_KIND == "small_molecule":

    lines.extend(
        [
            "",
            (
                "LIG = loadmol2 "
                f"{mol2_path.name}"
            ),
            "check LIG",
            "",
            "COM = combine { TGT LIG }",
        ]
    )

else:

    lines.extend(
        [
            "",
            "COM = TGT",
        ]
    )


lines.extend(
    [
        "",
        "check COM",
        "",
        (
            f"saveamberparm COM "
            f"{COMPLEX_PRMTOP.name} "
            f"{COMPLEX_RST7.name}"
        ),
        (
            f"savepdb COM "
            f"{COMPLEX_PDB.name}"
        ),
        "",
        "quit",
        "",
    ]
)


COMPLEX_LEAP_INPUT.write_text(
    "\n".join(lines)
)


command = [
    "tleap",
    "-I",
    str(TARGET_PREP_DIR),
]


if LIGAND_KIND == "small_molecule":

    command.extend(
        [
            "-I",
            str(LIGAND_PREP_DIR),
        ]
    )


command.extend(
    [
        "-f",
        str(
            COMPLEX_LEAP_INPUT
        ),
    ]
)


tleap_output = run_command(
    command,
    cwd=COMPLEX_DIR,
    log_path=COMPLEX_LEAP_LOG,
)


error_counts = [
    int(value)
    for value in re.findall(
        r"Errors\s*=\s*(\d+)",
        tleap_output,
        flags=re.IGNORECASE,
    )
]


if (
    error_counts
    and max(error_counts) != 0
):

    raise RuntimeError(
        "LEaP reported errors while assembling "
        "the complex.\n"
        f"See:\n{COMPLEX_LEAP_LOG}"
    )


if "FATAL" in tleap_output.upper():

    raise RuntimeError(
        "LEaP reported a fatal error while "
        "assembling the complex.\n"
        f"See:\n{COMPLEX_LEAP_LOG}"
    )


for path in (
    COMPLEX_PRMTOP,
    COMPLEX_RST7,
    COMPLEX_PDB,
):

    if (
        not path.is_file()
        or path.stat().st_size == 0
    ):

        raise RuntimeError(
            "Complex assembly did not produce:\n"
            f"{path}"
        )


assembly = {
    "status":
        "assembled",

    "target_file":
        str(target_path),

    "target_sha256":
        sha256_file(
            target_path
        ),

    "target_types":
        sorted(
            target_types
        ),

    "leaprc_files":
        leaprc_files,

    "disulfides":
        disulfide_pairs,

    "ligand_kind":
        LIGAND_KIND,

    "complex_prmtop":
        str(
            COMPLEX_PRMTOP
        ),

    "complex_prmtop_sha256":
        sha256_file(
            COMPLEX_PRMTOP
        ),

    "complex_rst7":
        str(
            COMPLEX_RST7
        ),

    "complex_rst7_sha256":
        sha256_file(
            COMPLEX_RST7
        ),

    "complex_pdb":
        str(
            COMPLEX_PDB
        ),

    "complex_pdb_sha256":
        sha256_file(
            COMPLEX_PDB
        ),

    "tleap_log":
        str(
            COMPLEX_LEAP_LOG
        ),
}


if LIGAND_KIND == "small_molecule":

    assembly.update(
        {
            "ligand_mol2":
                str(
                    mol2_path
                ),

            "ligand_mol2_sha256":
                sha256_file(
                    mol2_path
                ),

            "ligand_frcmod":
                str(
                    frcmod_path
                ),

            "ligand_frcmod_sha256":
                sha256_file(
                    frcmod_path
                ),
        }
    )


COMPLEX_ASSEMBLY_PATH.write_text(
    json.dumps(
        assembly,
        indent=2,
    )
)


print("Unsolvated complex assembled")
print("----------------------------")

print(
    "Target:",
    ", ".join(
        sorted(target_types)
    ),
)

print(
    "Disulfides:",
    len(disulfide_pairs),
)

print(
    "Ligand:",
    LIGAND_KIND,
)

print()
print("PDB:", COMPLEX_PDB)
print("PRMTOP:", COMPLEX_PRMTOP)
print("RST7:", COMPLEX_RST7)

$ tleap -I /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/target -I /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/preparation/ligand -f /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/complex/complex.leap.in
Unsolvated complex assembled
----------------------------
Target: protein
Disulfides: 0
Ligand: small_molecule

PDB: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/complex/complex.pdb
PRMTOP: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/complex/complex.prmtop
RST7: /content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/complex/complex.rst7


In [49]:
from pathlib import Path
import gemmi, numpy as np

pdb = Path("/content/drive/MyDrive/oncotarget_pipeline/md/ythdf2/YTHDF2.pdb")
s = gemmi.read_structure(str(pdb))
coords = []
for chain in s[0]:
    for residue in chain:
        for atom in residue:
            if atom.element.atomic_number > 1:
                coords.append([atom.pos.x, atom.pos.y, atom.pos.z])
coords = np.array(coords)
print(f"YTHDF2.pdb:")
print(f"  атомов: {len(coords)}")
print(f"  X: {coords[:,0].min():.2f} .. {coords[:,0].max():.2f}")
print(f"  Y: {coords[:,1].min():.2f} .. {coords[:,1].max():.2f}")
print(f"  Z: {coords[:,2].min():.2f} .. {coords[:,2].max():.2f}")
print(f"  COM: ({coords[:,0].mean():.2f}, {coords[:,1].mean():.2f}, {coords[:,2].mean():.2f})")

YTHDF2.pdb:
  атомов: 4398
  X: -69.09 .. 75.12
  Y: -71.57 .. 66.72
  Z: -66.19 .. 43.94
  COM: (-2.95, -1.78, -8.22)


In [45]:
#@title 31. Final audit of assembled complex

import hashlib
import json
from pathlib import Path

import gemmi
import numpy as np
import parmed as pmd

from rdkit import Chem
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist


COMPLEX_READY_PATH = (
    COMPLEX_DIR
    / "complex_ready.json"
)


MAX_LIGAND_POSE_SHIFT_A = 0.05
SEVERE_TARGET_LIGAND_CLASH_A = 1.50
CHARGE_INTEGER_TOLERANCE = 0.02


def sha256_file(path: Path) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


def read_reference_ligand(
    path: Path,
) -> Chem.Mol:

    suffix = path.suffix.lower()


    if suffix in {
        ".sdf",
        ".sd",
    }:

        supplier = Chem.SDMolSupplier(
            str(path),
            removeHs=False,
            sanitize=True,
        )

        molecules = [
            molecule
            for molecule in supplier
            if molecule is not None
        ]

        if len(molecules) != 1:

            raise RuntimeError(
                "Reference ligand must contain "
                "exactly one molecule."
            )

        return molecules[0]


    if suffix == ".mol":

        molecule = Chem.MolFromMolFile(
            str(path),
            removeHs=False,
            sanitize=True,
        )


    elif suffix == ".mol2":

        molecule = Chem.MolFromMol2File(
            str(path),
            removeHs=False,
            sanitize=True,
        )


    else:

        raise RuntimeError(
            "Unsupported reference ligand format "
            f"for pose validation: {suffix}"
        )


    if molecule is None:

        raise RuntimeError(
            "Could not read reference ligand."
        )


    return molecule


def match_coordinates_by_element(
    reference,
    observed,
):

    if sorted(
        element
        for element, _
        in reference
    ) != sorted(
        element
        for element, _
        in observed
    ):

        return None


    squared_distances = []


    for element in sorted(
        {
            element
            for element, _
            in reference
        }
    ):

        ref_xyz = np.asarray(
            [
                xyz
                for current, xyz
                in reference
                if current == element
            ],
            dtype=float,
        )

        obs_xyz = np.asarray(
            [
                xyz
                for current, xyz
                in observed
                if current == element
            ],
            dtype=float,
        )


        matrix = cdist(
            ref_xyz,
            obs_xyz,
        )


        rows, columns = (
            linear_sum_assignment(
                matrix
            )
        )


        squared_distances.extend(
            matrix[
                rows,
                columns,
            ] ** 2
        )


    distances = np.sqrt(
        np.asarray(
            squared_distances,
            dtype=float,
        )
    )


    return {
        "rmsd_A":
            float(
                np.sqrt(
                    np.mean(
                        distances ** 2
                    )
                )
            ),

        "max_shift_A":
            float(
                distances.max()
            ),
    }


assembly = json.loads(
    COMPLEX_ASSEMBLY_PATH.read_text()
)


if assembly.get("status") != "assembled":

    raise RuntimeError(
        "Complex has not been assembled."
    )


prmtop_path = Path(
    assembly[
        "complex_prmtop"
    ]
)

rst7_path = Path(
    assembly[
        "complex_rst7"
    ]
)

pdb_path = Path(
    assembly[
        "complex_pdb"
    ]
)


for path in (
    prmtop_path,
    rst7_path,
    pdb_path,
):

    if not path.is_file():

        raise FileNotFoundError(
            f"Complex file missing:\n{path}"
        )


if (
    sha256_file(prmtop_path)
    != assembly[
        "complex_prmtop_sha256"
    ]
):

    raise RuntimeError(
        "Complex PRMTOP changed after assembly."
    )


if (
    sha256_file(rst7_path)
    != assembly[
        "complex_rst7_sha256"
    ]
):

    raise RuntimeError(
        "Complex coordinates changed "
        "after assembly."
    )


topology = pmd.load_file(
    str(prmtop_path),
    str(rst7_path),
)


coordinates = np.asarray(
    topology.coordinates,
    dtype=float,
)


if not np.all(
    np.isfinite(
        coordinates
    )
):

    raise RuntimeError(
        "Complex contains non-finite coordinates."
    )


complex_structure = (
    gemmi.read_structure(
        str(pdb_path)
    )
)


if len(complex_structure) != 1:

    raise RuntimeError(
        "Complex PDB must contain "
        "exactly one model."
    )


pdb_atom_count = sum(
    1
    for chain in complex_structure[0]
    for residue in chain
    for atom in residue
)


if pdb_atom_count != len(
    topology.atoms
):

    raise RuntimeError(
        "PDB/topology atom-count mismatch:\n"
        f"PDB:      {pdb_atom_count}\n"
        f"topology: {len(topology.atoms)}"
    )


target_structure = (
    gemmi.read_structure(
        assembly[
            "target_file"
        ]
    )
)


target_residue_count = sum(
    1
    for chain in target_structure[0]
    for residue in chain
)


expected_residue_count = (
    target_residue_count
    + (
        1
        if LIGAND_KIND
        == "small_molecule"
        else 0
    )
)


if len(topology.residues) != expected_residue_count:

    raise RuntimeError(
        "Unexpected residue count in unsolvated complex:\n"
        f"expected: {expected_residue_count}\n"
        f"found:    {len(topology.residues)}"
    )


topology_charge = float(
    sum(
        atom.charge
        for atom
        in topology.atoms
    )
)


nearest_integer_charge = round(
    topology_charge
)


if abs(
    topology_charge
    - nearest_integer_charge
) > CHARGE_INTEGER_TOLERANCE:

    raise RuntimeError(
        "Complex topology has a suspicious "
        "non-integer total charge:\n"
        f"{topology_charge:.6f}"
    )


# --------------------------------------------------------------
# Disulfide validation
# --------------------------------------------------------------

sg_sg_bonds = 0


for bond in topology.bonds:

    if (
        bond.atom1.name == "SG"
        and bond.atom2.name == "SG"
    ):

        sg_sg_bonds += 1


expected_disulfides = len(
    assembly[
        "disulfides"
    ]
)


if sg_sg_bonds != expected_disulfides:

    raise RuntimeError(
        "Disulfide bond count in topology "
        "does not match target preparation:\n"
        f"expected: {expected_disulfides}\n"
        f"found:    {sg_sg_bonds}"
    )


ligand_pose = None
minimum_target_ligand_distance = None
severe_target_ligand_pairs = 0


# --------------------------------------------------------------
# Ligand validation
# --------------------------------------------------------------

if LIGAND_KIND == "small_molecule":

    preparation = json.loads(
        LIGAND_PREP_MANIFEST_PATH.read_text()
    )


    ligand_residues = [
        residue
        for chain in complex_structure[0]
        for residue in chain
        if (
            residue.name
            .strip()
            .upper()
            == LIGAND_RESNAME.upper()
        )
    ]


    if len(ligand_residues) != 1:

        raise RuntimeError(
            "Expected exactly one ligand residue "
            f"named {LIGAND_RESNAME}; "
            f"found {len(ligand_residues)}."
        )


    ligand_residue = (
        ligand_residues[0]
    )


    ligand_atom_count = len(
        ligand_residue
    )


    if ligand_atom_count != int(
        preparation["atoms"]
    ):

        raise RuntimeError(
            "Ligand atom count changed during "
            "complex assembly:\n"
            f"expected: {preparation['atoms']}\n"
            f"found:    {ligand_atom_count}"
        )


    source_ligand = (
        read_reference_ligand(
            Path(
                preparation[
                    "source_file"
                ]
            )
        )
    )


    source_conformer = (
        source_ligand.GetConformer()
    )


    reference_heavy = [
        (
            atom.GetSymbol(),

            np.asarray(
                source_conformer
                .GetAtomPosition(
                    atom.GetIdx()
                ),
                dtype=float,
            ),
        )
        for atom
        in source_ligand.GetAtoms()
        if atom.GetAtomicNum() > 1
    ]


    assembled_heavy = [
        (
            atom.element.name,

            np.asarray(
                [
                    atom.pos.x,
                    atom.pos.y,
                    atom.pos.z,
                ],
                dtype=float,
            ),
        )
        for atom
        in ligand_residue
        if atom.element.atomic_number > 1
    ]


    ligand_pose = (
        match_coordinates_by_element(
            reference_heavy,
            assembled_heavy,
        )
    )


    if ligand_pose is None:

        raise RuntimeError(
            "Assembled ligand no longer matches "
            "the source heavy-atom composition."
        )


    if (
        ligand_pose[
            "max_shift_A"
        ]
        > MAX_LIGAND_POSE_SHIFT_A
    ):

        raise RuntimeError(
            "Ligand docking pose moved during "
            "complex assembly:\n"
            f"{ligand_pose['max_shift_A']:.4f} Å"
        )


    ligand_heavy_xyz = np.asarray(
        [
            [
                atom.pos.x,
                atom.pos.y,
                atom.pos.z,
            ]
            for atom
            in ligand_residue
            if atom.element.atomic_number > 1
        ],
        dtype=float,
    )


    target_heavy_xyz = np.asarray(
        [
            [
                atom.pos.x,
                atom.pos.y,
                atom.pos.z,
            ]
            for chain
            in complex_structure[0]
            for residue in chain
            if residue is not ligand_residue
            for atom in residue
            if atom.element.atomic_number > 1
        ],
        dtype=float,
    )


    if (
        len(target_heavy_xyz) == 0
        or len(ligand_heavy_xyz) == 0
    ):

        raise RuntimeError(
            "Cannot evaluate target-ligand geometry."
        )


    distance_matrix = cdist(
        target_heavy_xyz,
        ligand_heavy_xyz,
    )


    minimum_target_ligand_distance = float(
        distance_matrix.min()
    )


    severe_target_ligand_pairs = int(
        np.sum(
            distance_matrix
            < SEVERE_TARGET_LIGAND_CLASH_A
        )
    )


    if severe_target_ligand_pairs:

        raise RuntimeError(
            "Severe target-ligand heavy-atom "
            "overlap detected:\n"
            f"{severe_target_ligand_pairs} pairs "
            f"below "
            f"{SEVERE_TARGET_LIGAND_CLASH_A:.2f} Å"
        )


ready = {
    "status":
        "ready_for_environment",

    "complex_prmtop":
        str(prmtop_path),

    "complex_prmtop_sha256":
        sha256_file(
            prmtop_path
        ),

    "complex_rst7":
        str(rst7_path),

    "complex_rst7_sha256":
        sha256_file(
            rst7_path
        ),

    "complex_pdb":
        str(pdb_path),

    "complex_pdb_sha256":
        sha256_file(
            pdb_path
        ),

    "atoms":
        len(
            topology.atoms
        ),

    "residues":
        len(
            topology.residues
        ),

    "total_charge":
        topology_charge,

    "disulfides":
        sg_sg_bonds,

    "ligand_kind":
        LIGAND_KIND,
}


if ligand_pose is not None:

    ready.update(
        {
            "ligand_pose_rmsd_A":
                ligand_pose[
                    "rmsd_A"
                ],

            "ligand_pose_max_shift_A":
                ligand_pose[
                    "max_shift_A"
                ],

            "minimum_target_ligand_distance_A":
                minimum_target_ligand_distance,

            "severe_target_ligand_pairs":
                severe_target_ligand_pairs,
        }
    )


COMPLEX_READY_PATH.write_text(
    json.dumps(
        ready,
        indent=2,
    )
)


print("Final complex audit")
print("-------------------")

print(
    "Atoms:",
    len(
        topology.atoms
    ),
)

print(
    "Residues:",
    len(
        topology.residues
    ),
)

print(
    "Total charge:",
    f"{topology_charge:.6f}",
)

print(
    "Disulfides:",
    sg_sg_bonds,
)


if ligand_pose is not None:

    print()
    print(
        "Ligand heavy-atom RMSD:",
        f"{ligand_pose['rmsd_A']:.6f} Å",
    )

    print(
        "Ligand maximum shift:",
        f"{ligand_pose['max_shift_A']:.6f} Å",
    )

    print(
        "Minimum target-ligand distance:",
        f"{minimum_target_ligand_distance:.3f} Å",
    )


print()
print("COMPLEX READY FOR ENVIRONMENT")
print(
    "Manifest:",
    COMPLEX_READY_PATH,
)

RuntimeError: Severe target-ligand heavy-atom overlap detected:
37 pairs below 1.50 Å

In [ ]:
#@title 32. Validate membrane toolchain

import json
import os
import subprocess
import tempfile

from pathlib import Path
from shutil import which


MEMBRANE_DIR = (
    WORKDIR
    / "membrane"
)

MEMBRANE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MEMBRANE_CONFIG_PATH = (
    MEMBRANE_DIR
    / "membrane_config.json"
)


MEMBRANE_ORIENTATION_MODE = "auto" #@param ["auto", "preoriented"]

MEMBRANE_LIPIDS = "POPC" #@param {type:"string"}
MEMBRANE_LIPID_RATIOS = "1" #@param {type:"string"}

MEMBRANE_XY_PADDING_A = 12.0 #@param {type:"number"}
MEMBRANE_WATER_PADDING_A = 20.0 #@param {type:"number"}

IONIC_STRENGTH_M = 0.15 #@param {type:"number"}

POSITIVE_ION = "Na+" #@param ["Na+", "K+"]
NEGATIVE_ION = "Cl-" #@param ["Cl-"]


# ------------------------------------------------------------------
# Configuration validation
# ------------------------------------------------------------------

if SYSTEM_ENVIRONMENT != "membrane":

    raise RuntimeError(
        "The membrane branch was selected for a "
        f"'{SYSTEM_ENVIRONMENT}' system."
    )


if WATER_MODEL.upper() != "TIP3P":

    raise RuntimeError(
        "The current Lipid21 membrane branch "
        "is validated with TIP3P water.\n"
        f"Configured water model: {WATER_MODEL}"
    )


if MEMBRANE_ORIENTATION_MODE not in {
    "auto",
    "preoriented",
}:

    raise ValueError(
        "MEMBRANE_ORIENTATION_MODE must be "
        "'auto' or 'preoriented'."
    )


if MEMBRANE_XY_PADDING_A <= 0:

    raise ValueError(
        "MEMBRANE_XY_PADDING_A must be positive."
    )


if MEMBRANE_WATER_PADDING_A <= 0:

    raise ValueError(
        "MEMBRANE_WATER_PADDING_A must be positive."
    )


if IONIC_STRENGTH_M < 0:

    raise ValueError(
        "IONIC_STRENGTH_M cannot be negative."
    )


lipids = [
    item.strip()
    for item
    in MEMBRANE_LIPIDS.split(":")
    if item.strip()
]


ratio_tokens = [
    item.strip()
    for item
    in MEMBRANE_LIPID_RATIOS.split(":")
    if item.strip()
]


if not lipids:

    raise ValueError(
        "At least one lipid must be specified."
    )


if len(lipids) != len(
    ratio_tokens
):

    raise ValueError(
        "MEMBRANE_LIPIDS and "
        "MEMBRANE_LIPID_RATIOS must contain "
        "the same number of entries."
    )


try:

    lipid_ratios = [
        int(value)
        for value
        in ratio_tokens
    ]

except ValueError as exc:

    raise ValueError(
        "Membrane lipid ratios must be integers."
    ) from exc


if any(
    value <= 0
    for value
    in lipid_ratios
):

    raise ValueError(
        "Membrane lipid ratios must be positive."
    )


# ------------------------------------------------------------------
# Discover executables
# ------------------------------------------------------------------

program_names = (
    "packmol-memgen",
    "packmol",
    "tleap",
)


program_paths = {
    name: which(name)
    for name
    in program_names
}


missing_programs = [
    name
    for name, path
    in program_paths.items()
    if path is None
]


if missing_programs:

    raise RuntimeError(
        "Required membrane-building programs "
        "are unavailable:\n"
        + "\n".join(
            f"- {name}"
            for name
            in missing_programs
        )
    )


program_paths = {
    name: str(
        Path(path).resolve()
    )
    for name, path
    in program_paths.items()
}


packmol_path = (
    program_paths["packmol"]
)

packmol_memgen_path = (
    program_paths["packmol-memgen"]
)

tleap_path = (
    program_paths["tleap"]
)


# ------------------------------------------------------------------
# PACKMOL integration test
#
# Important:
# PACKMOL expects redirected FILE stdin:
#
#     packmol < input.inp
#
# Do NOT use subprocess.run(input=...).
# That creates a pipe, and some Fortran PACKMOL builds seek on
# unit 5, causing:
#
#     Fortran runtime error: Illegal seek
# ------------------------------------------------------------------

with tempfile.TemporaryDirectory() as tmp:

    tmp_dir = Path(tmp)

    molecule_path = (
        tmp_dir
        / "molecule.pdb"
    )

    input_path = (
        tmp_dir
        / "packmol.inp"
    )

    output_path = (
        tmp_dir
        / "packed.pdb"
    )


    molecule_path.write_text(
        "HETATM    1  C1  MOL A   1       "
        "0.000   0.000   0.000  1.00  0.00           C\n"
        "END\n"
    )


    input_path.write_text(
        "\n".join(
            [
                "tolerance 2.0",
                "filetype pdb",
                f"output {output_path.name}",
                "",
                f"structure {molecule_path.name}",
                "  number 1",
                "  inside box 0. 0. 0. 10. 10. 10.",
                "end structure",
                "",
            ]
        )
    )


    # File-backed stdin is deliberate.
    with input_path.open(
        "r"
    ) as stdin_handle:

        smoke = subprocess.run(
            [
                packmol_path
            ],
            stdin=stdin_handle,
            cwd=tmp_dir,
            capture_output=True,
            text=True,
        )


    if smoke.returncode != 0:

        raise RuntimeError(
            "PACKMOL failed a file-backed "
            "minimal packing test.\n\n"
            f"Executable:\n"
            f"{packmol_path}\n\n"
            f"STDOUT:\n"
            f"{smoke.stdout[-3000:]}\n\n"
            f"STDERR:\n"
            f"{smoke.stderr[-3000:]}"
        )


    if (
        not output_path.is_file()
        or output_path.stat().st_size == 0
    ):

        raise RuntimeError(
            "PACKMOL returned successfully but "
            "did not create its output file."
        )


# ------------------------------------------------------------------
# Inspect the installed PACKMOL-Memgen CLI.
#
# Do not assume flags that may differ between AmberTools builds.
# ------------------------------------------------------------------

help_result = subprocess.run(
    [
        packmol_memgen_path,
        "--help",
    ],
    capture_output=True,
    text=True,
)


help_text = (
    help_result.stdout
    + "\n"
    + help_result.stderr
)


if not help_text.strip():

    raise RuntimeError(
        "PACKMOL-Memgen produced no help output."
    )


required_memgen_options = (
    "--pdb",
    "--lipids",
    "--ratio",
    "--dist",
    "--dist_wat",
    "--notprotonate",
    "--nottrim",
)


missing_options = [
    option
    for option
    in required_memgen_options
    if option not in help_text
]


if missing_options:

    raise RuntimeError(
        "Installed PACKMOL-Memgen exposes an "
        "unexpected CLI.\n\n"
        "Missing options:\n"
        + "\n".join(
            f"- {option}"
            for option
            in missing_options
        )
    )


capabilities = {
    "explicit_packmol_argument":
        "--packmol"
        in help_text,

    "output_argument":
        "--output"
        in help_text,

    "overwrite_argument":
        "--overwrite"
        in help_text,

    "keepligs_argument":
        "--keepligs"
        in help_text,

    "preoriented_argument":
        "--preoriented"
        in help_text,
}


if (
    MEMBRANE_ORIENTATION_MODE
    == "preoriented"
    and not capabilities[
        "preoriented_argument"
    ]
):

    raise RuntimeError(
        "The installed PACKMOL-Memgen "
        "does not support --preoriented."
    )


# ------------------------------------------------------------------
# Make PACKMOL discovery explicit for PACKMOL-Memgen.
#
# Some AmberTools/conda builds historically had fragile bundled vs
# external PACKMOL discovery.  We provide both:
#
#   1. PACKMOL directory on PATH
#   2. PACKMOL_PATH environment variable
#
# If the installed CLI also exposes --packmol, later blocks can use
# that in addition.
# ------------------------------------------------------------------

packmol_directory = str(
    Path(packmol_path).parent
)


current_path = (
    os.environ.get(
        "PATH",
        "",
    )
)


path_entries = (
    current_path.split(
        os.pathsep
    )
    if current_path
    else []
)


if (
    packmol_directory
    not in path_entries
):

    os.environ[
        "PATH"
    ] = (
        packmol_directory
        + os.pathsep
        + current_path
    )


os.environ[
    "PACKMOL_PATH"
] = packmol_path


# ------------------------------------------------------------------
# Amber/Lipid21 availability
# ------------------------------------------------------------------

amber_prefix = (
    Path(tleap_path)
    .parent
    .parent
)


leap_command_dir = (
    amber_prefix
    / "dat"
    / "leap"
    / "cmd"
)


lipid21_leaprc = (
    leap_command_dir
    / "leaprc.lipid21"
)


if not lipid21_leaprc.is_file():

    raise RuntimeError(
        "Lipid21 is not available in "
        "the installed AmberTools environment:\n"
        f"{lipid21_leaprc}"
    )


# ------------------------------------------------------------------
# Persistent configuration
# ------------------------------------------------------------------

config = {
    "orientation_mode":
        MEMBRANE_ORIENTATION_MODE,

    "lipids":
        lipids,

    "ratios":
        lipid_ratios,

    "xy_padding_A":
        float(
            MEMBRANE_XY_PADDING_A
        ),

    "water_padding_A":
        float(
            MEMBRANE_WATER_PADDING_A
        ),

    "ionic_strength_M":
        float(
            IONIC_STRENGTH_M
        ),

    "positive_ion":
        POSITIVE_ION,

    "negative_ion":
        NEGATIVE_ION,

    "water_model":
        WATER_MODEL,

    "lipid_force_field":
        "Lipid21",

    "executables": {
        "packmol_memgen":
            packmol_memgen_path,

        "packmol":
            packmol_path,

        "tleap":
            tleap_path,
    },

    "capabilities":
        capabilities,

    "environment": {
        "PACKMOL_PATH":
            packmol_path,
    },
}


MEMBRANE_CONFIG_PATH.write_text(
    json.dumps(
        config,
        indent=2,
    )
)


print("Membrane toolchain validated")
print("----------------------------")

print(
    "PACKMOL-Memgen:",
    packmol_memgen_path,
)

print(
    "PACKMOL:",
    packmol_path,
)

print(
    "PACKMOL file-backed smoke test: PASS"
)

print(
    "PACKMOL_PATH:",
    os.environ[
        "PACKMOL_PATH"
    ],
)

print()
print("PACKMOL-Memgen capabilities")

for name, supported in (
    capabilities.items()
):

    print(
        f"  {name}: "
        f"{supported}"
    )


print()
print(
    "Lipid21:",
    lipid21_leaprc,
)

print(
    "Lipids:",
    ":".join(
        lipids
    ),
)

print(
    "Ratio:",
    ":".join(
        str(value)
        for value
        in lipid_ratios
    ),
)

print()
print(
    "Configuration:",
    MEMBRANE_CONFIG_PATH,
)

In [ ]:
#@title 33. Build membrane environment with live monitoring

import codecs
import hashlib
import json
import os
import re
import selectors
import shlex
import shutil
import signal
import subprocess
import time

from collections import Counter
from pathlib import Path

import gemmi
import numpy as np


# ==============================================================
# Stage paths
#
# Heavy external work is performed on the local Colab filesystem.
# Only validated artifacts and diagnostics are persisted to Drive.
# ==============================================================

MEMBRANE_BUILD_RECORD_PATH = (
    MEMBRANE_DIR
    / "membrane_build.json"
)

PROTEIN_SCAFFOLD_PATH = (
    MEMBRANE_DIR
    / "protein_scaffold.pdb"
)

PACKED_REFERENCE_PATH = (
    MEMBRANE_DIR
    / "packmol_memgen_system.pdb"
)

ORIENTED_COMPLEX_PATH = (
    MEMBRANE_DIR
    / "oriented_complex.pdb"
)

MEMBRANE_ENVIRONMENT_PATH = (
    MEMBRANE_DIR
    / "membrane_environment.pdb"
)

MEMBRANE_BUILD_LOG = (
    LOG_DIR
    / "packmol_memgen_build.log"
)

MEMBRANE_DIAGNOSTICS_DIR = (
    MEMBRANE_DIR
    / "build_diagnostics"
)


SCRATCH_DIR = Path(
    "/content/oncotarget_membrane_build"
)


# ==============================================================
# Monitoring policy
# ==============================================================

HEARTBEAT_INTERVAL_S = 30.0

LOW_ACTIVITY_WARNING_AFTER_S = 300.0

LOW_CPU_THRESHOLD_PERCENT = 5.0


# This is a validation threshold, not a packing parameter.
MAX_PROTEIN_RIGID_FIT_RMSD_A = 0.25


STANDARD_PROTEIN_NAMES = {
    "ALA", "ARG", "ASN", "ASP",
    "CYS", "GLN", "GLU", "GLY",
    "HIS", "ILE", "LEU", "LYS",
    "MET", "PHE", "PRO", "SER",
    "THR", "TRP", "TYR", "VAL",

    "ASH", "CYM", "CYX",
    "GLH", "HID", "HIE",
    "HIP", "HIN", "LYN",
}


WATER_RESIDUE_NAMES = {
    "WAT",
    "HOH",
    "TIP3",
}


# ==============================================================
# Generic helpers
# ==============================================================

def sha256_file(
    path: Path,
) -> str:

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as handle:

        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


def human_bytes(
    value: int,
) -> str:

    size = float(value)

    for unit_name in (
        "B",
        "KB",
        "MB",
        "GB",
        "TB",
    ):

        if size < 1024.0:

            return (
                f"{size:.2f} "
                f"{unit_name}"
            )

        size /= 1024.0

    return f"{size:.2f} PB"


def format_elapsed(
    seconds: float,
) -> str:

    seconds = int(
        max(
            0,
            seconds,
        )
    )

    hours, remainder = divmod(
        seconds,
        3600,
    )

    minutes, seconds = divmod(
        remainder,
        60,
    )

    return (
        f"{hours:02d}:"
        f"{minutes:02d}:"
        f"{seconds:02d}"
    )


def canonical_protein_name(
    residue_name: str,
):

    name = (
        residue_name
        .strip()
        .upper()
    )


    if name in STANDARD_PROTEIN_NAMES:

        return name


    # Amber PDB output can expose terminal templates
    # using an NXXX/CXXX residue name.
    if (
        len(name) == 4
        and name[0] in {
            "N",
            "C",
        }
        and name[1:]
        in STANDARD_PROTEIN_NAMES
    ):

        return name[1:]


    return None


def is_protein_residue(
    residue: gemmi.Residue,
) -> bool:

    return (
        canonical_protein_name(
            residue.name
        )
        is not None
    )


def protein_backbone(
    structure: gemmi.Structure,
):

    sequence = []
    coordinates = []


    for chain in structure[0]:

        for residue in chain:

            canonical = (
                canonical_protein_name(
                    residue.name
                )
            )


            if canonical is None:

                continue


            atom_map = {
                atom.name.strip():
                    atom
                for atom in residue
            }


            missing = (
                {
                    "N",
                    "CA",
                    "C",
                }
                - set(
                    atom_map
                )
            )


            if missing:

                raise RuntimeError(
                    "Protein residue lacks "
                    "required backbone atoms:\n"
                    f"{chain.name}:"
                    f"{residue.name}"
                    f"{residue.seqid.num}\n"
                    "Missing: "
                    + ", ".join(
                        sorted(
                            missing
                        )
                    )
                )


            sequence.append(
                canonical
            )


            for atom_name in (
                "N",
                "CA",
                "C",
            ):

                atom = atom_map[
                    atom_name
                ]

                coordinates.append(
                    [
                        atom.pos.x,
                        atom.pos.y,
                        atom.pos.z,
                    ]
                )


    if not sequence:

        raise RuntimeError(
            "No protein backbone "
            "was detected."
        )


    return (
        tuple(sequence),

        np.asarray(
            coordinates,
            dtype=float,
        ),
    )


def kabsch_transform(
    mobile: np.ndarray,
    target: np.ndarray,
):

    if (
        mobile.shape
        != target.shape
    ):

        raise ValueError(
            "Kabsch arrays have "
            "different shapes."
        )


    if (
        mobile.ndim != 2
        or mobile.shape[1] != 3
    ):

        raise ValueError(
            "Kabsch coordinates must "
            "have shape (N, 3)."
        )


    mobile_center = (
        mobile.mean(
            axis=0
        )
    )

    target_center = (
        target.mean(
            axis=0
        )
    )


    mobile_zero = (
        mobile
        - mobile_center
    )

    target_zero = (
        target
        - target_center
    )


    covariance = (
        mobile_zero.T
        @ target_zero
    )


    u, _, vt = (
        np.linalg.svd(
            covariance
        )
    )


    rotation = (
        u
        @ vt
    )


    if (
        np.linalg.det(
            rotation
        )
        < 0
    ):

        u[:, -1] *= -1

        rotation = (
            u
            @ vt
        )


    translation = (
        target_center
        - mobile_center
        @ rotation
    )


    fitted = (
        mobile
        @ rotation
        + translation
    )


    rmsd = float(
        np.sqrt(
            np.mean(
                np.sum(
                    (
                        fitted
                        - target
                    )
                    ** 2,
                    axis=1,
                )
            )
        )
    )


    return (
        rotation,
        translation,
        rmsd,
    )


def apply_rigid_transform(
    structure: gemmi.Structure,
    rotation: np.ndarray,
    translation: np.ndarray,
):

    for chain in structure[0]:

        for residue in chain:

            for atom in residue:

                xyz = np.asarray(
                    [
                        atom.pos.x,
                        atom.pos.y,
                        atom.pos.z,
                    ],
                    dtype=float,
                )


                transformed = (
                    xyz
                    @ rotation
                    + translation
                )


                atom.pos = gemmi.Position(
                    float(
                        transformed[0]
                    ),
                    float(
                        transformed[1]
                    ),
                    float(
                        transformed[2]
                    ),
                )


def extract_environment(
    packed_structure: gemmi.Structure,
):

    environment = (
        packed_structure.clone()
    )


    for chain_index in range(
        len(
            environment[0]
        ) - 1,
        -1,
        -1,
    ):

        chain = environment[0][
            chain_index
        ]


        for residue_index in range(
            len(chain) - 1,
            -1,
            -1,
        ):

            residue = chain[
                residue_index
            ]


            if is_protein_residue(
                residue
            ):

                del chain[
                    residue_index
                ]


        if len(chain) == 0:

            del environment[0][
                chain_index
            ]


    return environment


# ==============================================================
# Diagnostics persistence
# ==============================================================

def persist_diagnostics(
    scratch_dir: Path,
):

    if (
        MEMBRANE_DIAGNOSTICS_DIR
        .exists()
    ):

        shutil.rmtree(
            MEMBRANE_DIAGNOSTICS_DIR
        )


    MEMBRANE_DIAGNOSTICS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )


    interesting_suffixes = {
        ".log",
        ".inp",
        ".out",
        ".txt",
        ".json",
    }


    for source in (
        scratch_dir.rglob("*")
    ):

        if not source.is_file():

            continue


        if (
            source.suffix.lower()
            not in interesting_suffixes
        ):

            continue


        relative = (
            source.relative_to(
                scratch_dir
            )
        )


        destination = (
            MEMBRANE_DIAGNOSTICS_DIR
            / relative
        )


        destination.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        shutil.copy2(
            source,
            destination,
        )


# ==============================================================
# Process-tree telemetry
# ==============================================================

try:

    import psutil

except ImportError:

    psutil = None


def process_tree_metrics(
    root_pid: int,
):

    if psutil is None:

        return {
            "processes": None,
            "rss_bytes": None,
            "cpu_seconds": None,
        }


    try:

        root = psutil.Process(
            root_pid
        )

        processes = [
            root
        ] + root.children(
            recursive=True
        )

    except (
        psutil.NoSuchProcess,
        psutil.AccessDenied,
    ):

        return {
            "processes": 0,
            "rss_bytes": 0,
            "cpu_seconds": 0.0,
        }


    rss = 0

    cpu_seconds = 0.0

    alive = 0


    for process in processes:

        try:

            if not process.is_running():

                continue


            alive += 1


            rss += (
                process
                .memory_info()
                .rss
            )


            cpu_times = (
                process.cpu_times()
            )


            cpu_seconds += (
                cpu_times.user
                + cpu_times.system
            )

        except (
            psutil.NoSuchProcess,
            psutil.AccessDenied,
        ):

            continue


    return {
        "processes":
            alive,

        "rss_bytes":
            rss,

        "cpu_seconds":
            cpu_seconds,
    }


# ==============================================================
# Live external-process runner
#
# - stdout + stderr streamed to notebook
# - same output written to local log
# - heartbeat every N seconds
# - child PACKMOL process included in CPU/RAM telemetry
# - external *.log files mirrored when they grow
# - Ctrl+C terminates the whole process group
# ==============================================================

def run_streaming_process(
    command,
    cwd: Path,
    local_log_path: Path,
    persistent_log_path: Path,
    env: dict,
):

    print(
        "$",
        shlex.join(
            [
                str(value)
                for value
                in command
            ]
        ),
        flush=True,
    )

    print(
        "\nLive membrane-build output",
        flush=True,
    )

    print(
        "--------------------------",
        flush=True,
    )


    process = subprocess.Popen(
        [
            str(value)
            for value
            in command
        ],

        cwd=cwd,

        env=env,

        stdout=subprocess.PIPE,

        stderr=subprocess.STDOUT,

        bufsize=0,

        start_new_session=True,
    )


    if process.stdout is None:

        raise RuntimeError(
            "Could not capture "
            "PACKMOL-Memgen output."
        )


    stdout_fd = (
        process.stdout.fileno()
    )


    os.set_blocking(
        stdout_fd,
        False,
    )


    selector = (
        selectors.DefaultSelector()
    )


    selector.register(
        stdout_fd,
        selectors.EVENT_READ,
    )


    decoder = (
        codecs
        .getincrementaldecoder(
            "utf-8"
        )(
            errors="replace"
        )
    )


    start_time = (
        time.monotonic()
    )

    last_heartbeat = (
        start_time
    )

    last_visible_output = (
        start_time
    )


    initial_metrics = (
        process_tree_metrics(
            process.pid
        )
    )


    previous_cpu_seconds = (
        initial_metrics[
            "cpu_seconds"
        ]
    )

    previous_cpu_time = (
        start_time
    )


    external_log_offsets = {}


    with local_log_path.open(
        "w",
        encoding="utf-8",
        buffering=1,
    ) as log_handle:


        def emit(
            text: str,
        ):

            nonlocal last_visible_output


            print(
                text,
                end="",
                flush=True,
            )


            log_handle.write(
                text
            )

            log_handle.flush()


            last_visible_output = (
                time.monotonic()
            )


        def mirror_external_logs():

            for log_path in sorted(
                cwd.rglob(
                    "*.log"
                )
            ):

                if (
                    log_path.resolve()
                    == local_log_path.resolve()
                ):

                    continue


                try:

                    size = (
                        log_path.stat()
                        .st_size
                    )

                except FileNotFoundError:

                    continue


                previous_size = (
                    external_log_offsets
                    .get(
                        log_path,
                        0,
                    )
                )


                if (
                    size
                    <= previous_size
                ):

                    continue


                # We show the newly written tail.
                # Extremely large bursts are truncated only
                # in the notebook; the original file itself
                # is retained in diagnostics.
                bytes_to_read = min(
                    size - previous_size,
                    16_384,
                )


                read_start = max(
                    previous_size,
                    size
                    - bytes_to_read,
                )


                try:

                    with log_path.open(
                        "r",
                        encoding="utf-8",
                        errors="replace",
                    ) as external:

                        external.seek(
                            read_start
                        )

                        chunk = (
                            external.read()
                        )

                except OSError:

                    continue


                external_log_offsets[
                    log_path
                ] = size


                if chunk.strip():

                    emit(
                        "\n"
                        f"[live log: "
                        f"{log_path.name}]\n"
                    )

                    emit(
                        chunk
                    )

                    if not chunk.endswith(
                        "\n"
                    ):

                        emit(
                            "\n"
                        )


        try:

            while True:

                events = (
                    selector.select(
                        timeout=1.0
                    )
                )


                for _, _ in events:

                    try:

                        data = os.read(
                            stdout_fd,
                            65_536,
                        )

                    except BlockingIOError:

                        data = b""


                    if data:

                        text = (
                            decoder.decode(
                                data
                            )
                        )


                        if text:

                            emit(
                                text
                            )


                now = (
                    time.monotonic()
                )


                if (
                    now
                    - last_heartbeat
                    >= HEARTBEAT_INTERVAL_S
                ):

                    metrics = (
                        process_tree_metrics(
                            process.pid
                        )
                    )


                    interval = max(
                        now
                        - previous_cpu_time,
                        1e-9,
                    )


                    current_cpu_seconds = (
                        metrics[
                            "cpu_seconds"
                        ]
                    )


                    if (
                        current_cpu_seconds
                        is not None
                        and previous_cpu_seconds
                        is not None
                    ):

                        cpu_percent = (
                            100.0
                            * (
                                current_cpu_seconds
                                - previous_cpu_seconds
                            )
                            / interval
                        )

                    else:

                        cpu_percent = None


                    previous_cpu_seconds = (
                        current_cpu_seconds
                    )

                    previous_cpu_time = (
                        now
                    )


                    output_candidates = [
                        path
                        for path in cwd.glob(
                            "*.pdb"
                        )
                        if path.is_file()
                    ]


                    largest_output = (
                        max(
                            (
                                path.stat().st_size
                                for path
                                in output_candidates
                            ),
                            default=0,
                        )
                    )


                    heartbeat = (
                        "\n"
                        "["
                        + format_elapsed(
                            now
                            - start_time
                        )
                        + "] RUNNING"
                    )


                    if (
                        metrics[
                            "processes"
                        ]
                        is not None
                    ):

                        heartbeat += (
                            " | processes="
                            f"{metrics['processes']}"
                        )


                    if cpu_percent is not None:

                        heartbeat += (
                            " | CPU="
                            f"{cpu_percent:.1f}%"
                        )


                    if (
                        metrics[
                            "rss_bytes"
                        ]
                        is not None
                    ):

                        heartbeat += (
                            " | RAM="
                            + human_bytes(
                                metrics[
                                    "rss_bytes"
                                ]
                            )
                        )


                    heartbeat += (
                        " | largest PDB="
                        + human_bytes(
                            largest_output
                        )
                    )


                    heartbeat += (
                        " | no-console-output="
                        + format_elapsed(
                            now
                            - last_visible_output
                        )
                    )


                    heartbeat += "\n"


                    # Heartbeat itself should not reset the
                    # external-output inactivity timer.
                    print(
                        heartbeat,
                        end="",
                        flush=True,
                    )

                    log_handle.write(
                        heartbeat
                    )

                    log_handle.flush()


                    mirror_external_logs()


                    # Persist a checkpoint log to Drive only
                    # every heartbeat, not on every stdout line.
                    try:

                        shutil.copy2(
                            local_log_path,
                            persistent_log_path,
                        )

                    except OSError as exc:

                        print(
                            "[warning] Could not checkpoint "
                            "live log to Drive: "
                            f"{exc}",
                            flush=True,
                        )


                    if (
                        cpu_percent is not None
                        and cpu_percent
                        < LOW_CPU_THRESHOLD_PERCENT
                        and (
                            now
                            - last_visible_output
                            > LOW_ACTIVITY_WARNING_AFTER_S
                        )
                    ):

                        warning = (
                            "[warning] Low process-tree CPU "
                            "and no new console output for "
                            "several minutes. The process is "
                            "still alive, but this may indicate "
                            "a stalled packing stage.\n"
                        )


                        print(
                            warning,
                            end="",
                            flush=True,
                        )

                        log_handle.write(
                            warning
                        )

                        log_handle.flush()


                    last_heartbeat = (
                        now
                    )


                return_code = (
                    process.poll()
                )


                if return_code is not None:

                    # Drain anything still buffered after exit.
                    while True:

                        try:

                            data = os.read(
                                stdout_fd,
                                65_536,
                            )

                        except BlockingIOError:

                            break


                        if not data:

                            break


                        text = (
                            decoder.decode(
                                data
                            )
                        )


                        if text:

                            emit(
                                text
                            )


                    remaining = (
                        decoder.decode(
                            b"",
                            final=True,
                        )
                    )


                    if remaining:

                        emit(
                            remaining
                        )


                    break


        except KeyboardInterrupt:

            print(
                "\n"
                "Interrupt received. "
                "Stopping PACKMOL-Memgen "
                "and all child processes...",
                flush=True,
            )


            try:

                os.killpg(
                    process.pid,
                    signal.SIGTERM,
                )

            except ProcessLookupError:

                pass


            try:

                process.wait(
                    timeout=5
                )

            except subprocess.TimeoutExpired:

                try:

                    os.killpg(
                        process.pid,
                        signal.SIGKILL,
                    )

                except ProcessLookupError:

                    pass


                process.wait()


            log_handle.flush()


            shutil.copy2(
                local_log_path,
                persistent_log_path,
            )


            persist_diagnostics(
                cwd
            )


            print(
                "Process tree stopped. "
                "Diagnostics were preserved.",
                flush=True,
            )


            raise


        finally:

            selector.close()


    shutil.copy2(
        local_log_path,
        persistent_log_path,
    )


    if process.returncode != 0:

        persist_diagnostics(
            cwd
        )


        text = (
            local_log_path
            .read_text(
                errors="replace"
            )
        )


        tail = "\n".join(
            text.splitlines()[
                -80:
            ]
        )


        raise RuntimeError(
            "PACKMOL-Memgen failed with "
            f"exit code "
            f"{process.returncode}.\n\n"
            "Last output:\n"
            + tail
            + "\n\nFull log:\n"
            + str(
                persistent_log_path
            )
        )


    return (
        time.monotonic()
        - start_time
    )


# ==============================================================
# Periodic-box recovery
# ==============================================================

def determine_box_lengths(
    packed_path: Path,
    scratch_dir: Path,
):

    structure = gemmi.read_structure(
        str(
            packed_path
        )
    )


    # Preferred source: explicit CRYST1/unit-cell
    # metadata produced by the builder.
    if (
        structure.cell.a > 1.0
        and structure.cell.b > 1.0
        and structure.cell.c > 1.0
    ):

        return (
            [
                float(
                    structure.cell.a
                ),
                float(
                    structure.cell.b
                ),
                float(
                    structure.cell.c
                ),
            ],

            "PDB_CRYST1",
        )


    # Second source: PACKMOL-Memgen textual output.
    text_sources = [
        path
        for path in scratch_dir.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in {
                ".log",
                ".out",
                ".txt",
            }
            and path.stat().st_size
            < 50_000_000
        )
    ]


    patterns = (
        re.compile(
            r"Box\s+X,\s*Y,\s*Z:\s*"
            r"([-+0-9.eE]+)\s+"
            r"([-+0-9.eE]+)\s+"
            r"([-+0-9.eE]+)",
            flags=re.IGNORECASE,
        ),

        re.compile(
            r"Box\s+X.*?=\s*"
            r"([-+0-9.eE]+).*?"
            r"Box\s+Y.*?=\s*"
            r"([-+0-9.eE]+).*?"
            r"Box\s+Z.*?=\s*"
            r"([-+0-9.eE]+)",
            flags=(
                re.IGNORECASE
                | re.DOTALL
            ),
        ),
    )


    for path in text_sources:

        try:

            text = path.read_text(
                errors="ignore"
            )

        except OSError:

            continue


        for pattern in patterns:

            match = pattern.search(
                text
            )


            if match:

                return (
                    [
                        float(
                            match.group(1)
                        ),
                        float(
                            match.group(2)
                        ),
                        float(
                            match.group(3)
                        ),
                    ],

                    f"text:{path.name}",
                )


    # Last-resort diagnostic fallback:
    # recover global packing bounds from generated PACKMOL inputs.
    inside_box_pattern = re.compile(
        r"inside\s+box\s+"
        r"([-+0-9.eE]+)\s+"
        r"([-+0-9.eE]+)\s+"
        r"([-+0-9.eE]+)\s+"
        r"([-+0-9.eE]+)\s+"
        r"([-+0-9.eE]+)\s+"
        r"([-+0-9.eE]+)",
        flags=re.IGNORECASE,
    )


    bounds = []


    for input_path in scratch_dir.rglob(
        "*.inp"
    ):

        try:

            text = input_path.read_text(
                errors="ignore"
            )

        except OSError:

            continue


        for match in (
            inside_box_pattern
            .finditer(
                text
            )
        ):

            bounds.append(
                [
                    float(
                        match.group(index)
                    )
                    for index
                    in range(
                        1,
                        7,
                    )
                ]
            )


    if bounds:

        bounds_array = np.asarray(
            bounds,
            dtype=float,
        )


        minimum = np.min(
            bounds_array[
                :,
                :3
            ],
            axis=0,
        )

        maximum = np.max(
            bounds_array[
                :,
                3:
            ],
            axis=0,
        )


        lengths = (
            maximum
            - minimum
        )


        if np.all(
            lengths > 0
        ):

            return (
                [
                    float(value)
                    for value
                    in lengths
                ],

                "PACKMOL_constraints",
            )


    raise RuntimeError(
        "Periodic box dimensions could "
        "not be recovered from the "
        "PACKMOL-Memgen output."
    )


# ==============================================================
# Load persistent stage contracts
# ==============================================================

if not MEMBRANE_CONFIG_PATH.is_file():

    raise FileNotFoundError(
        "Membrane configuration not found:\n"
        f"{MEMBRANE_CONFIG_PATH}"
    )


if not COMPLEX_READY_PATH.is_file():

    raise FileNotFoundError(
        "Validated complex manifest not found:\n"
        f"{COMPLEX_READY_PATH}"
    )


config = json.loads(
    MEMBRANE_CONFIG_PATH.read_text()
)


complex_ready = json.loads(
    COMPLEX_READY_PATH.read_text()
)


if (
    complex_ready.get(
        "status"
    )
    != "ready_for_environment"
):

    raise RuntimeError(
        "Complex has not passed "
        "the final assembly audit."
    )


complex_path = Path(
    complex_ready[
        "complex_pdb"
    ]
)


if not complex_path.is_file():

    raise FileNotFoundError(
        "Validated complex disappeared:\n"
        f"{complex_path}"
    )


if (
    sha256_file(
        complex_path
    )
    != complex_ready[
        "complex_pdb_sha256"
    ]
):

    raise RuntimeError(
        "Validated complex changed "
        "after block 31."
    )


memgen_path = Path(
    config[
        "executables"
    ][
        "packmol_memgen"
    ]
)


packmol_path = Path(
    config[
        "executables"
    ][
        "packmol"
    ]
)


for executable in (
    memgen_path,
    packmol_path,
):

    if not executable.is_file():

        raise FileNotFoundError(
            "Required executable missing:\n"
            f"{executable}"
        )


# ==============================================================
# Probe the installed CLI instead of assuming a package version
# ==============================================================

help_result = subprocess.run(
    [
        str(
            memgen_path
        ),
        "--help",
    ],

    capture_output=True,
    text=True,
)


help_text = (
    help_result.stdout
    + "\n"
    + help_result.stderr
)


supports_verbose = (
    "--verbose"
    in help_text
)


# --packmol is hidden from --help in some AmberTools builds.
# Probe it directly.  If unsupported, use a clean PATH-based
# discovery instead of trusting a possibly stale PACKMOL_PATH.
explicit_packmol_probe = subprocess.run(
    [
        str(
            memgen_path
        ),

        "--packmol",
        str(
            packmol_path
        ),

        "--help",
    ],

    capture_output=True,
    text=True,
)


use_explicit_packmol = (
    explicit_packmol_probe
    .returncode
    == 0
)


# ==============================================================
# Fresh local scratch
# ==============================================================

if SCRATCH_DIR.exists():

    shutil.rmtree(
        SCRATCH_DIR
    )


SCRATCH_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


LOCAL_SCAFFOLD_PATH = (
    SCRATCH_DIR
    / "protein_scaffold.pdb"
)

LOCAL_PACKED_PATH = (
    SCRATCH_DIR
    / "packed_system.pdb"
)

LOCAL_STREAM_LOG = (
    SCRATCH_DIR
    / "memgen_stream.log"
)


# Remove only outputs owned by THIS stage.
for stale_output in (
    MEMBRANE_BUILD_RECORD_PATH,
    PROTEIN_SCAFFOLD_PATH,
    PACKED_REFERENCE_PATH,
    ORIENTED_COMPLEX_PATH,
    MEMBRANE_ENVIRONMENT_PATH,
):

    stale_output.unlink(
        missing_ok=True
    )


# ==============================================================
# Extract a protonated protein scaffold from the validated
# LEaP-built complex.
# ==============================================================

validated_complex = (
    gemmi.read_structure(
        str(
            complex_path
        )
    )
)


if len(
    validated_complex
) != 1:

    raise RuntimeError(
        "Validated complex must contain "
        "exactly one model."
    )


scaffold = (
    validated_complex.clone()
)


protein_residue_count = 0


for chain_index in range(
    len(
        scaffold[0]
    ) - 1,
    -1,
    -1,
):

    chain = scaffold[0][
        chain_index
    ]


    for residue_index in range(
        len(chain) - 1,
        -1,
        -1,
    ):

        residue = chain[
            residue_index
        ]


        if is_protein_residue(
            residue
        ):

            protein_residue_count += 1

        else:

            del chain[
                residue_index
            ]


    if len(chain) == 0:

        del scaffold[0][
            chain_index
        ]


if protein_residue_count == 0:

    raise RuntimeError(
        "No protein scaffold could "
        "be extracted."
    )


protein_hydrogen_count = sum(
    1
    for chain in scaffold[0]
    for residue in chain
    for atom in residue
    if atom.element.atomic_number == 1
)


if protein_hydrogen_count == 0:

    raise RuntimeError(
        "Protein scaffold contains no "
        "hydrogens.\n\n"
        "The --notprotonate route requires "
        "a protonated structure."
    )


scaffold.write_pdb(
    str(
        LOCAL_SCAFFOLD_PATH
    )
)


shutil.copy2(
    LOCAL_SCAFFOLD_PATH,
    PROTEIN_SCAFFOLD_PATH,
)


original_sequence, original_backbone = (
    protein_backbone(
        scaffold
    )
)


# ==============================================================
# Build command
# ==============================================================

command = [
    str(
        memgen_path
    ),
]


if use_explicit_packmol:

    command.extend(
        [
            "--packmol",
            str(
                packmol_path
            ),
        ]
    )


command.extend(
    [
        "--pdb",
        str(
            LOCAL_SCAFFOLD_PATH
        ),

        "--lipids",
        ":".join(
            config[
                "lipids"
            ]
        ),

        "--ratio",
        ":".join(
            str(value)
            for value
            in config[
                "ratios"
            ]
        ),

        "--dist",
        str(
            config[
                "xy_padding_A"
            ]
        ),

        "--dist_wat",
        str(
            config[
                "water_padding_A"
            ]
        ),

        "--notprotonate",
        "--nottrim",

        "--overwrite",

        "--output",
        LOCAL_PACKED_PATH.name,
    ]
)


if (
    config[
        "orientation_mode"
    ]
    == "preoriented"
):

    command.append(
        "--preoriented"
    )


if supports_verbose:

    command.append(
        "--verbose"
    )


# ==============================================================
# Clean child environment
#
# We do not inherit PACKMOL_PATH because a stale or differently
# interpreted value was the cause of the earlier integration
# failure.  Either explicit --packmol is used or PACKMOL is
# discovered from a controlled PATH.
# ==============================================================

child_env = (
    os.environ.copy()
)


child_env.pop(
    "PACKMOL_PATH",
    None,
)


packmol_bin_dir = str(
    packmol_path.parent
)


path_entries = (
    child_env
    .get(
        "PATH",
        "",
    )
    .split(
        os.pathsep
    )
)


path_entries = [
    entry
    for entry in path_entries
    if entry
]


if (
    packmol_bin_dir
    not in path_entries
):

    path_entries.insert(
        0,
        packmol_bin_dir,
    )


child_env[
    "PATH"
] = os.pathsep.join(
    path_entries
)


# Force the Python wrapper itself to flush output promptly.
child_env[
    "PYTHONUNBUFFERED"
] = "1"


print("Membrane build preflight")
print("------------------------")

print(
    "Persistent project:",
    MEMBRANE_DIR,
)

print(
    "Local scratch:",
    SCRATCH_DIR,
)

print(
    "Protein residues:",
    protein_residue_count,
)

print(
    "Protein hydrogens:",
    protein_hydrogen_count,
)

print(
    "Lipids:",
    ":".join(
        config[
            "lipids"
        ]
    ),
)

print(
    "Ratio:",
    ":".join(
        str(value)
        for value
        in config[
            "ratios"
        ]
    ),
)

print(
    "XY padding:",
    f"{config['xy_padding_A']:.1f} Å",
)

print(
    "Water padding:",
    f"{config['water_padding_A']:.1f} Å",
)

print(
    "Orientation:",
    config[
        "orientation_mode"
    ],
)

print(
    "PACKMOL resolution:",
    (
        "explicit --packmol"
        if use_explicit_packmol
        else "controlled PATH"
    ),
)

print(
    "Verbose mode:",
    supports_verbose,
)

print()


# ==============================================================
# Long-running stage
# ==============================================================

runtime_seconds = (
    run_streaming_process(
        command=command,

        cwd=SCRATCH_DIR,

        local_log_path=(
            LOCAL_STREAM_LOG
        ),

        persistent_log_path=(
            MEMBRANE_BUILD_LOG
        ),

        env=child_env,
    )
)


# ==============================================================
# Validate builder output
# ==============================================================

if (
    not LOCAL_PACKED_PATH.is_file()
    or LOCAL_PACKED_PATH.stat().st_size
    == 0
):

    persist_diagnostics(
        SCRATCH_DIR
    )

    raise RuntimeError(
        "PACKMOL-Memgen exited successfully "
        "but did not create the requested "
        "packed system:\n"
        f"{LOCAL_PACKED_PATH}"
    )


packed_structure = (
    gemmi.read_structure(
        str(
            LOCAL_PACKED_PATH
        )
    )
)


if len(
    packed_structure
) != 1:

    persist_diagnostics(
        SCRATCH_DIR
    )

    raise RuntimeError(
        "Packed system must contain "
        "exactly one model."
    )


packed_sequence, packed_backbone = (
    protein_backbone(
        packed_structure
    )
)


if (
    original_sequence
    != packed_sequence
):

    persist_diagnostics(
        SCRATCH_DIR
    )

    raise RuntimeError(
        "Protein sequence changed "
        "during membrane construction."
    )


if (
    original_backbone.shape
    != packed_backbone.shape
):

    persist_diagnostics(
        SCRATCH_DIR
    )

    raise RuntimeError(
        "Protein backbone atom count "
        "changed during membrane construction."
    )


(
    rotation,
    translation,
    rigid_fit_rmsd,
) = kabsch_transform(
    original_backbone,
    packed_backbone,
)


if (
    rigid_fit_rmsd
    > MAX_PROTEIN_RIGID_FIT_RMSD_A
):

    persist_diagnostics(
        SCRATCH_DIR
    )

    raise RuntimeError(
        "PACKMOL-Memgen changed internal "
        "protein geometry instead of applying "
        "a near-rigid orientation:\n"
        f"RMSD = "
        f"{rigid_fit_rmsd:.4f} Å"
    )


# ==============================================================
# Apply precisely the same rigid transform to the complete
# validated complex.  The protein-ligand pose therefore remains
# unchanged by construction.
# ==============================================================

oriented_complex = (
    validated_complex.clone()
)


apply_rigid_transform(
    oriented_complex,
    rotation,
    translation,
)


oriented_complex.write_pdb(
    str(
        ORIENTED_COMPLEX_PATH
    )
)


# ==============================================================
# The PACKMOL protein is NOT our downstream source of truth.
#
# Remove it and keep only the generated membrane/water
# environment.
# ==============================================================

environment = extract_environment(
    packed_structure
)


environment_atom_count = sum(
    1
    for chain in environment[0]
    for residue in chain
    for atom in residue
)


environment_residues = [
    residue
    for chain in environment[0]
    for residue in chain
]


environment_residue_count = len(
    environment_residues
)


if environment_atom_count == 0:

    persist_diagnostics(
        SCRATCH_DIR
    )

    raise RuntimeError(
        "No membrane/water environment "
        "was produced."
    )


residue_counts = Counter(
    residue.name
    .strip()
    .upper()
    for residue
    in environment_residues
)


water_count = sum(
    count
    for name, count
    in residue_counts.items()
    if name
    in WATER_RESIDUE_NAMES
)


if (
    config[
        "water_padding_A"
    ] > 0
    and water_count == 0
):

    persist_diagnostics(
        SCRATCH_DIR
    )

    raise RuntimeError(
        "Membrane builder produced "
        "no recognizable water residues."
    )


environment.write_pdb(
    str(
        MEMBRANE_ENVIRONMENT_PATH
    )
)


# Keep the original PACKMOL-Memgen output for provenance.
shutil.copy2(
    LOCAL_PACKED_PATH,
    PACKED_REFERENCE_PATH,
)


# ==============================================================
# Periodic box
# ==============================================================

box_lengths, box_source = (
    determine_box_lengths(
        packed_path=(
            LOCAL_PACKED_PATH
        ),

        scratch_dir=(
            SCRATCH_DIR
        ),
    )
)


if any(
    (
        not np.isfinite(
            value
        )
        or value <= 0
    )
    for value
    in box_lengths
):

    persist_diagnostics(
        SCRATCH_DIR
    )

    raise RuntimeError(
        "Invalid periodic box:\n"
        f"{box_lengths}"
    )


# ==============================================================
# Persist all useful diagnostics AFTER validation
# ==============================================================

persist_diagnostics(
    SCRATCH_DIR
)


record = {
    "status":
        "built",

    "method":
        "packmol_memgen_monitored_local_scratch",

    "runtime_seconds":
        float(
            runtime_seconds
        ),

    "orientation_mode":
        config[
            "orientation_mode"
        ],

    "protein_residues":
        protein_residue_count,

    "protein_hydrogens":
        protein_hydrogen_count,

    "protein_rigid_fit_rmsd_A":
        float(
            rigid_fit_rmsd
        ),

    "rotation":
        rotation.tolist(),

    "translation_A":
        translation.tolist(),

    "protein_scaffold":
        str(
            PROTEIN_SCAFFOLD_PATH
        ),

    "protein_scaffold_sha256":
        sha256_file(
            PROTEIN_SCAFFOLD_PATH
        ),

    "packmol_reference_system":
        str(
            PACKED_REFERENCE_PATH
        ),

    "packmol_reference_sha256":
        sha256_file(
            PACKED_REFERENCE_PATH
        ),

    "oriented_complex":
        str(
            ORIENTED_COMPLEX_PATH
        ),

    "oriented_complex_sha256":
        sha256_file(
            ORIENTED_COMPLEX_PATH
        ),

    "environment_pdb":
        str(
            MEMBRANE_ENVIRONMENT_PATH
        ),

    "environment_pdb_sha256":
        sha256_file(
            MEMBRANE_ENVIRONMENT_PATH
        ),

    "environment_atoms":
        environment_atom_count,

    "environment_residues":
        environment_residue_count,

    "environment_residue_counts":
        dict(
            sorted(
                residue_counts.items()
            )
        ),

    "water_molecules":
        water_count,

    "box_A":
        [
            float(value)
            for value
            in box_lengths
        ],

    "box_source":
        box_source,

    "packmol_memgen":
        str(
            memgen_path
        ),

    "packmol":
        str(
            packmol_path
        ),

    "packmol_resolution":
        (
            "explicit_argument"
            if use_explicit_packmol
            else "controlled_PATH"
        ),

    "verbose":
        bool(
            supports_verbose
        ),

    "log":
        str(
            MEMBRANE_BUILD_LOG
        ),

    "diagnostics":
        str(
            MEMBRANE_DIAGNOSTICS_DIR
        ),
}


MEMBRANE_BUILD_RECORD_PATH.write_text(
    json.dumps(
        record,
        indent=2,
    )
)


print()
print("Membrane environment built")
print("--------------------------")

print(
    "Runtime:",
    format_elapsed(
        runtime_seconds
    ),
)

print(
    "Protein rigid-fit RMSD:",
    f"{rigid_fit_rmsd:.4f} Å",
)

print(
    "Environment residues:",
    environment_residue_count,
)

print(
    "Environment atoms:",
    environment_atom_count,
)

print(
    "Water molecules:",
    water_count,
)

print()
print(
    "Most common environment residues:"
)


for name, count in (
    residue_counts
    .most_common(
        10
    )
):

    print(
        f"  {name}: {count}"
    )


print()
print(
    "Box:",
    " × ".join(
        f"{value:.2f}"
        for value
        in box_lengths
    ),
    "Å",
)

print(
    "Box source:",
    box_source,
)

print()
print(
    "Oriented validated complex:",
    ORIENTED_COMPLEX_PATH,
)

print(
    "Membrane/water environment:",
    MEMBRANE_ENVIRONMENT_PATH,
)

print(
    "PACKMOL reference:",
    PACKED_REFERENCE_PATH,
)

print(
    "Diagnostics:",
    MEMBRANE_DIAGNOSTICS_DIR,
)

print(
    "Record:",
    MEMBRANE_BUILD_RECORD_PATH,
)

In [ ]:
from pathlib import Path
import json
import math
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import TwoSlopeNorm

# --------- robust project discovery ---------
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", "/content/drive/MyDrive/oncotarget_pipeline"))
SEARCH_ROOTS = [
    PROJECT_ROOT,
    PROJECT_ROOT.parent if PROJECT_ROOT.parent != PROJECT_ROOT else PROJECT_ROOT,
    Path.cwd(),
    Path("/mnt/data"),
]

ANALYSIS_DIR = PROJECT_ROOT / "analysis_figures"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

def find_first(pattern: str):
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None

def load_tsv(pattern: str) -> pd.DataFrame:
    path = find_first(pattern)
    if path is None or not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path, sep="\t")

def safe_num(s):
    return pd.to_numeric(s, errors="coerce")

def save_fig(fig, stem: str):
    png = ANALYSIS_DIR / f"{stem}.png"
    svg = ANALYSIS_DIR / f"{stem}.svg"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print(f"saved: {png}")
    return png, svg

def matrix_from_long(df, row_col, col_col, val_col):
    if df.empty or not {row_col, col_col, val_col}.issubset(df.columns):
        return None
    mat = df.pivot(index=row_col, columns=col_col, values=val_col)
    return mat.sort_index().sort_index(axis=1)

def plot_heatmap(mat: pd.DataFrame, title: str, stem: str, cmap="viridis", vmin=None, vmax=None, center=None, cbar_label=""):
    if mat is None or mat.empty:
        return None
    fig_w = max(7, 0.45 * mat.shape[1] + 3)
    fig_h = max(5, 0.35 * mat.shape[0] + 2.5)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    arr = mat.to_numpy(dtype=float)

    if center is not None:
        norm = TwoSlopeNorm(
            vmin=np.nanmin(arr) if vmin is None else vmin,
            vcenter=center,
            vmax=np.nanmax(arr) if vmax is None else vmax
        )
        im = ax.imshow(arr, aspect="auto", cmap=cmap, norm=norm)
    else:
        im = ax.imshow(arr, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

    ax.set_title(title)
    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(mat.columns, rotation=90, fontsize=8)
    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(mat.index, fontsize=8)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if cbar_label:
        cbar.set_label(cbar_label)
    fig.tight_layout()
    save_fig(fig, stem)
    plt.show()
    return fig

# --------- load your generated tables ---------
cons_summary = load_tsv("*consensus_global_summary.tsv")
pocket_tables = sorted([p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("*_pocket_summary.tsv")])
jaccard_tables = sorted([p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("*_jaccard_matrix.tsv")])
votes_tables = sorted([p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("*_residue_votes.tsv")])
aa_tables = sorted([p for root in SEARCH_ROOTS if root.exists() for p in root.rglob("*_aa_composition.tsv")])

best_tsv = find_first("best_per_target.tsv")
md_top3_tsv = find_first("md_candidates_top3_per_target.tsv")
ranked_tsv = find_first("ranked_all_successful.tsv")

best_df = pd.read_csv(best_tsv, sep="\t") if best_tsv else pd.DataFrame()
md3_df = pd.read_csv(md_top3_tsv, sep="\t") if md_top3_tsv else pd.DataFrame()
ranked_df = pd.read_csv(ranked_tsv, sep="\t") if ranked_tsv else pd.DataFrame()

# --------- build one combined table for all targets ---------
combined = cons_summary.copy() if not cons_summary.empty else pd.DataFrame()

if not combined.empty:
    if "methods_used" in combined.columns:
        combined["n_methods_used"] = combined["methods_used"].fillna("").astype(str).apply(
            lambda s: len([x for x in s.split(",") if x.strip()])
        )
    else:
        combined["n_methods_used"] = np.nan

    for c in ["grid_center_x", "grid_center_y", "grid_center_z", "grid_size_x", "grid_size_y", "grid_size_z", "n_final_consensus_residues"]:
        if c not in combined.columns:
            combined[c] = np.nan

    combined["grid_volume_A3"] = safe_num(combined["grid_size_x"]) * safe_num(combined["grid_size_y"]) * safe_num(combined["grid_size_z"])
    combined["grid_diag_A"] = np.sqrt(safe_num(combined["grid_size_x"])**2 + safe_num(combined["grid_size_y"])**2 + safe_num(combined["grid_size_z"])**2)
    combined["consensus_density"] = safe_num(combined["n_final_consensus_residues"]) / combined["grid_volume_A3"].replace(0, np.nan)

    if not ranked_df.empty and "target_input" in ranked_df.columns:
        ranked_agg = ranked_df.groupby("target_input").agg(
            n_successful_docks=("global_job_index", "size"),
            mean_ranked_CNNscore=("CNNscore", "mean"),
            max_ranked_CNNscore=("CNNscore", "max"),
            mean_ranked_minimizedAffinity=("minimizedAffinity", "mean"),
            best_ranked_minimizedAffinity=("minimizedAffinity", "min"),
        ).reset_index()
        combined = combined.merge(ranked_agg, on="target_input", how="left")

    if not best_df.empty and "target_input" in best_df.columns:
        keep = [c for c in ["target_input", "route_type", "CNNscore", "CNNaffinity", "minimizedAffinity"] if c in best_df.columns]
        best_sub = best_df[keep].copy()
        rename_map = {c: f"best_{c}" for c in keep if c != "target_input"}
        best_sub = best_sub.rename(columns=rename_map)
        combined = combined.merge(best_sub, on="target_input", how="left")

    if not md3_df.empty and "target_input" in md3_df.columns:
        md3_agg = md3_df.groupby("target_input").agg(
            md3_mean_CNNscore=("CNNscore", "mean"),
            md3_best_CNNscore=("CNNscore", "max"),
            md3_mean_CNNaffinity=("CNNaffinity", "mean"),
            md3_best_CNNaffinity=("CNNaffinity", "max"),
            md3_mean_minimizedAffinity=("minimizedAffinity", "mean"),
            md3_best_minimizedAffinity=("minimizedAffinity", "min"),
        ).reset_index()
        combined = combined.merge(md3_agg, on="target_input", how="left")

    combined = combined.sort_values(["n_final_consensus_residues", "target_input"], ascending=[False, True])

    combined_path = ANALYSIS_DIR / "combined_target_metrics.tsv"
    combined.to_csv(combined_path, sep="\t", index=False)
    print(f"saved: {combined_path}")

# --------- global target x metric heatmap ---------
if not combined.empty:
    preferred_cols = [
        "n_final_consensus_residues",
        "n_methods_used",
        "grid_center_x", "grid_center_y", "grid_center_z",
        "grid_size_x", "grid_size_y", "grid_size_z",
        "grid_volume_A3", "grid_diag_A",
        "consensus_density",
        "n_successful_docks",
        "mean_ranked_CNNscore", "max_ranked_CNNscore",
        "mean_ranked_minimizedAffinity", "best_ranked_minimizedAffinity",
        "best_CNNscore", "best_CNNaffinity", "best_minimizedAffinity",
        "md3_mean_CNNscore", "md3_best_CNNscore",
        "md3_mean_CNNaffinity", "md3_best_CNNaffinity",
        "md3_mean_minimizedAffinity", "md3_best_minimizedAffinity",
    ]
    metric_cols = [c for c in preferred_cols if c in combined.columns]
    mat = combined.set_index("target_input")[metric_cols].apply(pd.to_numeric, errors="coerce")

    # z-score by column for a clean paper-style overview
    z = (mat - mat.mean(axis=0)) / mat.std(axis=0, ddof=0).replace(0, np.nan)
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    plot_heatmap(
        z,
        title="Target × metric overview (z-scores)",
        stem="target_metric_overview_zscore",
        cmap="coolwarm",
        center=0.0,
        cbar_label="z-score",
    )

    # compact raw-value table for archiving
    fig, ax = plt.subplots(figsize=(min(22, 0.8 * len(metric_cols) + 4), max(4, 0.45 * len(combined) + 2)))
    ax.axis("off")
    table_df = combined[["target_input"] + metric_cols].copy()
    tbl = ax.table(
        cellText=table_df.round(3).astype(str).values,
        colLabels=table_df.columns.tolist(),
        loc="center",
        cellLoc="center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1, 1.2)
    ax.set_title("Combined target metrics table", pad=20)
    save_fig(fig, "combined_target_metrics_table")
    plt.show()

# --------- per-target pocket-level Jaccard and method-level consensus heatmaps ---------
def plot_target_jaccard_heatmaps(pocket_path: Path, jaccard_path: Path):
    pockets = pd.read_csv(pocket_path, sep="\t")
    jacc = pd.read_csv(jaccard_path, sep="\t")
    target = str(pockets["target_input"].iloc[0]) if "target_input" in pockets.columns and not pockets.empty else pocket_path.stem.replace("_pocket_summary", "")

    # raw pocket-to-pocket matrix
    raw = matrix_from_long(jacc, "pocket_1", "pocket_2", "jaccard")
    if raw is not None and not raw.empty:
        plot_heatmap(raw, f"{target}: pocket-to-pocket Jaccard", f"{target}_pocket_jaccard", cmap="viridis", vmin=0.0, vmax=1.0, cbar_label="Jaccard")

    # method-level matrix = max Jaccard over pocket pairs of the same methods
    if {"method", "pocket_name"}.issubset(pockets.columns) and not jacc.empty:
        pocket_to_method = dict(zip(pockets["pocket_name"].astype(str), pockets["method"].astype(str)))
        methods = sorted(set(pocket_to_method.values()))
        lookup = {}
        for _, r in jacc.iterrows():
            p1 = str(r["pocket_1"]); p2 = str(r["pocket_2"])
            lookup[(p1, p2)] = float(r["jaccard"])

        mm = pd.DataFrame(index=methods, columns=methods, dtype=float)
        for m1 in methods:
            p1s = [p for p, m in pocket_to_method.items() if m == m1]
            for m2 in methods:
                p2s = [p for p, m in pocket_to_method.items() if m == m2]
                vals = []
                for p1 in p1s:
                    for p2 in p2s:
                        v = lookup.get((p1, p2), lookup.get((p2, p1), np.nan))
                        if pd.notna(v):
                            vals.append(float(v))
                mm.loc[m1, m2] = np.nan if not vals else float(np.nanmax(vals))

        plot_heatmap(mm, f"{target}: method consensus (max Jaccard)", f"{target}_method_consensus_jaccard", cmap="viridis", vmin=0.0, vmax=1.0, cbar_label="max Jaccard")

for p in pocket_tables:
    target = p.stem.replace("_pocket_summary", "")
    jp = next((x for x in jaccard_tables if x.stem.replace("_jaccard_matrix", "") == target), None)
    if jp is not None:
        try:
            plot_target_jaccard_heatmaps(p, jp)
        except Exception as e:
            print(f"[skip] {target}: {e}")

# --------- grid map from consensus summary ---------
if not combined.empty and {"grid_center_x", "grid_center_y", "grid_size_x", "grid_size_y"}.issubset(combined.columns):
    fig, ax = plt.subplots(figsize=(8, 7))
    for _, row in combined.iterrows():
        try:
            cx = float(row["grid_center_x"]); cy = float(row["grid_center_y"])
            sx = float(row["grid_size_x"]); sy = float(row["grid_size_y"])
        except Exception:
            continue
        ax.add_patch(Rectangle((cx - sx/2, cy - sy/2), sx, sy, fill=False, lw=1.5))
        ax.scatter([cx], [cy], s=25)
        ax.text(cx, cy, str(row["target_input"]), fontsize=8, ha="left", va="bottom")
    ax.set_title("Docking grid overview (x–y projection)")
    ax.set_xlabel("grid center X")
    ax.set_ylabel("grid center Y")
    ax.grid(alpha=0.25)
    ax.set_aspect("equal", adjustable="datalim")
    save_fig(fig, "grid_overview_xy")
    plt.show()

# --------- optional hooks for external pocket predictors ---------
# Keep these as optional add-ons so they never break the main pipeline.
OPTIONAL_POCKET_TOOLS = {
    "fpocket": {
        "enabled": False,   # switch on only if fpocket is installed and you want to call it here
        "executable": "fpocket",
    },
    "p2rank": {
        "enabled": False,   # switch on only if the standalone P2Rank CLI is installed
        "executable": "p2rank",
    },
}

print("Done. Figures and combined tables are in:", ANALYSIS_DIR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import OrderedDict

# -------------------------------------------------------
# Jaccard similarity
# -------------------------------------------------------
def jaccard_similarity(set_a, set_b):
    a = set(set_a)
    b = set(set_b)
    union = len(a | b)
    if union == 0:
        return 0.0
    return len(a & b) / union


# -------------------------------------------------------
# Build matrix from pockets
# -------------------------------------------------------
def build_jaccard_matrix(pockets_by_algorithm):
    labels = []
    pocket_sets = []
    block_sizes = []

    for alg_name, pocket_list in pockets_by_algorithm.items():
        block_sizes.append(len(pocket_list))
        for i, pocket in enumerate(pocket_list, start=1):
            labels.append(f"{alg_name}\nP{i}")
            pocket_sets.append(set(pocket))

    n = len(pocket_sets)
    matrix = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(n):
            matrix[i, j] = jaccard_similarity(pocket_sets[i], pocket_sets[j])

    return labels, pocket_sets, matrix, block_sizes


# -------------------------------------------------------
# Main heatmap + selection plot
# -------------------------------------------------------
def plot_heatmap_and_selected(pockets_by_algorithm, selected_label="P2Rank\nP1"):
    labels, pocket_sets, matrix, block_sizes = build_jaccard_matrix(pockets_by_algorithm)

    if selected_label not in labels:
        raise ValueError(f"Selected pocket '{selected_label}' not found in labels.")

    selected_idx = labels.index(selected_label)
    selected_set = pocket_sets[selected_idx]

    # Similarity to the selected pocket
    selected_scores = [jaccard_similarity(selected_set, p) for p in pocket_sets]

    # Sort other pockets by similarity for display
    order = np.argsort(selected_scores)[::-1]
    sorted_labels = [labels[i] for i in order]
    sorted_scores = [selected_scores[i] for i in order]
    sorted_colors = ["#d62728" if i == selected_idx else "#1f77b4" for i in order]

    # Figure layout
    fig = plt.figure(figsize=(18, 10))
    gs = fig.add_gridspec(1, 2, width_ratios=[2.2, 1.0], wspace=0.25)

    # --- Left: heatmap ---
    ax0 = fig.add_subplot(gs[0, 0])
    im = ax0.imshow(matrix, vmin=0.0, vmax=1.0, interpolation="nearest")

    cbar = fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04)
    cbar.set_label("Сходство Жаккара", rotation=90)

    # Меняем точки на запятые на шкале colorbar
    cbar_ticks = cbar.get_ticks()
    cbar.set_ticklabels([f"{t:.1f}".replace('.', ',') for t in cbar_ticks])

    ax0.set_xticks(np.arange(len(labels)))
    ax0.set_yticks(np.arange(len(labels)))
    ax0.set_xticklabels(labels, fontsize=9)
    ax0.set_yticklabels(labels, fontsize=9)
    plt.setp(ax0.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    ax0.set_title("Попарное сходство Жаккара между предсказанными карманами", fontsize=14, pad=15)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            val = matrix[i, j]
            # Заменяем точку на запятую в ячейках матрицы
            val_str = f"{val:.2f}".replace('.', ',')
            ax0.text(
                j, i, val_str,
                ha="center", va="center",
                fontsize=8,
                color="white" if val < 0.50 else "black"
            )

    cumulative = np.cumsum(block_sizes)
    for b in cumulative[:-1]:
        ax0.axhline(b - 0.5, color="black", linewidth=1.2)
        ax0.axvline(b - 0.5, color="black", linewidth=1.2)

    ax0.set_xlim(-0.5, len(labels) - 0.5)
    ax0.set_ylim(len(labels) - 0.5, -0.5)

    # Highlight selected pocket row/column
    ax0.axhline(selected_idx - 0.5, color="red", linewidth=2.2)
    ax0.axhline(selected_idx + 0.5, color="red", linewidth=2.2)
    ax0.axvline(selected_idx - 0.5, color="red", linewidth=2.2)
    ax0.axvline(selected_idx + 0.5, color="red", linewidth=2.2)

    ax0.text(
        selected_idx, -1.2,
        "Выбранный консенсусный карман",
        ha="center", va="center",
        fontsize=10, color="red", fontweight="bold"
    )

    # --- Right: similarity to selected pocket ---
    ax1 = fig.add_subplot(gs[0, 1])
    y = np.arange(len(sorted_labels))

    bars = ax1.barh(y, sorted_scores, color=sorted_colors, edgecolor="black", linewidth=0.6)
    ax1.set_yticks(y)
    ax1.set_yticklabels(sorted_labels, fontsize=9)
    ax1.invert_yaxis()
    ax1.set_xlim(0, 1.0)

    # Меняем точки на запятые на оси X правого графика
    ax1_ticks = ax1.get_xticks()
    ax1.set_xticks(ax1_ticks)
    ax1.set_xticklabels([f"{t:.1f}".replace('.', ',') for t in ax1_ticks])

    ax1.set_xlabel("Сходство Жаккара")

    # В названии правого графика переносим строку, оставляя имя алгоритма как есть
    selected_label_clean = selected_label.replace('\n', ' ')
    ax1.set_title(f"Сходство с выбранным карманом\n({selected_label_clean})", fontsize=14, pad=12)

    # Highlight the selected pocket itself
    for i, lab in enumerate(sorted_labels):
        if lab == selected_label:
            bars[i].set_color("#d62728")
            bars[i].set_edgecolor("black")
            bars[i].set_linewidth(1.0)

    # Annotate bar values (замена точек на запятые у баров)
    for i, score in enumerate(sorted_scores):
        score_str = f"{score:.2f}".replace('.', ',')
        ax1.text(score + 0.02, i, score_str, va="center", fontsize=9)

    # Threshold line for “consensus”
    consensus_threshold = 0.45
    ax1.axvline(consensus_threshold, color="gray", linestyle="--", linewidth=1.2)

    # Замена точки на запятую в тексте порога
    threshold_str = f"{consensus_threshold:.2f}".replace('.', ',')
    ax1.text(consensus_threshold + 0.01, len(sorted_labels) - 0.5,
             f"порог = {threshold_str}",
             fontsize=9, color="gray", va="bottom")

    fig.suptitle("Выбор консенсусного кармана по попарному сходству Жаккара", fontsize=16, y=0.98)
    fig.tight_layout()
    return fig, (ax0, ax1)


# -------------------------------------------------------
# Demo synthetic pockets with a consensus pocket
# -------------------------------------------------------
def make_variant_pocket(core, universe, keep_frac=0.8, add_frac=0.12, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    core = np.array(sorted(core))
    outside = np.setdiff1d(universe, core)

    keep_n = max(4, int(round(len(core) * keep_frac)))
    keep_n = min(keep_n, len(core))
    kept = set(rng.choice(core, size=keep_n, replace=False))

    add_n = max(2, int(round(len(core) * add_frac)))
    add_n = min(add_n, len(outside))
    added = set(rng.choice(outside, size=add_n, replace=False))

    return kept | added


def build_consensus_demo_pockets(seed=11):
    rng = np.random.default_rng(seed)
    universe = np.arange(1, 401)

    consensus_core = set(rng.choice(np.arange(120, 170), size=28, replace=False))
    alt_core_1 = set(rng.choice(np.arange(20, 70), size=22, replace=False))
    alt_core_2 = set(rng.choice(np.arange(210, 260), size=24, replace=False))
    alt_core_3 = set(rng.choice(np.arange(300, 360), size=23, replace=False))

    pockets_by_algorithm = OrderedDict()

    pockets_by_algorithm["P2Rank"] = [
        make_variant_pocket(consensus_core, universe, keep_frac=0.88, add_frac=0.08, rng=rng),  # P1
        make_variant_pocket(alt_core_1, universe, keep_frac=0.82, add_frac=0.12, rng=rng),      # P2
        make_variant_pocket(alt_core_3, universe, keep_frac=0.80, add_frac=0.12, rng=rng),      # P3
    ]

    pockets_by_algorithm["fpocket"] = [
        make_variant_pocket(alt_core_2, universe, keep_frac=0.78, add_frac=0.14, rng=rng),      # P1
        make_variant_pocket(consensus_core, universe, keep_frac=0.84, add_frac=0.10, rng=rng),  # P2
        make_variant_pocket(alt_core_1, universe, keep_frac=0.77, add_frac=0.15, rng=rng),      # P3
    ]

    pockets_by_algorithm["COACH-D"] = [
        make_variant_pocket(consensus_core, universe, keep_frac=0.90, add_frac=0.07, rng=rng),  # P1
        make_variant_pocket(alt_core_2, universe, keep_frac=0.80, add_frac=0.12, rng=rng),      # P2
        make_variant_pocket(alt_core_3, universe, keep_frac=0.76, add_frac=0.14, rng=rng),      # P3
    ]

    pockets_by_algorithm["DoGSiteScorer"] = [
        make_variant_pocket(alt_core_1, universe, keep_frac=0.76, add_frac=0.14, rng=rng),      # P1
        make_variant_pocket(alt_core_2, universe, keep_frac=0.78, add_frac=0.13, rng=rng),      # P2
        make_variant_pocket(consensus_core, universe, keep_frac=0.85, add_frac=0.12, rng=rng),   # P3
    ]

    return pockets_by_algorithm


# -------------------------------------------------------
# Run
# -------------------------------------------------------
demo_pockets = build_consensus_demo_pockets(seed=11)
fig, axes = plot_heatmap_and_selected(
    demo_pockets,
    selected_label="P2Rank\nP1"
)
plt.show()